# Training pipeline

### Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

In [ ]:
from datetime import date, timedelta
import gc
import json
import math
import os
import shutil
import subprocess
import time

from dotenv import load_dotenv
import mlflow
import numba
import numpy as np
import polars as pl
from scipy.optimize import nnls
from scipy.special import ndtri
import torch
from torch import nn

DATA = ROOT / "data"
MODELS = ROOT / "models"
SUBS = ROOT / "submissions"
REFERENCE = ROOT / "archive" / "reference"
SETUP = DATA / "setup"
FEAT = DATA / "features"
PRE = DATA / "features_pre"
SEQ = DATA / "seq"
V1 = DATA / "features_v1"
CHRONOS = DATA / "chronos"
MEMBER_DIR = MODELS / "members"
LEGACY_DIR = MODELS / "legacy"
COMBINER_DIR = MODELS / "combiner"
ARCH = DATA / "archive"
for d in (
    DATA,
    MODELS,
    SUBS,
    SETUP,
    FEAT,
    FEAT / "x",
    FEAT / "e",
    FEAT / "f",
    FEAT / "rg",
    PRE,
    PRE / "x",
    PRE / "e",
    PRE / "f",
    PRE / "rg",
    SEQ,
    V1,
    CHRONOS,
    MEMBER_DIR,
    LEGACY_DIR,
    COMBINER_DIR,
    ARCH,
):
    d.mkdir(parents=True, exist_ok=True)

VPS_IP = "2.26.27.187"
EXPERIMENT = "ecup-prod"
load_dotenv(ROOT / ".env")
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", f"https://{VPS_IP}"))
mlflow.set_experiment(EXPERIMENT)
mlflow.enable_system_metrics_logging()
mlflow.set_system_metrics_sampling_interval(5)
mlflow.set_system_metrics_samples_before_logging(2)

### Download

In [ ]:
TRAIN_PATH = DATA / "train.parquet"
if not TRAIN_PATH.exists():
    curl = shutil.which("curl")
    part = TRAIN_PATH.with_suffix(".parquet.part")
    subprocess.run(
        [
            curl,
            "-sk",
            "-L",
            "--retry",
            "30",
            "--retry-all-errors",
            "--retry-delay",
            "2",
            "--create-dirs",
            "-o",
            str(part),
            f"https://{VPS_IP}/files/total.parquet",
        ],
        check=True,
    )
    pl.read_parquet(part, n_rows=1)
    part.replace(TRAIN_PATH)

### Load

In [ ]:
data = pl.read_parquet(TRAIN_PATH).sort("user_id", "event_date")
DATE_MIN, DATE_MAX = data["event_date"].min(), data["event_date"].max()

### Anchors, cohorts and targets

In [ ]:
HORIZON_DAYS = 30
MIN_HISTORY_DAYS = 60
CLEAN = ["2025-09-24", "2025-10-08", "2025-10-22"]
RECENT = ["2025-12-03", "2025-12-17", "2025-12-31", "2026-01-14"]
VAL_ANCHORS = [*CLEAN, *RECENT]
PANEL = ["2025-08-27", "2025-10-08", "2025-11-12", "2025-12-17"]
EARLY_ANCHORS = [date(2025, 3, 12), date(2025, 4, 9), date(2025, 5, 7), date(2025, 6, 4)]

TRAIN_DATES = list(EARLY_ANCHORS)
TRAIN_DATES += [date(2025, 7, 2) + timedelta(days=14 * k) for k in range(7)]
TRAIN_DATES += [date(2025, 10, 1) + timedelta(days=7 * k) for k in range(16)]
TRAIN_DATES = sorted(TRAIN_DATES)
ALL_DATES = [*TRAIN_DATES, DATE_MAX]
ANCHORS = [a.isoformat() for a in TRAIN_DATES]
SUBMIT = DATE_MAX.isoformat()
ANCHOR_IDX = {a: i for i, a in enumerate(ANCHORS)}

In [ ]:
CONFIG_PATH = SETUP / "config.json"
if not CONFIG_PATH.exists():
    users = data["user_id"].unique().sort().to_numpy()
    n_users = len(users)
    rows = np.searchsorted(users, data["user_id"].to_numpy())
    day = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    gmv_all = data["gmv"].to_numpy().astype(np.float64)
    first_day = day[np.searchsorted(rows, np.arange(n_users))]

    cohort = np.zeros((len(ALL_DATES), n_users), dtype=bool)
    targets = np.zeros((len(TRAIN_DATES), n_users), dtype=np.float64)
    active_next = np.zeros((len(TRAIN_DATES), n_users), dtype=bool)
    for i, anchor in enumerate(ALL_DATES):
        t = (anchor - DATE_MIN).days
        recent = (day > t - HORIZON_DAYS) & (day <= t)
        cohort[i] = (np.bincount(rows[recent], minlength=n_users) > 0) & (
            first_day <= t - MIN_HISTORY_DAYS
        )
        if i < len(TRAIN_DATES):
            nxt = (day > t) & (day <= t + HORIZON_DAYS)
            targets[i] = np.bincount(rows[nxt], weights=gmv_all[nxt], minlength=n_users)
            active_next[i] = np.bincount(rows[nxt], minlength=n_users) > 0

    levels = []
    for i, anchor in enumerate(TRAIN_DATES):
        m = cohort[i]
        y = targets[i][m]
        ly = np.log1p(y)
        levels.append({
            "anchor": anchor.isoformat(),
            "n_cohort": int(m.sum()),
            "p_active_next": float(active_next[i][m].mean()),
            "p_positive": float((y > 0).mean()),
            "e_log_given_positive": float(ly[y > 0].mean()),
            "mean_log1p": float(ly.mean()),
        })

    np.save(SETUP / "cohort.npy", cohort)
    np.save(SETUP / "targets.npy", targets)
    np.save(SETUP / "active_next.npy", active_next)
    np.save(SETUP / "user_ids.npy", users)
    (SETUP / "levels.json").write_text(json.dumps(levels, indent=1))
    CONFIG_PATH.write_text(
        json.dumps(
            {
                "date_min": DATE_MIN.isoformat(),
                "date_max": DATE_MAX.isoformat(),
                "horizon_days": HORIZON_DAYS,
                "min_history_days": MIN_HISTORY_DAYS,
                "n_users": n_users,
                "train_anchors": ANCHORS,
                "submit_anchor": SUBMIT,
            },
            indent=2,
        )
    )
    del cohort, targets, active_next, rows, day, gmv_all, first_day
    gc.collect()

CONFIG = json.loads(CONFIG_PATH.read_text())
COHORT = np.load(SETUP / "cohort.npy")
TARGETS = np.load(SETUP / "targets.npy")
ACTIVE_NEXT = np.load(SETUP / "active_next.npy")
USER_IDS = np.load(SETUP / "user_ids.npy")
N_USERS = CONFIG["n_users"]

### Aux targets

In [ ]:
AUX_NAMES = [
    "lgmv_30",
    "lgmv_search_30",
    "lgmv_cat_30",
    "lord_30",
    "lcart_30",
    "lsearches_30",
    "n_active_30",
    "n_order_30",
    "has_order_30",
    "lgmv_7",
    "lgmv_14",
    "lgmv_60",
]
AUX_USE = [
    "lgmv_search_30",
    "lgmv_cat_30",
    "lord_30",
    "lcart_30",
    "lsearches_30",
    "n_active_30",
    "n_order_30",
    "has_order_30",
    "lgmv_7",
    "lgmv_14",
]
AUX_PATH = DATA / "aux_targets.npy"
if not AUX_PATH.exists():
    rows = np.searchsorted(USER_IDS, data["user_id"].to_numpy())
    day = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    src_cols = {
        c: data[c].to_numpy().astype(np.float64)
        for c in ("gmv", "gmv_search", "gmv_cat", "to_ord", "to_cart", "searches")
    }
    has_ord_row = (src_cols["to_ord"] > 0).astype(np.float64)
    aux = np.zeros((len(TRAIN_DATES), N_USERS, len(AUX_NAMES)), dtype=np.float32)
    for i, anchor in enumerate(TRAIN_DATES):
        t = (anchor - DATE_MIN).days
        m30 = (day > t) & (day <= t + 30)
        for j, name in enumerate(("gmv", "gmv_search", "gmv_cat", "to_ord", "to_cart", "searches")):
            aux[i, :, j] = np.log1p(
                np.bincount(rows[m30], weights=src_cols[name][m30], minlength=N_USERS)
            )
        aux[i, :, 6] = np.bincount(rows[m30], minlength=N_USERS)
        aux[i, :, 7] = np.bincount(rows[m30], weights=has_ord_row[m30], minlength=N_USERS)
        aux[i, :, 8] = (aux[i, :, 7] > 0).astype(np.float32)
        for j, h in ((9, 7), (10, 14), (11, 60)):
            m = (day > t) & (day <= t + h)
            aux[i, :, j] = np.log1p(
                np.bincount(rows[m], weights=src_cols["gmv"][m], minlength=N_USERS)
            )
    np.save(AUX_PATH, aux)
    (DATA / "aux_names.json").write_text(json.dumps(AUX_NAMES, indent=1))
    print("aux targets", aux.shape, "max dev", np.abs(np.log1p(TARGETS) - aux[:, :, 0]).max())
    del aux, rows, day, src_cols, has_ord_row
    gc.collect()
AUX_IDX = [AUX_NAMES.index(c) for c in AUX_USE]

### Dense grid

In [ ]:
USER_BATCH = 25_000
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
HALF_LIVES = [3, 7, 14, 30, 60, 120, 240]
SUM_COLS = [
    "gmv",
    "gmv_search",
    "gmv_cat",
    "searches",
    "to_cart",
    "to_ord",
    "search_to_ord",
    "cat_to_ord",
    "cat",
    "active",
    "has_order",
    "has_cart",
    "loggmv",
]
EWM_COLS = ["gmv", "loggmv", "has_order", "active", "logsearches", "logto_cart", "logto_ord"]
EXTRA_SUM_COLS = [
    "search_only",
    "cat_only",
    "both_ch",
    "neither_ch",
    "search_to_cart",
    "cat_to_cart",
    "ord_t",
    "ord_t2",
    "loggmv2",
]
RANK_COLS = [
    "gmv_sum_30d",
    "gmv_sum_90d",
    "gmv_sum_365d",
    "to_ord_sum_30d",
    "to_ord_sum_90d",
    "to_ord_sum_365d",
    "searches_sum_30d",
    "active_sum_30d",
    "active_sum_365d",
    "has_order_sum_30d",
    "has_order_sum_365d",
    "loggmv_sum_365d",
    "days_since_last_event",
    "days_since_last_order",
    "ewm_loggmv_60",
    "ewm_loggmv_120",
    "ewm_has_order_60",
    "ewm_active_60",
    "ewm_logto_ord_60",
    "loggmv_per_order_365d",
]
V2_DENSE = [*SUM_COLS, "gmv2", "loggmv_t"]
V3_DENSE = [*V2_DENSE, *EXTRA_SUM_COLS]
CAP_COLS = [
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "days_since_first_order",
]
CAP_AT = 180.0
N_DAYS = (DATE_MAX - DATE_MIN).days + 1
ANCHOR_T = [(a - DATE_MIN).days for a in ALL_DATES]

In [ ]:
X_MISSING = [a for a in ALL_DATES if not (FEAT / "x" / f"X_{a.isoformat()}.npy").exists()]
E_MISSING = [a for a in ALL_DATES if not (FEAT / "e" / f"E_{a.isoformat()}.npy").exists()]
F_MISSING = [a for a in ALL_DATES if not (FEAT / "f" / f"F_{a.isoformat()}.npy").exists()]
NEED_PREP = bool(X_MISSING or E_MISSING or F_MISSING)

if NEED_PREP:
    UID = data["user_id"].to_numpy()
    DAY_IDX = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    gmv = data["gmv"].to_numpy().astype(np.float32)
    to_ord = data["to_ord"].to_numpy().astype(np.float32)
    to_cart = data["to_cart"].to_numpy().astype(np.float32)
    searches = data["searches"].to_numpy().astype(np.float32)
    cat = data["cat"].to_numpy().astype(np.float32)
    search = (searches > 0).astype(np.float32)
    has_order = (to_ord > 0).astype(np.float32)
    loggmv = np.log1p(gmv)
    fday = DAY_IDX.astype(np.float32)
    COLS = {
        "gmv": gmv,
        "gmv_search": data["gmv_search"].to_numpy().astype(np.float32),
        "gmv_cat": data["gmv_cat"].to_numpy().astype(np.float32),
        "searches": searches,
        "to_cart": to_cart,
        "to_ord": to_ord,
        "search_to_ord": data["search_to_ord"].to_numpy().astype(np.float32),
        "cat_to_ord": data["cat_to_ord"].to_numpy().astype(np.float32),
        "cat": cat,
        "active": np.ones(data.height, dtype=np.float32),
        "has_order": has_order,
        "has_cart": (to_cart > 0).astype(np.float32),
        "loggmv": loggmv,
        "gmv2": gmv * gmv,
        "logsearches": np.log1p(searches),
        "logto_cart": np.log1p(to_cart),
        "logto_ord": np.log1p(to_ord),
        "search_only": search * (1.0 - cat),
        "cat_only": (1.0 - search) * cat,
        "both_ch": search * cat,
        "neither_ch": (1.0 - search) * (1.0 - cat),
        "search_to_cart": data["search_to_cart"].to_numpy().astype(np.float32),
        "cat_to_cart": data["cat_to_cart"].to_numpy().astype(np.float32),
        "ord_t": has_order * fday,
        "ord_t2": has_order * fday * fday,
        "loggmv2": loggmv * loggmv,
        "loggmv_t": loggmv * fday,
    }
    BOUNDS = [*np.searchsorted(UID, USER_IDS[::USER_BATCH]).tolist(), len(UID)]
    del gmv, to_ord, to_cart, searches, cat, search, has_order, loggmv, fday
    gc.collect()

### Features

In [ ]:
N_X, N_E_BASE, N_RANK = 236, 62, 20
if X_MISSING or E_MISSING:
    XMAP, EMAP = {}, {}
    for a in ALL_DATES:
        key = a.isoformat()
        if a in X_MISSING:
            XMAP[key] = np.lib.format.open_memmap(
                FEAT / "x" / f"X_{key}.npy.part", mode="w+", dtype=np.float32, shape=(N_USERS, N_X)
            )
        if a in E_MISSING:
            EMAP[key] = np.lib.format.open_memmap(
                FEAT / "e" / f"E_{key}.npy.part",
                mode="w+",
                dtype=np.float32,
                shape=(N_USERS, N_E_BASE + N_RANK),
            )
    NAMES_X, NAMES_E = None, None
    for b in range(len(BOUNDS) - 1):
        lo, hi = BOUNDS[b], BOUNDS[b + 1]
        grow = np.searchsorted(USER_IDS, UID[lo:hi])
        base_row = int(grow[0])
        grow = grow - base_row
        nb = int(grow[-1]) + 1
        gday = DAY_IDX[lo:hi]

        CUM, KEPT = {}, {}
        for name in V3_DENSE:
            arr = np.zeros((nb, N_DAYS), dtype=np.float32)
            arr[grow, gday] = COLS[name][lo:hi]
            if name in ("active", "has_order", "has_cart") or name in EWM_COLS:
                KEPT[name] = arr
            CUM[name] = np.cumsum(arr, axis=1, dtype=np.float32)
        for name in EWM_COLS:
            if name not in KEPT:
                arr = np.zeros((nb, N_DAYS), dtype=np.float32)
                arr[grow, gday] = COLS[name][lo:hi]
                KEPT[name] = arr

        gmv_raw = KEPT["gmv"]
        run_max_gmv = np.maximum.accumulate(gmv_raw, axis=1)
        LAST = {}
        for name in ("active", "has_order", "has_cart"):
            idx = np.where(KEPT[name] > 0, np.arange(N_DAYS, dtype=np.int32), np.int32(-1))
            LAST[name] = np.maximum.accumulate(idx, axis=1)
        first_active = np.argmax(CUM["active"] > 0, axis=1).astype(np.float32)
        first_order = np.argmax(CUM["has_order"] > 0, axis=1).astype(np.float32)

        EWM = {}
        want = sorted(set(ANCHOR_T))
        for col in EWM_COLS:
            series = KEPT[col]
            for half_life in HALF_LIVES:
                alpha = 1.0 - 0.5 ** (1.0 / half_life)
                state = np.zeros(nb, dtype=np.float32)
                snaps, nxt = {}, 0
                for d in range(N_DAYS):
                    state += alpha * (series[:, d] - state)
                    while nxt < len(want) and want[nxt] == d:
                        snaps[d] = state.copy()
                        nxt += 1
                EWM[(col, half_life)] = snaps
        del KEPT
        gc.collect()

        for ai, a in enumerate(ALL_DATES):
            key = a.isoformat()
            if key not in XMAP and key not in EMAP:
                continue
            t = ANCHOR_T[ai]
            W = {}
            for col in V3_DENSE:
                for w in WINDOWS:
                    start = t - w
                    W[(col, w)] = CUM[col][:, t] - (CUM[col][:, start] if start >= 0 else 0.0)
            nx, fx = [], []
            for col in SUM_COLS:
                for w in WINDOWS:
                    nx.append(f"{col}_sum_{w}d")
                    fx.append(W[(col, w)])
            for w in (30, 90):
                n = np.maximum(W[("active", w)], 1.0)
                mean = W[("gmv", w)] / n
                nx.append(f"gmv_std_{w}d")
                fx.append(np.sqrt(np.maximum(W[("gmv2", w)] / n - mean * mean, 0.0)))
            for w in (7, 30, 90):
                nx.append(f"gmv_max_{w}d")
                fx.append(gmv_raw[:, max(t - w + 1, 0) : t + 1].max(axis=1))
            nx.append("gmv_max_life")
            fx.append(run_max_gmv[:, t])
            for w in (90, 365):
                n_span = float(min(w, t + 1))
                centre = t - (n_span - 1) / 2.0
                denom = n_span * (n_span * n_span - 1) / 12.0
                nx.append(f"loggmv_slope_{w}d")
                fx.append((W[("loggmv_t", w)] - centre * W[("loggmv", w)]) / denom)

            last_order = LAST["has_order"]
            nx.append("days_since_last_event")
            fx.append(t - LAST["active"][:, t])
            nx.append("days_since_last_order")
            fx.append(t - last_order[:, t])
            nx.append("days_since_last_cart")
            fx.append(t - LAST["has_cart"][:, t])
            nx.append("tenure_days")
            fx.append(np.where(CUM["active"][:, t] > 0, t - first_active, -1.0))
            nx.append("days_since_first_order")
            fx.append(np.where(CUM["has_order"][:, t] > 0, t - first_order, -1.0))
            nx.append("history_days")
            fx.append(np.full(nb, float(t + 1)))
            order_gap = float(min(365, t + 1)) / (W[("has_order", 365)] + 1.0)
            nx.append("order_gap_365d")
            fx.append(order_gap)
            nx.append("active_gap_90d")
            fx.append(float(min(90, t + 1)) / (W[("active", 90)] + 1.0))
            nx.append("recency_over_gap")
            fx.append((t - last_order[:, t]) / order_gap)
            last_t = last_order[:, t]
            prev_order = np.where(
                last_t >= 1, last_order[np.arange(nb), np.maximum(last_t - 1, 0)], -1
            )
            nx.append("days_since_prev_order")
            fx.append(np.where(prev_order >= 0, t - prev_order, N_DAYS))
            nx.append("last_order_gap")
            fx.append(np.where(prev_order >= 0, last_t - prev_order, N_DAYS))

            eps = 1.0
            for w in (30, 90, 365):
                nx.append(f"gmv_per_active_{w}d")
                fx.append(W[("gmv", w)] / (W[("active", w)] + eps))
                nx.append(f"basket_{w}d")
                fx.append(W[("gmv", w)] / (W[("to_ord", w)] + eps))
                nx.append(f"order_day_rate_{w}d")
                fx.append(W[("has_order", w)] / (W[("active", w)] + eps))
                nx.append(f"loggmv_per_order_{w}d")
                fx.append(W[("loggmv", w)] / (W[("has_order", w)] + eps))
                nx.append(f"loggmv_per_active_{w}d")
                fx.append(W[("loggmv", w)] / (W[("active", w)] + eps))
            for w in (7, 30, 90, 365):
                nx.append(f"active_rate_{w}d")
                fx.append(W[("active", w)] / float(min(w, t + 1)))
            for w in (30, 90):
                nx.append(f"cart_to_ord_{w}d")
                fx.append(W[("to_ord", w)] / (W[("to_cart", w)] + eps))
                nx.append(f"search_to_cart_{w}d")
                fx.append(W[("to_cart", w)] / (W[("searches", w)] + eps))
                nx.append(f"cat_gmv_share_{w}d")
                fx.append(W[("gmv_cat", w)] / (W[("gmv", w)] + eps))
                nx.append(f"cat_ord_share_{w}d")
                fx.append(W[("cat_to_ord", w)] / (W[("to_ord", w)] + eps))
            for col in ("gmv", "to_ord", "searches", "active", "loggmv"):
                nx.append(f"{col}_trend_7_30")
                fx.append(W[(col, 7)] / (W[(col, 30)] + eps))
                nx.append(f"{col}_trend_30_90")
                fx.append(W[(col, 30)] / (W[(col, 90)] + eps))
                nx.append(f"{col}_trend_90_365")
                fx.append(W[(col, 90)] / (W[(col, 365)] + eps))
                nx.append(f"{col}_m2")
                fx.append(W[(col, 60)] - W[(col, 30)])
                nx.append(f"{col}_m3")
                fx.append(W[(col, 90)] - W[(col, 60)])
            months = float(min(365, t + 1)) / 30.0
            nx.append("gmv_vs_yearly_rate")
            fx.append(W[("gmv", 30)] / (W[("gmv", 365)] / months + eps))
            nx.append("ord_vs_yearly_rate")
            fx.append(W[("to_ord", 30)] / (W[("to_ord", 365)] / months + eps))

            for col in EWM_COLS:
                for half_life in HALF_LIVES:
                    nx.append(f"ewm_{col}_{half_life}")
                    fx.append(EWM[(col, half_life)][t])
            for half_life in HALF_LIVES:
                nx.append(f"ewm_loggmv_per_active_{half_life}")
                fx.append(EWM[("loggmv", half_life)][t] / (EWM[("active", half_life)][t] + 0.01))
                nx.append(f"ewm_loggmv_per_order_{half_life}")
                fx.append(EWM[("loggmv", half_life)][t] / (EWM[("has_order", half_life)][t] + 0.01))
            for col in ("loggmv", "has_order", "active"):
                for fast, slow in ((7, 60), (14, 120), (30, 240)):
                    nx.append(f"ewm_{col}_ratio_{fast}_{slow}")
                    fx.append(EWM[(col, fast)][t] / (EWM[(col, slow)][t] + 0.01))

            block_x = np.column_stack([np.asarray(v, dtype=np.float32) for v in fx])
            NAMES_X = nx if NAMES_X is None else NAMES_X
            if key in XMAP:
                XMAP[key][base_row : base_row + nb] = block_x
            del block_x, fx

            if key in EMAP:
                ne, fe = [], []
                for col in (
                    "search_only",
                    "cat_only",
                    "both_ch",
                    "neither_ch",
                    "search_to_cart",
                    "cat_to_cart",
                ):
                    for w in WINDOWS:
                        ne.append(f"{col}_sum_{w}d")
                        fe.append(W[(col, w)])
                for w in (30, 90):
                    act = W[("active", w)] + eps
                    ne.append(f"search_only_share_{w}d")
                    fe.append(W[("search_only", w)] / act)
                    ne.append(f"cat_any_share_{w}d")
                    fe.append((W[("cat_only", w)] + W[("both_ch", w)]) / act)
                    ne.append(f"neither_share_{w}d")
                    fe.append(W[("neither_ch", w)] / act)
                    ne.append(f"cat_cart_share_{w}d")
                    fe.append(
                        W[("cat_to_cart", w)]
                        / (W[("search_to_cart", w)] + W[("cat_to_cart", w)] + eps)
                    )
                for w in (90, 365):
                    n = W[("has_order", w)]
                    nz = np.maximum(n, 1.0)
                    m1 = W[("ord_t", w)] / nz
                    var = np.maximum(W[("ord_t2", w)] / nz - m1 * m1, 0.0)
                    ne.append(f"order_pos_sd_{w}d")
                    fe.append(np.where(n > 1, np.sqrt(var), -1.0))
                    ne.append(f"order_centroid_recency_{w}d")
                    fe.append(np.where(n > 0, t - m1, float(N_DAYS)))
                    lg = W[("loggmv", w)]
                    lm = lg / nz
                    lvar = np.maximum(W[("loggmv2", w)] / nz - lm * lm, 0.0)
                    ne.append(f"loggmv_sd_{w}d")
                    fe.append(np.where(n > 1, np.sqrt(lvar), -1.0))
                co = CUM["has_order"]
                total = co[:, t]
                for k in (3, 5, 10):
                    target = total - (k - 1)
                    pos = np.argmax(co >= target[:, None], axis=1)
                    ne.append(f"days_since_order_{k}")
                    fe.append(np.where(total >= k, t - pos, float(N_DAYS)).astype(np.float32))
                idx = np.arange(nb)
                last_gmv = np.where(last_t >= 0, gmv_raw[idx, np.maximum(last_t, 0)], 0.0)
                prev_pos = last_order[idx, np.maximum(last_t - 1, 0)]
                prev_gmv = np.where(prev_pos >= 0, gmv_raw[idx, np.maximum(prev_pos, 0)], 0.0)
                ne.append("last_order_loggmv")
                fe.append(np.log1p(last_gmv))
                ne.append("prev_order_loggmv")
                fe.append(np.log1p(prev_gmv))
                mean_ord = W[("loggmv", 365)] / np.maximum(W[("has_order", 365)], 1.0)
                ne.append("last_over_mean_order")
                fe.append(np.log1p(last_gmv) / (mean_ord + 0.01))
                block_e = np.column_stack([np.asarray(v, dtype=np.float32) for v in fe])
                NAMES_E = ne if NAMES_E is None else NAMES_E
                EMAP[key][base_row : base_row + nb, :N_E_BASE] = block_e
                del block_e, fe
            del W
        del CUM, EWM, gmv_raw, run_max_gmv, LAST
        gc.collect()
        print(f"v2 v3 batch {b + 1}/{len(BOUNDS) - 1}", flush=True)

    for m in XMAP.values():
        m.flush()
    for a in X_MISSING:
        key = a.isoformat()
        del XMAP[key]
        (FEAT / "x" / f"X_{key}.npy.part").replace(FEAT / "x" / f"X_{key}.npy")
    (FEAT / "names_x.json").write_text(json.dumps(NAMES_X))
    np.save(
        FEAT / "keep_idx.npy",
        np.array([j for j, nm in enumerate(NAMES_X) if nm != "history_days"], dtype=np.int32),
    )

### Rank columns

In [ ]:
if E_MISSING:
    for m in EMAP.values():
        m.flush()
    rank_src = [NAMES_X.index(c) for c in RANK_COLS]
    for a in E_MISSING:
        key = a.isoformat()
        base = np.load(FEAT / "x" / f"X_{key}.npy", mmap_mode="r")
        emap = EMAP[key]
        for j, col in enumerate(rank_src):
            v = np.asarray(base[:, col], dtype=np.float64)
            n = len(v)
            order = np.argsort(v, kind="mergesort")
            sv = v[order]
            new = np.empty(n, dtype=bool)
            new[0] = True
            np.not_equal(sv[1:], sv[:-1], out=new[1:])
            grp = np.cumsum(new) - 1
            counts = np.bincount(grp)
            starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
            ranked = np.empty(n, dtype=np.float64)
            ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
            emap[:, N_E_BASE + j] = (ranked / n).astype(np.float32)
        emap.flush()
        del emap
        del EMAP[key]
        (FEAT / "e" / f"E_{key}.npy.part").replace(FEAT / "e" / f"E_{key}.npy")
        print("ranks", key, flush=True)
    NAMES_E_FULL = [*NAMES_E, *[f"rank_{c}" for c in RANK_COLS]]
    (FEAT / "names_e.json").write_text(json.dumps(NAMES_E_FULL, indent=1))

### Features

In [ ]:
N_BLOCK, BLOCK, N_WEEK, MISS, ORDER_LOOKBACK = 12, 30, 10, -1.0, 365
ORDER_NAMES = [
    "os_n_orders",
    "os_gap_mean",
    "os_gap_sd",
    "os_gap_max",
    "os_gap_min",
    "os_gap_le30",
    "os_gap_le60",
    "os_recency",
    "os_lgmv_mean",
    "os_lgmv_max",
    "os_lgmv_sd",
    "os_span",
    "os_rate",
    "os_rec_over_gap",
]


@numba.njit(parallel=True)
def order_stats(starts, ends, days, gmvs, ts, lookback, out):
    n_users = starts.shape[0]
    n_anchor = ts.shape[0]
    for u in numba.prange(n_users):
        lo, hi = starts[u], ends[u]
        for ai in range(n_anchor):
            t = ts[ai]
            n = 0
            s_gap = 0.0
            s_gap2 = 0.0
            g_max = 0.0
            g_min = 1e9
            le30 = 0.0
            le60 = 0.0
            s_lg = 0.0
            s_lg2 = 0.0
            lg_max = 0.0
            prev = -1
            first = -1
            last = -1
            for i in range(lo, hi):
                d = days[i]
                if d > t:
                    break
                if d <= t - lookback:
                    continue
                if first < 0:
                    first = d
                last = d
                lg = np.log1p(gmvs[i])
                s_lg += lg
                s_lg2 += lg * lg
                lg_max = max(lg_max, lg)
                if prev >= 0:
                    g = float(d - prev)
                    s_gap += g
                    s_gap2 += g * g
                    g_max = max(g_max, g)
                    g_min = min(g_min, g)
                    if g <= 30.0:
                        le30 += 1.0
                    if g <= 60.0:
                        le60 += 1.0
                prev = d
                n += 1
            o = out[ai, u]
            if n == 0:
                for j in range(14):
                    o[j] = MISS
                continue
            rec = float(t - last)
            o[0] = float(n)
            o[7] = rec
            o[8] = s_lg / n
            o[9] = lg_max
            o[10] = np.sqrt(max(s_lg2 / n - (s_lg / n) ** 2, 0.0))
            span = float(min(t - first, lookback))
            o[11] = span
            o[12] = float(n) / max(span, 1.0)
            k = n - 1
            if k <= 0:
                o[1] = MISS
                o[2] = MISS
                o[3] = MISS
                o[4] = MISS
                o[5] = MISS
                o[6] = MISS
                o[13] = MISS
                continue
            gm = s_gap / k
            o[1] = gm
            o[2] = np.sqrt(max(s_gap2 / k - gm * gm, 0.0))
            o[3] = g_max
            o[4] = g_min
            o[5] = le30 / k
            o[6] = le60 / k
            o[13] = rec / max(gm, 1.0)


NAMES_F = []
for tag in ("lgmv", "ord", "act", "has"):
    NAMES_F += [f"blk_{tag}_{k}" for k in range(N_BLOCK)]
NAMES_F += [
    "blk_lgmv_mean",
    "blk_lgmv_sd",
    "blk_lgmv_max",
    "blk_lgmv_min",
    "blk_lgmv_mean3",
    "blk_lgmv_mean6",
    "blk_lgmv_slope",
    "blk_lgmv_last_over_mean",
    "blk_has_rate",
    "blk_has_rate3",
    "blk_has_rate6",
    "blk_n_avail",
    "blk_lgmv_nz_mean",
    "blk_lgmv_gt_last",
]
for tag in ("act", "ord", "lgmv"):
    NAMES_F += [f"wk_{tag}_{w}" for w in range(N_WEEK)]
NAMES_F += [
    "streak_inactive_now",
    "streak_active_max_90",
    "streak_inactive_max_90",
    "streak_active_max_365",
]
NAMES_F += [f"dow_ord_{d}" for d in range(7)]
NAMES_F += [f"dow_act_{d}" for d in range(7)]
NAMES_F += ["dow_weekend_ord", "dow_weekend_act", "dow_ord_conc", "dow_act_conc"]
NAMES_F += ORDER_NAMES
N_F = len(NAMES_F)

In [ ]:
if F_MISSING:
    dow_of_day = ((np.arange(N_DAYS) + DATE_MIN.weekday()) % 7).astype(np.int64)
    FMAP = {}
    for a in F_MISSING:
        key = a.isoformat()
        FMAP[key] = np.lib.format.open_memmap(
            FEAT / "f" / f"F_{key}.npy.part", mode="w+", dtype=np.float32, shape=(N_USERS, N_F)
        )
    for b in range(len(BOUNDS) - 1):
        lo, hi = BOUNDS[b], BOUNDS[b + 1]
        grow = np.searchsorted(USER_IDS, UID[lo:hi])
        base_row = int(grow[0])
        grow = grow - base_row
        nb = int(grow[-1]) + 1
        gday = DAY_IDX[lo:hi]
        gmv = np.zeros((nb, N_DAYS), dtype=np.float32)
        gmv[grow, gday] = COLS["gmv"][lo:hi]
        to_ord = np.zeros((nb, N_DAYS), dtype=np.float32)
        to_ord[grow, gday] = COLS["to_ord"][lo:hi]
        active = np.zeros((nb, N_DAYS), dtype=np.float32)
        active[grow, gday] = COLS["active"][lo:hi]
        has_ord = (to_ord > 0).astype(np.float32)
        cum_gmv = np.cumsum(gmv, axis=1, dtype=np.float64)
        cum_ord = np.cumsum(to_ord, axis=1, dtype=np.float32)
        cum_act = np.cumsum(active, axis=1, dtype=np.float32)
        cum_has = np.cumsum(has_ord, axis=1, dtype=np.float32)

        r, c = np.nonzero(has_ord)
        ostats = np.empty((len(ANCHOR_T), nb, 14), dtype=np.float32)
        order_stats(
            np.searchsorted(r, np.arange(nb)).astype(np.int64),
            np.searchsorted(r, np.arange(nb), side="right").astype(np.int64),
            c.astype(np.int64),
            gmv[r, c].astype(np.float64),
            np.asarray(ANCHOR_T, dtype=np.int64),
            np.int64(ORDER_LOOKBACK),
            ostats,
        )

        for ai, a in enumerate(ALL_DATES):
            key = a.isoformat()
            if key not in FMAP:
                continue
            t = ANCHOR_T[ai]
            cols = []
            blk_lg, blk_or, blk_ac, blk_hs = [], [], [], []
            n_av = 0
            for k in range(N_BLOCK):
                w0, w1 = k * BLOCK, k * BLOCK + BLOCK - 1
                if t - w1 >= 0:
                    n_av += 1
                    p0 = t - w1 - 1
                    blk_lg.append(
                        np.log1p(cum_gmv[:, t - w0] - (cum_gmv[:, p0] if p0 >= 0 else 0.0)).astype(
                            np.float32
                        )
                    )
                    blk_or.append(
                        np.log1p(cum_ord[:, t - w0] - (cum_ord[:, p0] if p0 >= 0 else 0.0))
                    )
                    blk_ac.append(cum_act[:, t - w0] - (cum_act[:, p0] if p0 >= 0 else 0.0))
                    blk_hs.append(
                        ((cum_has[:, t - w0] - (cum_has[:, p0] if p0 >= 0 else 0.0)) > 0).astype(
                            np.float32
                        )
                    )
                else:
                    z = np.full(nb, MISS, dtype=np.float32)
                    blk_lg.append(z)
                    blk_or.append(z)
                    blk_ac.append(z)
                    blk_hs.append(z)
            cols += blk_lg + blk_or + blk_ac + blk_hs
            lg = np.stack(blk_lg[:n_av])
            hs = np.stack(blk_hs[:n_av])
            mblk = lg.mean(axis=0)
            xax = np.arange(n_av, dtype=np.float32)
            xc = xax - xax.mean()
            cols += [
                mblk,
                lg.std(axis=0),
                lg.max(axis=0),
                lg.min(axis=0),
                lg[: min(3, n_av)].mean(axis=0),
                lg[: min(6, n_av)].mean(axis=0),
                (lg * xc[:, None]).sum(axis=0) / max(float((xc * xc).sum()), 1.0),
                lg[0] / (mblk + 0.01),
                hs.mean(axis=0),
                hs[: min(3, n_av)].mean(axis=0),
                hs[: min(6, n_av)].mean(axis=0),
                np.full(nb, float(n_av), dtype=np.float32),
                lg.sum(axis=0) / np.maximum(hs.sum(axis=0), 1.0),
                (lg > lg[0]).sum(axis=0).astype(np.float32),
            ]
            for cum, as_log in ((cum_act, False), (cum_ord, True), (cum_gmv, True)):
                for w in range(N_WEEK):
                    w0, w1 = w * 7, w * 7 + 6
                    p0 = t - w1 - 1
                    v = cum[:, t - w0] - (cum[:, p0] if p0 >= 0 else 0.0)
                    cols.append(np.log1p(np.maximum(v, 0.0)) if as_log else v)

            inact = 1.0 - active[:, max(t - 89, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_i = np.zeros(nb, dtype=np.float32)
            for j in range(inact.shape[1]):
                run = (run + 1.0) * inact[:, j]
                best_i = np.maximum(best_i, run)
            run_i = run
            act90 = active[:, max(t - 89, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_a90 = np.zeros(nb, dtype=np.float32)
            for j in range(act90.shape[1]):
                run = (run + 1.0) * act90[:, j]
                best_a90 = np.maximum(best_a90, run)
            act365 = active[:, max(t - 364, 0) : t + 1]
            run = np.zeros(nb, dtype=np.float32)
            best_a365 = np.zeros(nb, dtype=np.float32)
            for j in range(act365.shape[1]):
                run = (run + 1.0) * act365[:, j]
                best_a365 = np.maximum(best_a365, run)
            cols += [run_i, best_a90, best_i, best_a365]

            d0 = max(t - 364, 0)
            dows = dow_of_day[d0 : t + 1]
            ord_w = has_ord[:, d0 : t + 1]
            act_w = active[:, d0 : t + 1]
            tot_o = np.maximum(ord_w.sum(axis=1), 1.0)
            tot_a = np.maximum(act_w.sum(axis=1), 1.0)
            dow_o = [ord_w[:, dows == d].sum(axis=1) / tot_o for d in range(7)]
            dow_a = [act_w[:, dows == d].sum(axis=1) / tot_a for d in range(7)]
            cols += dow_o + dow_a
            cols += [
                dow_o[5] + dow_o[6],
                dow_a[5] + dow_a[6],
                np.stack(dow_o).max(axis=0),
                np.stack(dow_a).max(axis=0),
            ]
            block = np.column_stack([np.asarray(v, dtype=np.float32) for v in cols])
            block = np.concatenate([block, ostats[ai]], axis=1)
            FMAP[key][base_row : base_row + nb] = np.nan_to_num(
                block, nan=MISS, posinf=1e9, neginf=MISS
            )
            del block, cols
        del gmv, to_ord, active, has_ord, cum_gmv, cum_ord, cum_act, cum_has, ostats
        gc.collect()
        print(f"v4 batch {b + 1}/{len(BOUNDS) - 1}", flush=True)

    for m in FMAP.values():
        m.flush()
    for a in F_MISSING:
        key = a.isoformat()
        del FMAP[key]
        (FEAT / "f" / f"F_{key}.npy.part").replace(FEAT / "f" / f"F_{key}.npy")
    (FEAT / "names_f.json").write_text(json.dumps(NAMES_F, indent=1))

if NEED_PREP:
    del COLS, UID, DAY_IDX
    gc.collect()

### Context

In [ ]:
NAMES_X = json.loads((FEAT / "names_x.json").read_text())
NAMES_E_FULL = json.loads((FEAT / "names_e.json").read_text())
NAMES_F = json.loads((FEAT / "names_f.json").read_text())
KEEP_IDX = np.load(FEAT / "keep_idx.npy")
BASE_NAMES = [NAMES_X[j] for j in KEEP_IDX] + NAMES_E_FULL
FEATURE_NAMES = BASE_NAMES + NAMES_F
N_BASE = len(BASE_NAMES)
CAP_IDX = np.array([BASE_NAMES.index(c) for c in CAP_COLS])

MU, MU_POS = {}, {}
for a in ANCHORS:
    i = ANCHOR_IDX[a]
    m = COHORT[i]
    raw = TARGETS[i][m]
    ly = np.log1p(raw)
    MU[a] = float(ly.mean())
    MU_POS[a] = float(ly[raw > 0].mean())

### Rank gauss

In [ ]:
for a in [*ANCHORS, SUBMIT]:
    rg_path = FEAT / "rg" / f"RG_{a}.npy"
    if rg_path.exists():
        continue
    xa = np.asarray(np.load(FEAT / "x" / f"X_{a}.npy", mmap_mode="r"))[:, KEEP_IDX]
    ea = np.asarray(np.load(FEAT / "e" / f"E_{a}.npy", mmap_mode="r"))
    fa = np.asarray(np.load(FEAT / "f" / f"F_{a}.npy", mmap_mode="r"))
    xa = np.concatenate([xa, ea], axis=1)
    xa[:, CAP_IDX] = np.minimum(xa[:, CAP_IDX], CAP_AT)
    xa = np.concatenate([xa, fa], axis=1)
    out_rg = np.empty(xa.shape, dtype=np.float32)
    n = xa.shape[0]
    for j in range(xa.shape[1]):
        v = np.asarray(xa[:, j], dtype=np.float64)
        order = np.argsort(v, kind="mergesort")
        sv = v[order]
        new = np.empty(n, dtype=bool)
        new[0] = True
        np.not_equal(sv[1:], sv[:-1], out=new[1:])
        grp = np.cumsum(new) - 1
        counts = np.bincount(grp)
        starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
        ranked = np.empty(n, dtype=np.float64)
        ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
        r = (ranked / n + 0.5 / n).clip(1e-6, 1.0 - 1e-6)
        out_rg[:, j] = ndtri(r).astype(np.float32)
    part = FEAT / "rg" / f"RG_{a}.npy.part"
    with part.open("wb") as fh:
        np.save(fh, out_rg.astype(np.float16))
    part.replace(rg_path)
    del xa, ea, fa, out_rg
    gc.collect()
    print("rank gauss", a, flush=True)

### Daily tensors

### Two feature spaces

In [ ]:
SPACES = {
    "pre": {
        "dir": PRE,
        "window_len": False,
        "order_lookback": 10**9,
        "prev_order": False,
        "tie_average": False,
        "scale_days": None,
        "cuda_seed": False,
    },
    "repaired": {
        "dir": FEAT,
        "window_len": True,
        "order_lookback": 365,
        "prev_order": True,
        "tie_average": True,
        "scale_days": 90,
        "cuda_seed": True,
    },
}


def wlen(t, w, repaired):
    return float(min(w, t + 1)) if repaired else float(w)


def avg_rank(v, tie_average):
    v = np.asarray(v, dtype=np.float64).ravel()
    n = len(v)
    if not tie_average:
        out = np.empty(n, dtype=np.float64)
        out[np.argsort(v, kind="stable")] = np.arange(n, dtype=np.float64)
        return out / n
    order = np.argsort(v, kind="mergesort")
    s = v[order]
    new = np.empty(n, dtype=bool)
    new[0] = True
    np.not_equal(s[1:], s[:-1], out=new[1:])
    grp = np.cumsum(new) - 1
    counts = np.bincount(grp)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    out = np.empty(n, dtype=np.float64)
    out[order] = starts[grp] + (counts[grp] - 1) / 2.0
    return out / n


def rank_gauss_cols(x, tie_average):
    out = np.empty(x.shape, dtype=np.float32)
    n = x.shape[0]
    for j in range(x.shape[1]):
        r = (avg_rank(x[:, j], tie_average) + 0.5 / n).clip(1e-6, 1.0 - 1e-6)
        out[:, j] = ndtri(r).astype(np.float32)
    return out


def torch_seed(seed, cuda_seed):
    torch.manual_seed(seed)
    if cuda_seed:
        torch.cuda.manual_seed_all(seed)

### First activity day

In [ ]:
FIRST_DAY_PATH = SETUP / "first_day.npy"
if not FIRST_DAY_PATH.exists():
    rows_fd = np.searchsorted(USER_IDS, data["user_id"].to_numpy())
    day_fd = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    np.save(FIRST_DAY_PATH, day_fd[np.searchsorted(rows_fd, np.arange(N_USERS))])
    del rows_fd, day_fd
    gc.collect()
FIRST_DAY = np.load(FIRST_DAY_PATH)
print(f"first activity day: min {FIRST_DAY.min()} max {FIRST_DAY.max()}")

### v1 window aggregates

In [ ]:
V1_VALUE_COLS = ["gmv", "searches", "to_cart", "to_ord"]
V1_WINDOWS = [7, 14, 30, 60, 90]
V1_MIN_HISTORY = 90
V1_STRIDE = 14
V1_ANCHORS = sorted(
    (DATE_MAX - timedelta(days=HORIZON_DAYS) - timedelta(days=i * V1_STRIDE)).isoformat()
    for i in range(4)
)
V1_ALL = [*V1_ANCHORS, SUBMIT]
V1_NAMES_PATH = V1 / "names.json"

if not all((V1 / f"X_{a}.npy").exists() for a in V1_ALL):
    uid_v1 = data["user_id"].to_numpy()
    day_v1 = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    src_v1 = {c: data[c].to_numpy().astype(np.float64) for c in V1_VALUE_COLS}
    rows_v1 = np.searchsorted(USER_IDS, uid_v1)
    first_v1 = day_v1[np.searchsorted(rows_v1, np.arange(N_USERS))]
    last_v1 = np.maximum.reduceat(day_v1, np.searchsorted(rows_v1, np.arange(N_USERS)))
    v1_names = None
    for a in V1_ALL:
        path = V1 / f"X_{a}.npy"
        if path.exists():
            continue
        t = (date.fromisoformat(a) - DATE_MIN).days
        names, feats = [], []
        for w in V1_WINDOWS:
            sel = (day_v1 > t - w) & (day_v1 <= t)
            r = rows_v1[sel]
            n_act = np.bincount(r, minlength=N_USERS).astype(np.float64)
            for col in V1_VALUE_COLS:
                v = src_v1[col][sel]
                s = np.bincount(r, weights=v, minlength=N_USERS)
                s2 = np.bincount(r, weights=v * v, minlength=N_USERS)
                mx = np.zeros(N_USERS, dtype=np.float64)
                np.maximum.at(mx, r, v)
                den = np.maximum(n_act, 1.0)
                mean = s / den
                names += [
                    f"{col}_sum_{w}d",
                    f"{col}_max_{w}d",
                    f"{col}_mean_{w}d",
                    f"{col}_std_{w}d",
                ]
                feats += [s, mx, mean, np.sqrt(np.maximum(s2 / den - mean * mean, 0.0))]
            names.append(f"active_days_{w}d")
            feats.append(n_act)
        for tag, col in (("event", None), ("order", "to_ord"), ("cart", "to_cart")):
            sel = day_v1 <= t if col is None else (day_v1 <= t) & (src_v1[col] > 0)
            last = np.full(N_USERS, -1.0)
            np.maximum.at(last, rows_v1[sel], day_v1[sel].astype(np.float64))
            names.append(f"days_since_last_{tag}")
            feats.append(np.where(last >= 0, t - last, float(N_DAYS)))
        v1_names = names
        np.save(path, np.column_stack(feats).astype(np.float32))
        print("v1 features", a, flush=True)
    if v1_names is not None:
        V1_NAMES_PATH.write_text(json.dumps(v1_names, indent=1))
    del uid_v1, day_v1, src_v1, rows_v1
    gc.collect()

V1_NAMES = json.loads(V1_NAMES_PATH.read_text())
print(f"v1 space {len(V1_NAMES)} columns over {len(V1_ALL)} anchors")

### Chronos series

In [ ]:
CH_VALUE_COLS = ["gmv", "to_ord", "to_cart", "searches"]
CH_COVARIATES = ["to_ord", "to_cart", "searches", "active"]
CH_ANCHORS = [*V1_ANCHORS, SUBMIT]

if not (CHRONOS / "meta.json").exists():
    uid_c = data["user_id"].to_numpy()
    rows_c = np.searchsorted(USER_IDS, uid_c)
    day_c = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    cols_c = {c: data[c].to_numpy().astype(np.float64) for c in CH_VALUE_COLS}
    cols_c["active"] = np.ones(data.height, dtype=np.float64)
    for bin_days in (10, 30):
        for col in [*CH_VALUE_COLS, "active"]:
            path = CHRONOS / f"b{bin_days}_{col}.npy"
            if path.exists():
                continue
            n_bin = (N_DAYS + bin_days - 1) // bin_days
            b = day_c // bin_days
            flat = np.bincount(rows_c * n_bin + b, weights=cols_c[col], minlength=N_USERS * n_bin)
            np.save(path, flat.reshape(N_USERS, n_bin).astype(np.float32))
            print(f"chronos bins {bin_days}d {col}", flush=True)
    (CHRONOS / "meta.json").write_text(
        json.dumps(
            {
                "bin_days": [10, 30],
                "value_cols": CH_VALUE_COLS,
                "covariates": CH_COVARIATES,
                "date_min": DATE_MIN.isoformat(),
                "n_days": N_DAYS,
            },
            indent=2,
        )
    )
    del uid_c, rows_c, day_c, cols_c
    gc.collect()
print("chronos series ready")

### Calendar level shift

In [ ]:
DAILY_IDX = (
    data
    .group_by("event_date")
    .agg(pl.len().alias("dau"), pl.col("gmv").sum().alias("gmv"))
    .sort("event_date")
)


def window_index(start, end):
    w = DAILY_IDX.filter(pl.col("event_date").is_between(start, end))
    return float(w["gmv"].sum()) / float(w["dau"].sum())


def calendar_shift(anchor_list, damping=1.0):
    persistence = window_index(DATE_MAX - timedelta(days=HORIZON_DAYS - 1), DATE_MAX)
    year_before = window_index(
        DATE_MAX - timedelta(days=365 + HORIZON_DAYS - 1), DATE_MAX - timedelta(days=365)
    )
    year_target = window_index(
        DATE_MAX - timedelta(days=364), DATE_MAX - timedelta(days=365 - HORIZON_DAYS)
    )
    submit_index = persistence * (year_target / year_before)
    train_mean = float(
        np.mean([
            np.log(
                window_index(
                    date.fromisoformat(a) + timedelta(days=1),
                    date.fromisoformat(a) + timedelta(days=HORIZON_DAYS),
                )
            )
            for a in anchor_list
        ])
    )
    shift = damping * (float(np.log(submit_index)) - train_mean)
    print(f"train mean log index {train_mean}  submit forecast {submit_index}")
    print(f"seasonal drift {submit_index / persistence}  level shift {shift}")
    return shift


V2_SHIFT = calendar_shift(V1_ANCHORS)

### Seasonal ratios

In [ ]:
LEVELS = {r["anchor"]: r for r in json.loads((SETUP / "levels.json").read_text())}
SEASON_PATH = SETUP / "season.json"
if not SEASON_PATH.exists():
    uid_s2 = data["user_id"].to_numpy()
    rows_s2 = np.searchsorted(USER_IDS, uid_s2)
    day_s2 = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    gmv_s2 = data["gmv"].to_numpy().astype(np.float64)
    fixed = data.filter(pl.col("event_date") <= date(2025, 1, 14))["user_id"].unique().to_numpy()
    fixed_rows = np.searchsorted(USER_IDS, fixed)
    season = {"n_fixed_cohort": len(fixed)}
    for label, lo, hi in (
        ("jan_2025", date(2025, 1, 15), date(2025, 2, 13)),
        ("feb_2025", date(2025, 2, 14), date(2025, 3, 15)),
    ):
        a0, b0 = (lo - DATE_MIN).days, (hi - DATE_MIN).days
        m = (day_s2 >= a0) & (day_s2 <= b0)
        y = np.bincount(rows_s2[m], weights=gmv_s2[m], minlength=N_USERS)[fixed_rows]
        act = np.bincount(rows_s2[m], minlength=N_USERS)[fixed_rows] > 0
        ly = np.log1p(y)
        season[label] = {
            "p_active": float(act.mean()),
            "p_order_given_active": float((y > 0).sum() / act.sum()),
            "e_log_given_positive": float(ly[y > 0].mean()),
            "mean_log1p": float(ly.mean()),
        }
    season["ratio_p_order_given_active"] = (
        season["feb_2025"]["p_order_given_active"] / season["jan_2025"]["p_order_given_active"]
    )
    season["ratio_e_log_given_positive"] = (
        season["feb_2025"]["e_log_given_positive"] / season["jan_2025"]["e_log_given_positive"]
    )
    SEASON_PATH.write_text(json.dumps(season, indent=2))
    del uid_s2, rows_s2, day_s2, gmv_s2
    gc.collect()
SEASON = json.loads(SEASON_PATH.read_text())
print(
    json.dumps(SEASON["ratio_p_order_given_active"]),
    json.dumps(SEASON["ratio_e_log_given_positive"]),
)

In [ ]:
SCALE_DAYS = 90
CHANNELS9 = [
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "cat",
    "active",
]
CHANNELS10 = [
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "search",
    "cat",
    "active",
]
LOG_CHANNELS = {
    "gmv_search",
    "gmv_cat",
    "searches",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
}
N_CAL = 7

for tag, channels in (("daily9", CHANNELS9), ("daily10", CHANNELS10)):
    tensor_path = SEQ / f"{tag}_f16.npy"
    if tensor_path.exists():
        continue
    uid_s = data["user_id"].to_numpy()
    day_s = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    chan = {}
    for name in channels:
        if name == "active":
            v = np.ones(data.height, dtype=np.float32)
        elif name == "search":
            v = (data["searches"].to_numpy().astype(np.float32) > 0).astype(np.float32)
        else:
            v = data[name].to_numpy().astype(np.float32)
        chan[name] = np.log1p(v) if name in LOG_CHANNELS else v
    scale_days = min(SCALE_DAYS, N_DAYS)
    fit = day_s < scale_days
    n_cells = N_USERS * scale_days
    scale = {}
    for name in channels:
        v = chan[name][fit]
        mean = v.sum() / n_cells
        var = (v * v).sum() / n_cells - mean * mean
        scale[name] = float(np.sqrt(max(var, 1e-12)))
    out_t = np.lib.format.open_memmap(
        tensor_path.with_suffix(".npy.part"),
        mode="w+",
        dtype=np.float16,
        shape=(N_USERS, N_DAYS, len(channels)),
    )
    seq_bounds = [*np.searchsorted(uid_s, USER_IDS[::USER_BATCH]).tolist(), len(uid_s)]
    for b in range(len(seq_bounds) - 1):
        lo, hi = seq_bounds[b], seq_bounds[b + 1]
        rows = np.searchsorted(USER_IDS, uid_s[lo:hi])
        base_row = int(rows[0])
        rows = rows - base_row
        nb = int(rows[-1]) + 1
        block = np.zeros((nb, N_DAYS, len(channels)), dtype=np.float32)
        for ci, name in enumerate(channels):
            block[rows, day_s[lo:hi], ci] = chan[name][lo:hi] / scale[name]
        out_t[base_row : base_row + nb] = block.astype(np.float16)
        del block
        print(f"{tag} batch {b + 1}/{len(seq_bounds) - 1}", flush=True)
    out_t.flush()
    del out_t
    tensor_path.with_suffix(".npy.part").replace(tensor_path)
    (SEQ / f"{tag}_meta.json").write_text(
        json.dumps(
            {
                "channels": channels,
                "scale": scale,
                "n_users": N_USERS,
                "n_days": N_DAYS,
                "scale_days": scale_days,
                "date_min": DATE_MIN.isoformat(),
            },
            indent=2,
        )
    )
    del chan, uid_s, day_s
    gc.collect()
    print(tag)

CAL_PATH = SEQ / "calendar.npy"
if not CAL_PATH.exists():
    di = (
        data
        .group_by("event_date")
        .agg(pl.len().alias("dau"), pl.col("gmv").sum().alias("gmv"))
        .sort("event_date")
    )
    idx = np.zeros(N_DAYS, dtype=np.float64)
    dpos = (
        di
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    idx[dpos] = di["gmv"].to_numpy() / np.maximum(di["dau"].to_numpy(), 1)
    li = np.log1p(idx)
    li = (li - li.mean()) / max(li.std(), 1e-9)
    tt = np.arange(N_DAYS, dtype=np.float64)
    dow = (tt + DATE_MIN.weekday()) % 7
    doy = (tt + DATE_MIN.timetuple().tm_yday - 1) % 365.25
    cal = np.column_stack([
        li,
        np.sin(2 * np.pi * dow / 7),
        np.cos(2 * np.pi * dow / 7),
        np.sin(2 * np.pi * doy / 365.25),
        np.cos(2 * np.pi * doy / 365.25),
        tt / N_DAYS,
        np.ones(N_DAYS),
    ]).astype(np.float32)
    np.save(CAL_PATH, cal)

In [ ]:
data = None
gc.collect()

### Members

In [ ]:
SEED_POOL = [
    42,
    7,
    2024,
    555,
    31337,
    101,
    202,
    303,
    909,
    1234,
    5678,
    4242,
    777,
    1111,
    2222,
    3333,
    8080,
    6060,
    4040,
    2020,
]
RG_KINDS = ("mlp", "mlpce", "tabm", "mlpord")
FOLDS = [*VAL_ANCHORS, SUBMIT]
FIT_ANCHORS = ["2025-09-24", "2025-10-08", "2025-10-22", "2025-12-03"]
EVAL_ANCHORS = ["2025-12-17", "2025-12-31", "2026-01-14"]
N_TRAIN_ANCHORS = 10

SEQ_ARCH = {
    "v3_seq": {
        "patch": 8,
        "n_patch": 22,
        "dim": 192,
        "layers": 6,
        "heads": 6,
        "ff": 2,
        "epochs": 8,
        "valid": False,
        "side": "v3",
        "pool13": False,
        "side_hidden": 256,
        "head_hidden": 256,
        "final_norm": True,
        "mix_anchors": True,
    },
    "seq_b": {
        "patch": 7,
        "n_patch": 52,
        "dim": 256,
        "layers": 6,
        "heads": 8,
        "ff": 3,
        "epochs": 14,
        "valid": True,
        "side": "all",
        "pool13": True,
        "side_hidden": 512,
        "head_hidden": 512,
        "final_norm": True,
        "mix_anchors": True,
    },
}
SEQ2_ARCH = {
    "patch": 7,
    "n_patch": 52,
    "dim": 256,
    "layers": 6,
    "heads": 8,
    "ff": 3,
    "side_hidden": 512,
    "head_hidden": 512,
    "final_norm": True,
    "mix_anchors": True,
}
V3_SIDE_COLS = [
    "days_since_last_event",
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "active_rate_30d",
    "active_rate_90d",
    "order_day_rate_30d",
    "loggmv_per_order_365d",
    "ewm_loggmv_60",
    "ewm_has_order_60",
    "gmv_trend_7_30",
    "gmv_trend_30_90",
]

MEMBERS = {
    "catf_a": {"kind": "cat", "v4": True, "iters": 1000, "lr": 0.03, "depth": 8, "l2": 30.0},
    "catf_b": {"kind": "cat", "v4": True, "iters": 1500, "lr": 0.03, "depth": 6, "l2": 6.0},
    "catf_d": {"kind": "cat", "v4": False, "iters": 1500, "lr": 0.03, "depth": 8, "l2": 100.0},
    "catf_f": {
        "kind": "cat",
        "v4": True,
        "iters": 1200,
        "lr": 0.03,
        "depth": 10,
        "l2": 30.0,
        "extra": {"grow_policy": "Lossguide", "max_leaves": 192, "min_data_in_leaf": 128},
    },
    "cat_a": {"kind": "cat", "v4": True, "iters": 3000, "lr": 0.03, "depth": 8},
    "cat_b": {
        "kind": "cat",
        "v4": True,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
    },
    "xgbf_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 150,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgbf_b": {
        "kind": "xgb",
        "v4": False,
        "rounds": 250,
        "lr": 0.02,
        "depth": 8,
        "colsample": 0.7,
        "mcw": 100.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgb_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 2500,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
    },
    "lgbf_a": {
        "kind": "lgb",
        "v4": False,
        "rounds": 150,
        "lr": 0.03,
        "leaves": 255,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgbf_b": {
        "kind": "lgb",
        "v4": True,
        "rounds": 250,
        "lr": 0.02,
        "leaves": 127,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgb_a": {"kind": "lgb", "v4": False, "rounds": 2500, "lr": 0.03, "leaves": 255},
    "mlpf_a": {
        "kind": "mlp",
        "v4": True,
        "epochs": 4,
        "width": 2560,
        "drop": 0.35,
        "lr": 2e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_b": {
        "kind": "mlp",
        "v4": False,
        "epochs": 3,
        "width": 1536,
        "drop": 0.2,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_c": {"kind": "mlpce", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "mlpf_d": {
        "kind": "mlp",
        "v4": True,
        "epochs": 3,
        "width": 1024,
        "drop": 0.35,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_o": {"kind": "mlpord", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "tabm_a": {
        "kind": "tabm",
        "v4": True,
        "epochs": 8,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": False,
        "n_oof": 3,
        "n_sub": 6,
    },
    "tabm_b": {
        "kind": "tabm",
        "v4": False,
        "epochs": 4,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": True,
        "n_oof": 2,
        "n_sub": 4,
    },
    "v3_cat": {
        "kind": "cat",
        "v4": False,
        "iters": 3000,
        "lr": 0.03,
        "depth": 8,
        "l2": 6.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_cat2": {
        "kind": "cat",
        "v4": False,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
        "rs": 2.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_mlp": {
        "kind": "mlpv3",
        "v4": False,
        "epochs": 20,
        "width": 1024,
        "drop": 0.15,
        "lr": 3e-3,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_seq": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 2},
    "seq_b": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 3},
    "seq_c": {
        "kind": "seq2",
        "v4": False,
        "epochs": 3,
        "lr": 1e-3,
        "head_lr_mult": 1.0,
        "n_oof": 1,
        "n_sub": 3,
    },
}
for m in ("catf_d", "catf_b", "xgbf_a", "catf_f", "xgbf_b", "catf_a", "lgbf_b", "lgbf_a", "cat_a"):
    MEMBERS[f"{m}_sp"] = {**MEMBERS[m], "min_gap": 28}
for cfg in MEMBERS.values():
    cfg.setdefault("min_gap", 0)
    cfg.setdefault("n_oof", 2)
    cfg.setdefault("n_sub", 3)

MEMBER_ORDER = sorted(MEMBERS)
print(", ".join(MEMBER_ORDER))

### Torch modules

In [ ]:
TABM_K, PLE_BINS, PLE_DIM = 32, 8, 8
HL_LO, HL_HI, HL_BINS = -2.8, 9.2, 48
HL_EDGES = np.linspace(HL_LO, HL_HI, HL_BINS + 1)
HL_CENTERS = 0.5 * (HL_EDGES[:-1] + HL_EDGES[1:])
HL_SIGMA = 0.75 * (HL_EDGES[1] - HL_EDGES[0])
ORD_LO, ORD_HI, ORD_BINS = -2.8, 9.2, 32
ORD_EDGES = np.linspace(ORD_LO, ORD_HI, ORD_BINS + 1)
ORD_CENTERS = 0.5 * (ORD_EDGES[:-1] + ORD_EDGES[1:])


class LinearBE(nn.Module):
    def __init__(self, d_in, d_out, k, first):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(d_in, d_out))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        sign = torch.randint(0, 2, (k, d_in), dtype=torch.float32) * 2.0 - 1.0
        self.r = nn.Parameter(sign if first else torch.ones(k, d_in))
        self.s = nn.Parameter(torch.ones(k, d_out))
        self.bias = nn.Parameter(torch.zeros(k, d_out))

    def forward(self, x):
        return ((x * self.r) @ self.weight) * self.s + self.bias


class PLE(nn.Module):
    def __init__(self, edges):
        super().__init__()
        d, n_edge = edges.shape
        t_bins = n_edge - 1
        self.register_buffer("lo", edges[:, :-1].contiguous())
        self.register_buffer("width", (edges[:, 1:] - edges[:, :-1]).clamp_min(1e-6))
        self.weight = nn.Parameter(torch.randn(d, t_bins, PLE_DIM) * (1.0 / t_bins**0.5))
        self.bias = nn.Parameter(torch.zeros(d, PLE_DIM))
        self.out_dim = d * PLE_DIM

    def forward(self, x):
        t = ((x[..., None] - self.lo) / self.width).clamp(0.0, 1.0)
        return (torch.einsum("bdt,dte->bde", t, self.weight) + self.bias).flatten(1)


class TabM(nn.Module):
    def __init__(self, d_in, k, width, blocks, drop, n_out, edges=None):
        super().__init__()
        self.k = k
        self.emb = PLE(edges) if edges is not None else None
        d = self.emb.out_dim if self.emb is not None else d_in
        self.layers = nn.ModuleList([
            LinearBE(d if i == 0 else width, width, k, i == 0) for i in range(blocks)
        ])
        self.drop = nn.Dropout(drop)
        self.head = LinearBE(width, n_out, k, False)

    def forward(self, x):
        if self.emb is not None:
            x = self.emb(x)
        h = x[:, None].expand(-1, self.k, -1)
        for lin in self.layers:
            h = self.drop(nn.functional.gelu(lin(h)))
        return self.head(h)


class Seq(nn.Module):
    def __init__(self, arch, n_side, n_ch):
        super().__init__()
        dim = arch["dim"]
        self.valid = bool(arch["valid"])
        self.pool13 = bool(arch["pool13"])
        n_extra = 2 if self.valid else 1
        n_pool = 5 if self.pool13 else 4
        self.proj = nn.Linear(2 * n_ch + n_extra, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )
        self.attn = nn.Linear(dim, 1)
        side_layers = [nn.Linear(n_side, arch["side_hidden"]), nn.GELU()]
        if self.valid:
            side_layers.append(nn.Dropout(0.15))
        side_layers.append(nn.Linear(arch["side_hidden"], dim))
        self.side = nn.Sequential(*side_layers)
        self.head = nn.Sequential(
            nn.Linear(n_pool * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.enc(self.proj(seq) + self.pos)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1)]
        if self.pool13:
            pooled.append(h[:, -13:].mean(1))
        pooled += [(h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]


class Encoder(nn.Module):
    def __init__(self, arch, n_feat):
        super().__init__()
        dim = arch["dim"]
        self.proj = nn.Linear(n_feat, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )

    def forward(self, seq, mask=None):
        h = self.proj(seq)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        return self.enc(h + self.pos)


class SeqNet(nn.Module):
    def __init__(self, arch, n_feat, n_side):
        super().__init__()
        dim = arch["dim"]
        self.encoder = Encoder(arch, n_feat)
        self.attn = nn.Linear(dim, 1)
        self.side = nn.Sequential(
            nn.Linear(n_side, arch["side_hidden"]),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(arch["side_hidden"], dim),
        )
        self.head = nn.Sequential(
            nn.Linear(5 * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.encoder(seq)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1), h[:, -13:].mean(1), (h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]

### Fitters

In [ ]:
from catboost import CatBoostRegressor, Pool
import lightgbm as lgb
import xgboost as xgb

MODEL_EXT = {
    "cat": "cbm",
    "xgb": "ubj",
    "lgb": "txt",
    "mlp": "pt",
    "mlpce": "pt",
    "mlpord": "pt",
    "mlpv3": "pt",
    "tabm": "pt",
    "seq": "pt",
    "seq2": "pt",
}


def fold_anchors(va, n_train=N_TRAIN_ANCHORS, min_gap=0):
    if va == SUBMIT:
        eligible = list(ANCHORS)
    else:
        cut = date.fromisoformat(va) - timedelta(days=HORIZON_DAYS)
        eligible = [a for a in ANCHORS if date.fromisoformat(a) <= cut]
    if min_gap <= 0:
        return eligible[-n_train:]
    picked = []
    for a in reversed(eligible):
        if len(picked) >= n_train:
            break
        if picked and (date.fromisoformat(picked[-1]) - date.fromisoformat(a)).days < min_gap:
            continue
        picked.append(a)
    return sorted(picked)


def load_space_x(root, anchor):
    x = np.asarray(np.load(root / "x" / f"X_{anchor}.npy", mmap_mode="r"))[:, KEEP_IDX]
    x = np.concatenate(
        [x, np.asarray(np.load(root / "e" / f"E_{anchor}.npy", mmap_mode="r"))], axis=1
    )
    x[:, CAP_IDX] = np.minimum(x[:, CAP_IDX], CAP_AT)
    return np.concatenate(
        [x, np.asarray(np.load(root / "f" / f"F_{anchor}.npy", mmap_mode="r"))], axis=1
    )


def stack_fold(root, tr, va, is_rg):
    xs, ls, mus, mps = [], [], [], []
    xva = None
    for a in [*tr, va]:
        xa = (
            np.asarray(np.load(root / "rg" / f"RG_{a}.npy", mmap_mode="r"))
            if is_rg
            else load_space_x(root, a)
        )
        if a == va:
            xva = xa
            break
        i = ANCHOR_IDX[a]
        m = COHORT[i]
        xs.append(xa[m])
        n = int(m.sum())
        ls.append(np.log1p(TARGETS[i][m]))
        mus.append(np.full(n, MU[a], dtype=np.float32))
        mps.append(np.full(n, MU_POS[a], dtype=np.float32))
        del xa
    xtr = np.concatenate(xs)
    ln = np.concatenate(ls).astype(np.float32)
    mu_row = np.concatenate(mus)
    mupos_row = np.concatenate(mps)
    del xs, ls, mus, mps
    gc.collect()
    y_all = (ln - mu_row).astype(np.float32)
    pos_all = (ln > 0).astype(np.float32)
    aux_pair = np.column_stack([
        pos_all,
        np.clip(ln - mu_row, -3, 6).astype(np.float32) ** 2 / 10.0,
    ]).astype(np.float32)
    return xtr, xva, ln, mu_row, mupos_row, y_all, pos_all, aux_pair


def rank_gauss_matrix(src):
    out = np.empty(src.shape, dtype=np.float32)
    nrow = src.shape[0]
    for j in range(src.shape[1]):
        v = np.asarray(src[:, j], dtype=np.float64)
        order = np.argsort(v, kind="mergesort")
        sv = v[order]
        new = np.empty(nrow, dtype=bool)
        new[0] = True
        np.not_equal(sv[1:], sv[:-1], out=new[1:])
        grp = np.cumsum(new) - 1
        counts = np.bincount(grp)
        starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
        ranked = np.empty(nrow, dtype=np.float64)
        ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
        r = (ranked / nrow + 0.5 / nrow).clip(1e-6, 1.0 - 1e-6)
        out[:, j] = ndtri(r).astype(np.float32)
    return out


def cat_params(cfg, seed, loss="RMSE"):
    p = {
        "loss_function": loss,
        "iterations": cfg["iters"],
        "learning_rate": min(cfg["lr"], 0.01)
        if loss.startswith(("Tweedie", "Poisson"))
        else cfg["lr"],
        "task_type": "CPU" if loss.startswith("Tweedie") else "GPU",
        "depth": cfg["depth"],
        "l2_leaf_reg": cfg.get("l2", 6.0),
        "border_count": cfg.get("border", 128),
        "bootstrap_type": "Bernoulli",
        "subsample": cfg.get("sub", 0.8),
        "random_strength": cfg.get("rs", 1.0),
        "boosting_type": "Plain",
        "max_ctr_complexity": 0,
        "verbose": False,
        "allow_writing_files": False,
        "random_seed": seed,
    }
    if p["task_type"] == "GPU":
        p["devices"] = "0"
    p.update(cfg.get("extra") or {})
    return p


def fit_torch_member(kind, cfg, xt, xv, y_all, pos_all, aux_pair, seed, cuda_seed=True):
    torch.manual_seed(seed)
    if cuda_seed:
        torch.cuda.manual_seed_all(seed)
    d_in = xt.shape[1]
    if kind == "tabm":
        edges = None
        if cfg["ple"]:
            q = np.linspace(0.0, 1.0, PLE_BINS + 1)
            sub_rows = xt[:: max(len(xt) // 200_000, 1)].astype(np.float32)
            e = np.quantile(sub_rows, q, axis=0).T.astype(np.float32)
            e[:, 0] -= 1e-3
            e[:, -1] += 1e-3
            e = np.maximum.accumulate(e, axis=1)
            edges = torch.from_numpy(np.ascontiguousarray(e)).to("cuda")
        model = TabM(d_in, cfg["k"], cfg["width"], cfg["blocks"], cfg["drop"], 2, edges).to("cuda")
        batch, lr_max, wd = 4096, cfg["lr"], 3e-4
    else:
        width = cfg["width"]
        drop = cfg.get("drop", 0.2)
        n_out = {"mlp": 3, "mlpce": HL_BINS, "mlpord": ORD_BINS - 1, "mlpv3": 2}[kind]
        model = nn.ModuleDict({
            "body": nn.Sequential(
                nn.Linear(d_in, width),
                nn.BatchNorm1d(width),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(width, width // 2),
                nn.BatchNorm1d(width // 2),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(width // 2, 256),
                nn.BatchNorm1d(256),
                nn.GELU(),
            ),
            "skip": nn.Linear(d_in, 256),
            "head": nn.Linear(256, n_out),
        }).to("cuda")
        batch, lr_max, wd = 8192, cfg.get("lr", 3e-3), 1e-4
    epochs = cfg["epochs"]
    n = len(y_all)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr_max, total_steps=epochs * ((n + batch - 1) // batch)
    )
    xt_t = torch.from_numpy(xt)
    yt_t = torch.from_numpy(y_all)
    pt_t = torch.from_numpy(pos_all)
    at_t = torch.from_numpy(aux_pair)
    edges_t = torch.tensor(HL_EDGES, dtype=torch.float32, device="cuda")
    centers_t = torch.tensor(HL_CENTERS, dtype=torch.float32, device="cuda")
    thr_t = torch.tensor(ORD_EDGES[1:-1], dtype=torch.float32, device="cuda")
    ord_centers = torch.tensor(ORD_CENTERS, dtype=torch.float32, device="cuda")
    mse, bce = nn.MSELoss(), nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for kk in range(0, n, batch):
            i = perm[kk : kk + batch]
            xb = xt_t[i].to("cuda").float()
            yb = yt_t[i].to("cuda")
            if kind == "tabm":
                pb = pt_t[i].to("cuda")
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    o = model(xb).float()
                loss = mse(o[:, :, 0], yb[:, None].expand(-1, cfg["k"])) + 0.3 * bce(
                    o[:, :, 1], pb[:, None].expand(-1, cfg["k"])
                )
            elif kind == "mlpce":
                cdf = torch.special.ndtr((edges_t[None, :] - yb[:, None]) / HL_SIGMA)
                tgt = cdf[:, 1:] - cdf[:, :-1]
                tgt = tgt / tgt.sum(dim=1, keepdim=True).clamp_min(1e-8)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    h = model["body"](xb) + model["skip"](xb)
                    logits = model["head"](h).float()
                loss = -(tgt * torch.log_softmax(logits, dim=1)).sum(dim=1).mean()
            elif kind == "mlpord":
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    h = model["body"](xb) + model["skip"](xb)
                    logits = model["head"](h).float()
                tgt = (yb[:, None] > thr_t).float()
                act = torch.cat([torch.ones_like(tgt[:, :1]), tgt[:, :-1]], dim=1)
                raw_l = nn.functional.binary_cross_entropy_with_logits(
                    logits, tgt, reduction="none"
                )
                loss = (raw_l * act).sum() / act.sum().clamp_min(1.0)
            elif kind == "mlpv3":
                pb = pt_t[i].to("cuda")
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    h = model["body"](xb) + model["skip"](xb)
                    o = model["head"](h)
                    loss = mse(o[:, 0].float(), yb) + 0.3 * bce(o[:, 1].float(), pb)
            else:
                ab = at_t[i].to("cuda")
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    h = model["body"](xb) + model["skip"](xb)
                    o = model["head"](h)
                    loss = mse(o[:, 0].float(), yb) + 0.3 * mse(o[:, 1:].float(), ab)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if kind == "tabm":
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
    model.eval()
    chunk = 8192 if kind == "tabm" else 16384
    out_chunks = []
    with torch.no_grad():
        for kk in range(0, len(xv), chunk):
            xb = torch.from_numpy(np.ascontiguousarray(xv[kk : kk + chunk])).to("cuda").float()
            with torch.autocast("cuda", dtype=torch.bfloat16):
                if kind == "tabm":
                    o = model(xb).float()
                else:
                    h = model["body"](xb) + model["skip"](xb)
                    o = model["head"](h).float()
            if kind == "tabm":
                out_chunks.append(o[:, :, 0].mean(dim=1).cpu().numpy())
            elif kind == "mlpce":
                out_chunks.append((torch.softmax(o, dim=1) @ centers_t).cpu().numpy())
            elif kind == "mlpord":
                p = torch.sigmoid(o)
                surv = torch.cat([torch.ones_like(p[:, :1]), torch.cumprod(p, dim=1)], dim=1)
                nxt = torch.cat([torch.cumprod(p, dim=1), torch.zeros_like(p[:, :1])], dim=1)
                prob = surv - nxt
                out_chunks.append(
                    (prob / prob.sum(dim=1, keepdim=True).clamp_min(1e-8) @ ord_centers)
                    .cpu()
                    .numpy()
                )
            else:
                out_chunks.append(o[:, 0].cpu().numpy())
    pred = np.concatenate(out_chunks).astype(np.float64)
    state = model.state_dict()
    del model, xt_t, yt_t, pt_t, at_t
    torch.cuda.empty_cache()
    return pred, state


def fit_member(name, cfg, xt, xv, y_all, pos_all, aux_pair, seeds, is_sub, out_dir, cuda_seed=True):
    kind = cfg["kind"]
    ext = MODEL_EXT[kind]
    preds = []
    for seed in seeds:
        t0 = time.time()
        out_path = out_dir / f"sub_{seed}.{ext}"
        if kind == "cat":
            booster = CatBoostRegressor(**cat_params(cfg, seed, cfg.get("loss", "RMSE")))
            booster.fit(Pool(xt, y_all))
            pred = np.asarray(booster.predict(xv), dtype=np.float64)
            if is_sub:
                booster.save_model(str(out_path))
            del booster
        elif kind == "xgb":
            dtr = xgb.QuantileDMatrix(xt, label=y_all, max_bin=256)
            booster = xgb.train(
                {
                    "objective": "reg:squarederror",
                    "eval_metric": "rmse",
                    "tree_method": "hist",
                    "device": "cuda",
                    "max_depth": cfg["depth"],
                    "eta": cfg["lr"],
                    "subsample": cfg.get("subsample", 0.8),
                    "colsample_bytree": cfg.get("colsample", 0.5),
                    "colsample_bynode": 0.8,
                    "min_child_weight": cfg.get("mcw", 32.0),
                    "lambda": cfg.get("reg_lambda", 20.0),
                    "max_bin": 256,
                    "seed": seed,
                },
                dtr,
                num_boost_round=cfg["rounds"],
            )
            pred = np.asarray(booster.inplace_predict(xv), dtype=np.float64)
            if is_sub:
                booster.save_model(str(out_path))
            del booster, dtr
        elif kind == "lgb":
            ds = lgb.Dataset(xt, y_all, feature_name=list(FEATURE_NAMES[: xt.shape[1]]))
            booster = lgb.train(
                {
                    "objective": "regression",
                    "metric": "rmse",
                    "learning_rate": cfg["lr"],
                    "num_leaves": cfg["leaves"],
                    "min_data_in_leaf": cfg.get("min_data", 200),
                    "feature_fraction": cfg.get("ff", 0.4),
                    "feature_fraction_bynode": 0.7,
                    "bagging_fraction": 0.7,
                    "bagging_freq": 1,
                    "lambda_l2": cfg.get("l2", 30.0),
                    "max_bin": 127,
                    "num_threads": 10,
                    "force_col_wise": True,
                    "verbosity": -1,
                    "seed": seed,
                },
                ds,
                num_boost_round=cfg["rounds"],
            )
            pred = np.asarray(booster.predict(xv), dtype=np.float64)
            if is_sub:
                booster.save_model(str(out_path))
            del booster, ds
        else:
            pred, state = fit_torch_member(
                kind, cfg, xt, xv, y_all, pos_all, aux_pair, seed, cuda_seed
            )
            if is_sub:
                torch.save(state, out_path)
        preds.append(pred)
        print(f"    {name} seed {seed} {time.time() - t0}s", flush=True)
        gc.collect()
    return np.mean(preds, axis=0)


def train_pool(space, prefix, members, folds, n_train=N_TRAIN_ANCHORS):
    root = SPACES[space]["dir"]
    cuda_seed = SPACES[space]["cuda_seed"]
    tab = [m for m in members if MEMBERS[m]["kind"] not in ("seq", "seq2")]
    for name in members:
        (MEMBER_DIR / f"{prefix}{name}").mkdir(parents=True, exist_ok=True)

    def done(name, va):
        d = MEMBER_DIR / f"{prefix}{name}"
        if va != SUBMIT:
            return (d / f"oof_{va}.npy").exists()
        cfg = MEMBERS[name]
        return (d / "sub.npy").exists() and all(
            (d / f"sub_{s}.{MODEL_EXT[cfg['kind']]}").exists() for s in SEED_POOL[: cfg["n_sub"]]
        )

    for va in folds:
        is_sub = va == SUBMIT
        for min_gap in (0, 28):
            todo = [m for m in tab if MEMBERS[m]["min_gap"] == min_gap and not done(m, va)]
            if not todo:
                continue
            tr = fold_anchors(va, n_train, min_gap)
            print(f"\n{prefix or 'pool '}{va}  min_gap={min_gap}  {len(todo)} members", flush=True)
            for is_rg in (False, True):
                group = [m for m in todo if (MEMBERS[m]["kind"] in RG_KINDS) == is_rg]
                if not group:
                    continue
                xtr, xva, _, _, _, y_all, pos_all, aux_pair = stack_fold(root, tr, va, is_rg)
                print(f"  matrix {xtr.shape}  val {xva.shape}", flush=True)
                for name in group:
                    t_member = time.time()
                    cfg = MEMBERS[name]
                    nc = xtr.shape[1] if cfg["v4"] else N_BASE
                    xt, xv = xtr[:, :nc], xva[:, :nc]
                    if cfg["kind"] == "mlpv3":
                        xt, xv = rank_gauss_matrix(xt), rank_gauss_matrix(xv)
                        gc.collect()
                    raw = fit_member(
                        name,
                        cfg,
                        xt,
                        xv,
                        y_all,
                        pos_all,
                        aux_pair,
                        SEED_POOL[: cfg["n_sub"] if is_sub else cfg["n_oof"]],
                        is_sub,
                        MEMBER_DIR / f"{prefix}{name}",
                        cuda_seed,
                    )
                    out_np = (MEMBER_DIR / f"{prefix}{name}") / (
                        "sub.npy" if is_sub else f"oof_{va}.npy"
                    )
                    np.save(out_np, raw.astype(np.float32))
                    print(f"  {name}{time.time() - t_member}s", flush=True)
                    del raw, xt, xv
                    gc.collect()
                del xtr, xva, y_all, pos_all, aux_pair
                gc.collect()

### Sequence fitter

In [ ]:
SIDE_IDX = {"all": np.arange(N_BASE)}
SIDE_IDX["v3"] = np.array([
    BASE_NAMES.index(c) for c in [n for n in BASE_NAMES if n.startswith("rank_")] + V3_SIDE_COLS
])


def patch_window(daily, cal, users, t, patch, n_patch, n_ch, is_seq2, valid_flag):
    context_len = patch * n_patch
    lo_d = t - context_len + 1
    n = len(users)
    if lo_d >= 0:
        win = daily[users, lo_d : t + 1].float()
        valid = torch.ones(n, n_patch, 1, device="cuda")
        cwin = cal[lo_d : t + 1]
    else:
        pad = -lo_d
        win = torch.zeros(n, context_len, n_ch, device="cuda")
        win[:, pad:] = daily[users, 0 : t + 1].float()
        v = torch.ones(context_len, device="cuda")
        v[:pad] = 0.0
        valid = v.view(n_patch, patch).amax(dim=1).view(1, -1, 1).expand(n, -1, -1)
        cwin = torch.zeros(context_len, cal.shape[1], device="cuda")
        cwin[pad:] = cal[0 : t + 1]
    win = win.view(n, n_patch, patch, n_ch)
    if is_seq2:
        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True) / patch
        content = torch.cat([win.sum(dim=2), win.amax(dim=2), cnt], dim=2)
        cpatch = cwin.view(n_patch, patch, -1).mean(dim=1).unsqueeze(0).expand(n, -1, -1)
        return torch.cat([content, valid, cpatch], dim=2)
    cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True)
    parts = [win.sum(dim=2), win.amax(dim=2), cnt]
    if valid_flag:
        parts.append(valid)
    return torch.cat(parts, dim=2)


def train_seq_pool(space, prefix, members, folds, n_train=N_TRAIN_ANCHORS):
    root = SPACES[space]["dir"]
    cuda_seed = SPACES[space]["cuda_seed"]
    for name in members:
        cfg = MEMBERS[name]
        out_dir = MEMBER_DIR / f"{prefix}{name}"
        out_dir.mkdir(parents=True, exist_ok=True)
        pending = [
            va
            for va in folds
            if not ((out_dir / "sub.npy") if va == SUBMIT else (out_dir / f"oof_{va}.npy")).exists()
        ]
        if not pending:
            continue
        is_seq2 = cfg["kind"] == "seq2"
        arch = SEQ2_ARCH if is_seq2 else SEQ_ARCH[name]
        tensor_tag = "daily10" if is_seq2 else "daily9"
        meta = json.loads((SEQ / f"{tensor_tag}_meta.json").read_text())
        n_ch = len(meta["channels"])
        side_idx = SIDE_IDX["all"] if is_seq2 else SIDE_IDX[str(arch["side"])]
        n_side = len(side_idx)
        n_feat = 2 * n_ch + 1 + 1 + N_CAL
        patch, n_patch = int(arch["patch"]), int(arch["n_patch"])
        daily = torch.from_numpy(np.load(SEQ / f"{tensor_tag}_f16.npy")).to("cuda")
        cal = torch.from_numpy(np.load(CAL_PATH)).to("cuda")
        for va in pending:
            is_sub = va == SUBMIT
            tr = fold_anchors(va, n_train, 0)
            train_sets, val_set = [], None
            for a in [*tr, va]:
                side_np = np.asarray(np.load(root / "rg" / f"RG_{a}.npy", mmap_mode="r"))[
                    :, side_idx
                ]
                t_a = ANCHOR_T[[d.isoformat() for d in ALL_DATES].index(a)]
                if a == va:
                    val_set = (
                        torch.arange(side_np.shape[0], device="cuda"),
                        t_a,
                        torch.from_numpy(np.ascontiguousarray(side_np)).to("cuda"),
                    )
                    break
                i = ANCHOR_IDX[a]
                m = COHORT[i]
                ly = np.log1p(TARGETS[i][m])
                train_sets.append((
                    torch.from_numpy(np.where(m)[0]).to("cuda"),
                    t_a,
                    torch.from_numpy((ly - MU[a]).astype(np.float32)).to("cuda"),
                    torch.from_numpy((TARGETS[i][m] > 0).astype(np.float32)).to("cuda"),
                    torch.from_numpy(np.ascontiguousarray(side_np[m])).to("cuda"),
                ))
                del side_np
            gc.collect()
            preds = []
            for seed in SEED_POOL[: cfg["n_sub"] if is_sub else cfg["n_oof"]]:
                t0 = time.time()
                torch.manual_seed(seed)
                if cuda_seed:
                    torch.cuda.manual_seed_all(seed)
                if is_seq2:
                    net = SeqNet(arch, n_feat, n_side).to("cuda")
                    enc_p = list(net.encoder.parameters())
                    other_p = [
                        p for nm, p in net.named_parameters() if not nm.startswith("encoder.")
                    ]
                    base_lr = cfg["lr"]
                    head_lr = base_lr * cfg["head_lr_mult"]
                    opt = torch.optim.AdamW(
                        [{"params": enc_p, "lr": base_lr}, {"params": other_p, "lr": head_lr}],
                        weight_decay=0.01,
                    )
                    epochs = int(cfg["epochs"])
                    steps = epochs * sum((len(s[0]) + 2047) // 2048 for s in train_sets)
                    sched = torch.optim.lr_scheduler.OneCycleLR(
                        opt, max_lr=[base_lr, head_lr], total_steps=steps
                    )
                else:
                    net = Seq(arch, n_side, n_ch).to("cuda")
                    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)
                    epochs = int(arch["epochs"])
                    steps = epochs * sum((len(s[0]) + 2047) // 2048 for s in train_sets)
                    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps)
                mse, bce = nn.MSELoss(), nn.BCEWithLogitsLoss()
                for _ in range(epochs):
                    net.train()
                    order = []
                    for si, s in enumerate(train_sets):
                        perm = torch.randperm(len(s[0]), device="cuda")
                        order += [(si, perm[k : k + 2048]) for k in range(0, len(perm), 2048)]
                    if bool(arch["mix_anchors"]):
                        order = [order[j] for j in torch.randperm(len(order)).tolist()]
                    for si, i in order:
                        users, t_a, y_s, pos_s, side_s = train_sets[si]
                        seq = patch_window(
                            daily,
                            cal,
                            users[i],
                            t_a,
                            patch,
                            n_patch,
                            n_ch,
                            is_seq2,
                            bool(arch.get("valid")),
                        )
                        with torch.autocast("cuda", dtype=torch.bfloat16):
                            mu_out, logit = net(seq, side_s[i].float())
                            loss = mse(mu_out.float(), y_s[i]) + 0.3 * bce(logit.float(), pos_s[i])
                        opt.zero_grad(set_to_none=True)
                        loss.backward()
                        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                        opt.step()
                        sched.step()
                net.eval()
                vu, vt, vside = val_set
                out_chunks = []
                with torch.no_grad():
                    for k in range(0, len(vu), 4096):
                        u = vu[k : k + 4096]
                        seq = patch_window(
                            daily,
                            cal,
                            u,
                            vt,
                            patch,
                            n_patch,
                            n_ch,
                            is_seq2,
                            bool(arch.get("valid")),
                        )
                        with torch.autocast("cuda", dtype=torch.bfloat16):
                            mu_out, _ = net(seq, vside[k : k + 4096].float())
                        out_chunks.append(mu_out.float().cpu().numpy())
                preds.append(np.concatenate(out_chunks).astype(np.float64))
                if is_sub:
                    torch.save(net.state_dict(), out_dir / f"sub_{seed}.pt")
                del net
                torch.cuda.empty_cache()
                print(f"    {prefix}{name} {va} seed {seed} {time.time() - t0}s", flush=True)
            np.save(
                out_dir / ("sub.npy" if is_sub else f"oof_{va}.npy"),
                np.mean(preds, axis=0).astype(np.float32),
            )
            del train_sets, val_set, preds
            torch.cuda.empty_cache()
            gc.collect()
        del daily, cal
        torch.cuda.empty_cache()
        gc.collect()

### The repaired pool

In [ ]:
SEQ_MEMBERS = ["v3_seq", "seq_b", "seq_c"]
train_pool("repaired", "", MEMBER_ORDER, FOLDS)
train_seq_pool("repaired", "", SEQ_MEMBERS, FOLDS)

### Panel

In [ ]:
PANEL_ANCHORS = [*FIT_ANCHORS, *EVAL_ANCHORS, SUBMIT]
CACHE_M, CACHE_Y = {}, {}
for a in PANEL_ANCHORS:
    m = COHORT[-1] if a == SUBMIT else COHORT[ANCHOR_IDX[a]]
    cols = []
    for name in MEMBER_ORDER:
        f = (MEMBER_DIR / name / "sub.npy") if a == SUBMIT else (MEMBER_DIR / name / f"oof_{a}.npy")
        raw = np.load(f).astype(np.float64)[m]
        sd = raw.std()
        cols.append((raw - raw.mean()) / (sd if sd > 1e-12 else 1.0))
    CACHE_M[a] = np.column_stack(cols)
    if a != SUBMIT:
        y = np.log1p(TARGETS[ANCHOR_IDX[a]][m])
        CACHE_Y[a] = (y - y.mean()) / y.std()
    print(f"{a} {CACHE_M[a].shape}", flush=True)

Xf = np.vstack([CACHE_M[a] for a in FIT_ANCHORS])
yf = np.concatenate([CACHE_Y[a] for a in FIT_ANCHORS])

### Linear blend

In [ ]:
w_nnls_all, _ = nnls(Xf, yf)
KEEP = np.flatnonzero(w_nnls_all > 1e-8)
KEPT_MEMBERS = [MEMBER_ORDER[i] for i in KEEP]
SHIPPED_KEPT = [
    "cat_a_sp",
    "cat_b",
    "catf_b",
    "catf_b_sp",
    "catf_f",
    "catf_f_sp",
    "seq_b",
    "seq_c",
    "v3_mlp",
    "v3_seq",
    "xgb_a",
    "xgbf_a_sp",
]
SCOPES = {"kept": KEEP, "all": np.arange(len(MEMBER_ORDER))}
print(f"non-negative solve keeps {len(KEEP)} of {len(MEMBER_ORDER)}")
print(", ".join(KEPT_MEMBERS))
print(", ".join(SHIPPED_KEPT))
print(KEPT_MEMBERS == SHIPPED_KEPT)

### Residual combiners

In [ ]:
TUNED_FAMILY = {"kept": "lgbm", "all": "xgb"}
TUNED_FILE = {"kept": "v46_lgbm_residual.txt", "all": "v47_xgb_residual.ubj"}
TUNED_ROUNDS = {"kept": 151, "all": 230}
TUNED_PARAMS = {
    "kept": {
        "learning_rate": 0.010416117992371583,
        "num_leaves": 7,
        "min_data_in_leaf": 509,
        "feature_fraction": 0.6326418272631035,
        "bagging_fraction": 0.5689126760755702,
        "lambda_l2": 4.460350509886045,
    },
    "all": {
        "eta": 0.01033334897515964,
        "max_depth": 2,
        "min_child_weight": 42.975658562793505,
        "subsample": 0.7134313151821792,
        "colsample_bytree": 0.6845696095993247,
        "lambda": 0.2142552553768538,
    },
}
COMBINER_SEED = 42
BOOSTERS, NNLS_W = {}, {}
for scope, cols in SCOPES.items():
    Xc = Xf[:, cols]
    w_scope, _ = nnls(Xc, yf)
    lin = Xc @ w_scope
    NNLS_W[scope] = w_scope
    (COMBINER_DIR / f"nnls_{scope}.json").write_text(
        json.dumps(
            {
                "scope": scope,
                "members": [MEMBER_ORDER[i] for i in cols],
                "columns": [int(i) for i in cols],
                "weights": [float(v) for v in w_scope],
            },
            indent=1,
        )
    )
    xs = np.column_stack([Xc, lin])
    ys = yf - lin
    booster_path = COMBINER_DIR / TUNED_FILE[scope]
    if TUNED_FAMILY[scope] == "lgbm" and booster_path.exists():
        booster = lgb.Booster(model_file=str(booster_path))
    elif TUNED_FAMILY[scope] == "lgbm":
        booster = lgb.train(
            {
                "objective": "regression",
                "verbose": -1,
                "seed": COMBINER_SEED,
                "num_threads": 16,
                "force_row_wise": True,
                "bagging_freq": 1,
                **TUNED_PARAMS[scope],
            },
            lgb.Dataset(xs, label=ys),
            num_boost_round=TUNED_ROUNDS[scope],
        )
        booster.save_model(str(booster_path))
    elif booster_path.exists():
        booster = xgb.Booster()
        booster.load_model(str(booster_path))
    else:
        booster = xgb.train(
            {
                "objective": "reg:squarederror",
                "tree_method": "hist",
                "device": "cuda",
                "seed": COMBINER_SEED,
                **TUNED_PARAMS[scope],
            },
            xgb.DMatrix(xs, label=ys),
            num_boost_round=TUNED_ROUNDS[scope],
        )
        booster.save_model(str(booster_path))
    BOOSTERS[scope] = booster
    print(f"{scope}  {TUNED_FAMILY[scope]} on residual, {len(cols)} members -> {booster_path.name}")

(COMBINER_DIR / "members.json").write_text(
    json.dumps(
        {
            "members": MEMBER_ORDER,
            "kept": KEPT_MEMBERS,
            "family": TUNED_FAMILY,
            "file": TUNED_FILE,
            "rounds": TUNED_ROUNDS,
            "params": TUNED_PARAMS,
        },
        indent=1,
    )
)

### Forward check

In [ ]:
RECORDED = {"linear": 0.6780457242978827, "kept": 0.6778257186532514, "all": 0.6778140406740206}
FORWARD = {}
pz, py = [], []
for a in EVAL_ANCHORS:
    p = CACHE_M[a] @ w_nnls_all
    pz.append((p - p.mean()) / p.std())
    py.append(CACHE_Y[a])
FORWARD["linear"] = float(np.corrcoef(np.concatenate(pz), np.concatenate(py))[0, 1])
for scope, cols in SCOPES.items():
    pz = []
    for a in EVAL_ANCHORS:
        Z = CACHE_M[a][:, cols]
        lz = Z @ NNLS_W[scope]
        xs = np.column_stack([Z, lz])
        f = BOOSTERS[scope].predict(xs if scope == "kept" else xgb.DMatrix(xs))
        p = lz + f
        pz.append((p - p.mean()) / p.std())
    FORWARD[scope] = float(np.corrcoef(np.concatenate(pz), np.concatenate(py))[0, 1])

for tag, label in (
    ("linear", "linear nnls"),
    ("kept", "lgbm on residual"),
    ("all", "xgb on residual"),
):
    print(f"{label}{FORWARD[tag]}{RECORDED[tag]}{FORWARD[tag] - RECORDED[tag]}")

### Member scores

In [ ]:
MEMBER_CV = {}
for name in MEMBER_ORDER:
    num, den = 0.0, 0
    per = {}
    for a in VAL_ANCHORS:
        m = COHORT[ANCHOR_IDX[a]]
        y = np.log1p(TARGETS[ANCHOR_IDX[a]][m])
        r = np.load(MEMBER_DIR / name / f"oof_{a}.npy").astype(np.float64)[m]
        pr = np.clip(r - r.mean() + y.mean(), 0.0, None)
        per[a] = float(np.sqrt(((pr - y) ** 2).mean()))
        num += float(((pr - y) ** 2).sum())
        den += len(y)
    MEMBER_CV[name] = {**per, "POOLED_ALL": float(np.sqrt(num / den))}
    (MEMBER_DIR / name / "cv.json").write_text(json.dumps(MEMBER_CV[name], indent=1))

print(f"{'member'}{'pooled'}")
for name in sorted(MEMBER_ORDER, key=lambda n: MEMBER_CV[n]["POOLED_ALL"]):
    print(f"{name}{MEMBER_CV[name]['POOLED_ALL']}")

### Recorded public scores

In [ ]:
SCORED = {
    "catboost_v1": ("submit_catboost", 1.695010),
    "catboost_v2": ("submit_catboost_v2", 1.661364),
    "cb_v2_dropz": ("submit_catboost_v2_drop_zeros", 1.817905),
    "chronos2": ("submit_chronos_2", 1.747750),
    "chronosL": ("submit_chronos_large", 1.840316),
    "v3_persist": ("submit_v3_blend_persist", 1.654585),
    "v3_seasonal": ("submit_v3_blend_seasonal", 1.651686),
    "v4_blend_fit": ("submit_v4_blend_fit", 1.65078581308201),
    "v4_blend_flat": ("submit_v4_blend_flat", 1.6503612359886002),
    "v4_mix_fit": ("submit_v4_mix_fit", 1.651312729793189),
    "v4_mix_flat": ("submit_v4_mix_flat", 1.6509095214573077),
    "v5_beta": ("submit_v5_beta", 1.6497376211422583),
    "v6_vall": ("submit_v6_vall", 1.649484498307152),
    "v6_vtree": ("submit_v6_vtree", 1.649715997059653),
    "v6_vnn": ("submit_v6_vnn", 1.650206718615339),
    "v7_all": ("submit_v7_all", 1.6495745974941638),
    "v8_probeopt": ("submit_v8_probeopt", 1.6493681),
    "v9_opt4": ("submit_v9_opt4", 1.6492880),
    "v10_resid": ("submit_v10_resid", 1.6491904),
    "p1_resid": ("submit_p1_resid", 1.6555198),
    "p2_spaced": ("submit_p2_spaced", 1.6493549265),
    "v11_spaced": ("submit_v11_spaced", 1.6487584),
    "p3_spaced9": ("submit_p3_spaced9", 1.6494531),
    "v12_harvest": ("submit_v12_harvest", 1.6482018433),
    "v13_fleet": ("submit_v13_fleet", 1.6482244574),
    "v21_banked": ("submit_v21_banked", 1.6476093577642303),
    "v_btyd_opt": ("submit_v_btyd_opt", 1.6475303399615397),
    "u_probe_btyd": ("submit_u_probe_btyd", 1.6475994542392367),
    "t_blend5": ("submit_t_blend5", 1.6476223900838673),
    "x_probe_deflate": ("submit_x_probe_deflate", 1.6476995945223647),
    "w_probe_factors": ("submit_w_probe_factors", 1.6477488740736896),
    "r_blend_shift20_mask10": ("submit_r_blend_shift20_mask10", 1.64782886683583),
    "p_shift20_zeros_1": ("submit_p_shift20_zeros_1", 1.6479258506119911),
    "r_pool_s20": ("submit_r_pool_s20", 1.6480483367),
    "q_mask_stable05": ("submit_q_mask_stable05", 1.6486590331415325),
    "q_mask_stable10": ("submit_q_mask_stable10", 1.6480942379205155),
    "q_span_pall": ("submit_q_span_pall", 1.6484244522061702),
    "p_shift20": ("submit_p_shift20", 1.6479108007353238),
    "p_shift": ("submit_p_shift", 1.6486021993215545),
    "p_adv": ("submit_p_adv", 1.64881537099646),
    "v22_merge": ("submit_v22_merge", 1.647502147436395),
    "y_cap250": ("submit_y_cap250", 1.6480523392),
    "y_yearago": ("submit_y_yearago", 1.648021364398094),
    "z_bank": ("submit_z_bank", 1.647465966),
    "v29all": ("submit_v29all", 1.6493583),
    "v29tree": ("submit_v29tree", 1.6495142),
    "v29nn": ("submit_v29nn", 1.6497271),
    "v30_lean5": ("submit_v30_lean5", 1.6496033013),
    "v30_bank": ("submit_v30_bank", 1.647654043577311),
    "v30_bank2": ("submit_v30_bank2", 1.6477619386022426),
    "v31_seq010": ("submit_v31_seq010", 1.6472032470305245),
    "v32_seqopt": ("submit_v32_seqopt", 1.6470992629545775),
    "v34_sw": ("submit_v34_sw", 1.6471889832),
    "v34_swta": ("submit_v34_swta", 1.6472111644),
    "v35_aug150": ("submit_v35_aug150", 1.6470558749),
    "v33_derived": ("submit_v33_derived", 1.64696961),
    "v37_curve": ("submit_v37_curve", 1.6469886627),
    "v41_ridge": ("submit_v41_ridge", 1.646388098176818),
    "v42_pass2": ("submit_v42_pass2", 1.6473970612),
    "v43_calib": ("submit_v43_calib", 1.6462497938220484),
    "v44_seqmix": ("submit_v44_seqmix", 1.6463103088544133),
    "v45_combined": ("submit_v45_combined", 1.646297069),
    "v46_nl_kept": ("submit_v46_nl_kept", 1.648787092067305),
    "v47_nl_all": ("submit_v47_nl_all", 1.6487327968),
    "v48_newdir": ("submit_v48_newdir", 1.6460816563),
}

PROBE_SCORES = {
    "shift": 1.6486021993215545,
    "shift20": 1.6479108007353238,
    "adv": 1.64881537099646,
    "mask_stable05": 1.6486590331415325,
    "mask_stable10": 1.6480942379205155,
    "span_pall": 1.6484244522061702,
}
V36_PROBE = {"alpha": 0.012, "cov": 0.009695, "b_current": 1.6295100386053305}
print(f"{len(SCORED)} recorded public scores")

## Submissions

In [ ]:
TARGET_MEAN = 2.3312
W_TOTAL = 5.368096
SIGMA = float(np.sqrt(W_TOTAL))


def z_score(x):
    x = np.asarray(x, dtype=np.float64)
    sd = x.std()
    return (x - x.mean()) / (sd if sd > 1e-12 else 1.0)


def solve_level(z, beta, target_mean=TARGET_MEAN):
    lo, hi = -30.0, 30.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if float(np.clip(beta * z + mid, 0.0, None).mean()) < target_mean:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def write_submission(name, raw, beta, source="", target_mean=TARGET_MEAN):
    z = z_score(raw)
    level = solve_level(z, beta, target_mean)
    adj = np.clip(beta * z + level, 0.0, None)
    pred = np.expm1(adj)
    path = SUBS / f"{name}.csv"
    pl.DataFrame({"user_id": USER_IDS, "predict": pred.astype(np.float32)}).write_csv(path)
    meta = {
        "beta": beta,
        "level": level,
        "target_mean": target_mean,
        "mean_log1p": float(adj.mean()),
        "sd_log1p": float(adj.std()),
        "zero_frac": float((pred == 0).mean()),
        "source": source,
        "file": path.name,
    }
    (SUBS / f"meta_{name}.json").write_text(json.dumps(meta, indent=1))
    print(f"{name} beta={beta} level={level} zeros={meta['zero_frac']}")
    return meta


def write_raw_submission(name, pred, source=""):
    pred = np.asarray(pred, dtype=np.float64).clip(min=0.0)
    path = SUBS / f"{name}.csv"
    pl.DataFrame({"user_id": USER_IDS, "predict": pred.astype(np.float32)}).write_csv(path)
    meta = {
        "mean": float(pred.mean()),
        "max": float(pred.max()),
        "mean_log1p": float(np.log1p(pred).mean()),
        "zero_frac": float((pred <= 0).mean()),
        "source": source,
        "file": path.name,
    }
    (SUBS / f"meta_{name}.json").write_text(json.dumps(meta, indent=1))
    print(f"{name} mean={meta['mean']} log1p={meta['mean_log1p']}")
    return meta


def submission_vector(stem):
    d = pl.read_csv(SUBS / f"{stem}.csv").sort("user_id")
    return np.log1p(np.clip(d["predict"].to_numpy().astype(np.float64), 0.0, None))


def b_from_score(score, beta):
    return (W_TOTAL + beta * beta - score * score) / (2.0 * beta)


def score_from_b(b, beta=None):
    beta = b if beta is None else beta
    return float(np.sqrt(max(W_TOTAL - 2.0 * beta * b + beta * beta, 0.0)))


def b_of(name):
    v = submission_vector(SCORED[name][0])
    return b_from_score(SCORED[name][1], beta=float(v.std()))


def cov_from_probe(score, beta, alpha, b_current):
    b_obs = b_from_score(score, beta)
    return float((b_obs * np.sqrt(1.0 + alpha * alpha) - b_current) / alpha)


def mix_raw(current, d, alpha):
    return z_score(current) + alpha * z_score(d)


def orthogonalise(x, base):
    x = np.asarray(x, dtype=np.float64)
    b = np.asarray(base, dtype=np.float64)
    return x - b * float(x @ b / (b @ b))

## Blend helpers

In [ ]:
CLEAN = ["2025-09-24", "2025-10-08", "2025-10-22"]
RECENT = ["2025-12-03", "2025-12-17", "2025-12-31", "2026-01-14"]
PANEL = ["2025-08-27", "2025-10-08", "2025-11-12", "2025-12-17"]


def y_log(anchor):
    i = ANCHOR_IDX[anchor]
    return np.log1p(TARGETS[i][COHORT[i]])


def cohort_mask(anchor):
    return COHORT[-1] if anchor == SUBMIT else COHORT[ANCHOR_IDX[anchor]]


def target_z(anchor):
    return z_score(y_log(anchor))


def rmse_log(p, y):
    return float(np.sqrt(((np.clip(p, 0.0, None) - y) ** 2).mean()))


def shape_rmse(score, y):
    s = z_score(score)
    beta = float((s * (y - y.mean())).sum() / (s * s).sum())
    return rmse_log(beta * s + y.mean(), y)


def pooled_shape(by_anchor, anchor_list):
    num = sum(shape_rmse(by_anchor[a], y_log(a)) ** 2 * len(y_log(a)) for a in anchor_list)
    den = sum(len(y_log(a)) for a in anchor_list)
    return float(np.sqrt(num / den))


def member_z(store, name, anchor):
    d = store(name)
    f = (d / "sub.npy") if anchor == SUBMIT else (d / f"oof_{anchor}.npy")
    raw = np.load(f).astype(np.float64)
    return z_score(raw[cohort_mask(anchor)] if raw.shape[0] == N_USERS else raw)


def member_cols(store, anchor, names):
    return np.column_stack([member_z(store, n, anchor) for n in names])


def fit_nnls_weights(store, names, anchor_list, ridge=0.0):
    xs, ys = [], []
    for a in anchor_list:
        y = y_log(a)
        xs.append(member_cols(store, a, names))
        ys.append((y - y.mean()) / y.std())
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    if ridge > 0:
        n = len(names)
        x = np.vstack([x, np.sqrt(ridge * len(y)) * np.eye(n)])
        y = np.concatenate([y, np.zeros(n)])
    w, _ = nnls(x, y)
    return w / w.sum() if w.sum() > 0 else np.full(len(names), 1.0 / len(names))


def pool_store(name):
    return MEMBER_DIR / name


def pre_store(name):
    return MEMBER_DIR / f"pre_{name}"


def fit_and_flat_beta(oof_by_anchor, target_mean=TARGET_MEAN):
    levels, betas = [], []
    for a in VAL_ANCHORS:
        o = z_score(oof_by_anchor[a])
        y = y_log(a)
        levels.append(float(y.mean()))
        betas.append(float((o * (y - y.mean())).mean()))
    lv, bt = np.array(levels), np.array(betas)
    idx = [VAL_ANCHORS.index(a) for a in RECENT]
    slope, inter = np.polyfit(lv[idx], bt[idx], 1)
    return float(inter + slope * target_mean), float(bt[idx].mean())


def rho_at(vec_by_anchor, anchor_list):
    zs = [z_score(vec_by_anchor[a]) for a in anchor_list]
    ys = [target_z(a) for a in anchor_list]
    return float(np.mean([float(np.corrcoef(z, y)[0, 1]) for z, y in zip(zs, ys, strict=True)]))

## The pre-audit pool

In [ ]:
PRE_POOL = [
    "v3_cat",
    "v3_cat2",
    "v3_mlp",
    "v3_seq",
    "cat_a",
    "cat_b",
    "catf_a",
    "catf_b",
    "catf_d",
    "catf_f",
    "lgb_a",
    "lgbf_a",
    "lgbf_b",
    "mlpf_a",
    "mlpf_b",
    "mlpf_c",
    "mlpf_d",
    "mlpf_o",
    "seq_b",
    "seq_c",
    "tabm_a",
    "tabm_b",
    "xgb_a",
    "xgbf_a",
    "xgbf_b",
]
SPACED_MEMBERS = [
    "catf_d",
    "catf_b",
    "xgbf_a",
    "catf_f",
    "xgbf_b",
    "catf_a",
    "lgbf_b",
    "lgbf_a",
    "cat_a",
]
PRE_ALL = [*PRE_POOL, *[f"{m}_sp" for m in SPACED_MEMBERS]]
V4_BLEND_MODELS = [
    "v3_cat",
    "v3_cat2",
    "v3_mlp",
    "v3_seq",
    "cat_a",
    "cat_b",
    "catf_a",
    "lgb_a",
    "xgb_a",
]
V4_MIX_MODELS = [
    "v3_cat",
    "v3_cat2",
    "v3_mlp",
    "v3_seq",
    "cat_a",
    "cat_b",
    "catf_a",
    "catf_b",
    "catf_d",
    "catf_f",
    "lgb_a",
    "lgbf_a",
    "lgbf_b",
    "mlpf_a",
    "mlpf_b",
    "mlpf_c",
    "mlpf_d",
    "xgb_a",
    "xgbf_a",
    "xgbf_b",
]
V6_MODELS = [*V4_MIX_MODELS, "seq_b"]
V7_MODELS = list(PRE_POOL)
V3_MODELS = ["v3_cat", "v3_cat2", "v3_mlp", "v3_seq"]
TREE_PREFIX = ("cat", "xgb", "lgb", "hur", "v3_cat")
NN_PREFIX = ("mlp", "seq", "tabm", "v3_mlp", "v3_seq")


def group_by_prefix(names, prefixes):
    return [n for n in names if any(n.startswith(p) for p in prefixes)]


train_pool("pre", "pre_", PRE_ALL, FOLDS)
train_seq_pool("pre", "pre_", [m for m in PRE_POOL if MEMBERS[m]["kind"] in ("seq", "seq2")], FOLDS)

## Era one: single models

In [ ]:
CB1_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "task_type": "GPU",
    "border_count": 128,
    "random_seed": 42,
    "allow_writing_files": False,
    "learning_rate": 0.032943954665687655,
    "depth": 6,
    "l2_leaf_reg": 12.269468194188697,
    "random_strength": 0.05356605711263488,
    "grow_policy": "SymmetricTree",
    "bootstrap_type": "Bernoulli",
    "subsample": 0.6993810602304712,
    "iterations": 2262,
    "verbose": False,
}

if not (SUBS / "submit_catboost.csv").exists():
    xs, ys = [], []
    for a in V1_ANCHORS:
        i = ANCHOR_IDX[a]
        t = (date.fromisoformat(a) - DATE_MIN).days
        m = t - V1_MIN_HISTORY >= FIRST_DAY
        xs.append(np.load(V1 / f"X_{a}.npy")[m])
        ys.append(np.log1p(TARGETS[i][m]))
    xtr = np.concatenate(xs)
    ytr = np.concatenate(ys).astype(np.float32)
    del xs, ys
    gc.collect()
    booster = CatBoostRegressor(**CB1_PARAMS)
    booster.fit(Pool(xtr, ytr))
    booster.save_model(str(LEGACY_DIR / "catboost_v1.cbm"))
    pred = np.expm1(np.clip(booster.predict(np.load(V1 / f"X_{SUBMIT}.npy")), 0.0, None))
    write_raw_submission("submit_catboost", pred, source="catboost v1, 95 window aggregates")
    del xtr, ytr, booster
    gc.collect()

In [ ]:
CB2_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "task_type": "GPU",
    "random_seed": 42,
    "allow_writing_files": False,
    "learning_rate": 0.0221,
    "depth": 9,
    "l2_leaf_reg": 4.44,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 3.75,
    "iterations": 1336,
    "verbose": False,
}
CB2_SEEDS = [42, 7, 2024]
CB2_ANCHORS = ANCHORS[-15:]


def fit_catboost_v2(drop_zeros):
    xs, ys = [], []
    for a in CB2_ANCHORS:
        i = ANCHOR_IDX[a]
        m = COHORT[i].copy()
        if drop_zeros:
            m &= TARGETS[i] > 0
        xa = np.asarray(np.load(PRE / "x" / f"X_{a}.npy", mmap_mode="r"))
        xs.append(xa[m])
        ys.append(np.log1p(TARGETS[i][m]))
        del xa
    xtr = np.concatenate(xs)
    ytr = np.concatenate(ys).astype(np.float32)
    del xs, ys
    gc.collect()
    xsub = np.asarray(np.load(PRE / "x" / f"X_{SUBMIT}.npy", mmap_mode="r"))
    preds = []
    for seed in CB2_SEEDS:
        booster = CatBoostRegressor(**{**CB2_PARAMS, "random_seed": seed})
        booster.fit(Pool(xtr, ytr))
        preds.append(np.asarray(booster.predict(xsub), dtype=np.float64))
        del booster
        gc.collect()
    del xtr, ytr, xsub
    gc.collect()
    return np.mean(preds, axis=0)


for tag, drop, stem in (
    ("catboost_v2", False, "submit_catboost_v2"),
    ("cb_v2_dropz", True, "submit_catboost_v2_drop_zeros"),
):
    if (SUBS / f"{stem}.csv").exists():
        continue
    raw = fit_catboost_v2(drop)
    np.save(LEGACY_DIR / f"{tag}_sub.npy", raw.astype(np.float32))
    write_raw_submission(
        stem,
        np.expm1(np.clip(raw + V2_SHIFT, 0.0, None)),
        source=f"catboost v2 dense grid, 3 seeds, calendar shift {V2_SHIFT:+.4f}",
    )

### Chronos zero shot

In [ ]:
CHRONOS_CFG = {
    "chronos2": {
        "model": "amazon/chronos-2",
        "bin_days": 10,
        "context_bins": 37,
        "horizon_bins": 3,
        "quantile": 0.70,
        "covariates": True,
    },
    "chronosL": {
        "model": "amazon/chronos-t5-large",
        "bin_days": 30,
        "context_bins": 12,
        "horizon_bins": 1,
        "quantile": 0.85,
        "num_samples": 100,
        "covariates": False,
    },
}
CHRONOS_STEM = {"chronos2": "submit_chronos_2", "chronosL": "submit_chronos_large"}
CHUNK_USERS = 50_000


def chronos_context(tag, anchor):
    cfg = CHRONOS_CFG[tag]
    bd = cfg["bin_days"]
    t = (date.fromisoformat(anchor) - DATE_MIN).days
    n_bin = min((t + 1) // bd, cfg["context_bins"])
    series = {}
    for col in [*CH_VALUE_COLS, "active"] if cfg["covariates"] else ["gmv"]:
        arr = np.load(CHRONOS / f"b{bd}_{col}.npy", mmap_mode="r")
        hi = (t + 1) // bd
        series[col] = np.asarray(arr[:, hi - n_bin : hi], dtype=np.float32)
    return series, n_bin


def chronos_forecast(tag, anchor):
    import torch as _torch

    cfg = CHRONOS_CFG[tag]
    series, _ = chronos_context(tag, anchor)
    ctx = np.log1p(series["gmv"])
    n = ctx.shape[0]
    out = np.zeros(n, dtype=np.float64)
    if tag == "chronos2":
        from chronos import Chronos2Pipeline

        pipe = Chronos2Pipeline.from_pretrained(cfg["model"], device_map="cuda")
        cov = np.stack([np.log1p(series[c]) for c in CH_COVARIATES], axis=-1)
        for lo in range(0, n, CHUNK_USERS):
            hi = min(lo + CHUNK_USERS, n)
            tasks = [
                {
                    "target": ctx[i],
                    "past_covariates": {c: cov[i, :, j] for j, c in enumerate(CH_COVARIATES)},
                }
                for i in range(lo, hi)
            ]
            q = pipe.predict_quantiles(
                tasks,
                prediction_length=cfg["horizon_bins"],
                quantile_levels=[cfg["quantile"]],
            )
            out[lo:hi] = np.expm1(np.stack([np.asarray(x)[:, 0] for x in q])).sum(axis=1)
            print(f"  {tag} {anchor} {hi}/{n}", flush=True)
    else:
        from chronos import ChronosPipeline

        pipe = ChronosPipeline.from_pretrained(
            cfg["model"], device_map="cuda", torch_dtype=_torch.bfloat16
        )
        for lo in range(0, n, CHUNK_USERS):
            hi = min(lo + CHUNK_USERS, n)
            fc = pipe.predict(
                _torch.from_numpy(ctx[lo:hi]),
                prediction_length=cfg["horizon_bins"],
                num_samples=cfg["num_samples"],
            )
            q = _torch.quantile(fc.float(), cfg["quantile"], dim=1).cpu().numpy()
            out[lo:hi] = np.expm1(q).sum(axis=1)
            print(f"  {tag} {anchor} {hi}/{n}", flush=True)
    del pipe
    torch.cuda.empty_cache()
    return np.clip(out, 0.0, None)


for tag, stem in CHRONOS_STEM.items():
    if (SUBS / f"{stem}.csv").exists():
        continue
    cal_path = CHRONOS / f"{tag}_calib.json"
    if not cal_path.exists():
        slopes, inters = [], []
        for a in V1_ANCHORS:
            i = ANCHOR_IDX[a]
            m = COHORT[i]
            p = chronos_forecast(tag, a)[m]
            y = np.log1p(TARGETS[i][m])
            x = np.log1p(p)
            var = float(x.var())
            slope = float(((x - x.mean()) * (y - y.mean())).mean() / var) if var > 0 else 0.0
            slopes.append(slope)
            inters.append(float(y.mean() - slope * x.mean()))
        cal_path.write_text(
            json.dumps({"slope": float(np.mean(slopes)), "intercept": float(np.mean(inters))})
        )
    cal = json.loads(cal_path.read_text())
    raw = chronos_forecast(tag, SUBMIT)
    np.save(LEGACY_DIR / f"{tag}_sub.npy", raw.astype(np.float32))
    log_pred = cal["slope"] * np.log1p(raw) + cal["intercept"]
    write_raw_submission(
        stem,
        np.expm1(np.clip(log_pred, 0.0, 30.0)),
        source=f"{CHRONOS_CFG[tag]['model']} zero shot, quantile {CHRONOS_CFG[tag]['quantile']}",
    )

### v3 blend

In [ ]:
Q_ACTIVE = 0.954

if not (SUBS / "submit_v3_blend_seasonal.csv").exists():
    xs, ys = [], []
    for a in RECENT:
        mu = float(y_log(a).mean())
        xs.append(
            np.column_stack([
                np.load(pre_store(m) / f"oof_{a}.npy").astype(np.float64)[cohort_mask(a)] + mu
                for m in V3_MODELS
            ])
        )
        ys.append(y_log(a))
    w3, _ = nnls(np.concatenate(xs), np.concatenate(ys))
    w3 = w3 / w3.sum() if w3.sum() > 0 else np.full(len(V3_MODELS), 1.0 / len(V3_MODELS))
    print("v3 weights:", dict(zip(V3_MODELS, np.round(w3, 4), strict=True)))
    raw3 = (
        np.column_stack([np.load(pre_store(m) / "sub.npy").astype(np.float64) for m in V3_MODELS])
        @ w3
    )
    np.save(LEGACY_DIR / "v3_blend_sub.npy", raw3.astype(np.float32))

    last = LEVELS[ANCHORS[-1]]
    p_order_base = last["p_positive"] / last["p_active_next"]
    l_seasonal = (
        Q_ACTIVE
        * p_order_base
        * SEASON["ratio_p_order_given_active"]
        * last["e_log_given_positive"]
        * SEASON["ratio_e_log_given_positive"]
    )
    for tag, lvl in (("seasonal", l_seasonal), ("persist", last["mean_log1p"])):
        write_raw_submission(
            f"submit_v3_blend_{tag}",
            np.expm1(np.clip(raw3 + lvl, 0.0, None)),
            source=f"v3 four member blend, {tag} level {lvl}",
        )

### v4 blend and mix

In [ ]:
STACK_FEATS = [
    "days_since_last_order",
    "days_since_last_event",
    "tenure_days",
    "active_rate_30d",
    "active_rate_365d",
    "order_day_rate_90d",
    "has_order_sum_365d",
    "has_order_sum_30d",
    "loggmv_per_order_365d",
    "gmv_sum_30d",
    "gmv_sum_365d",
    "rank_gmv_sum_365d",
    "ewm_has_order_60",
    "ewm_loggmv_60",
    "os_n_orders",
    "os_gap_mean",
    "os_rec_over_gap",
    "blk_has_rate",
    "blk_lgmv_mean",
    "blk_lgmv_slope",
]
STACK_ROUNDS, STACK_LEAVES, MIX_LAM = 400, 31, 0.5


def fit_stack(xs, ys, xva, seed=42):
    ds = lgb.Dataset(np.concatenate(xs), np.concatenate(ys))
    return lgb.train(
        {
            "objective": "regression",
            "metric": "rmse",
            "learning_rate": 0.04,
            "num_leaves": STACK_LEAVES,
            "min_data_in_leaf": 500,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 1,
            "lambda_l2": 10.0,
            "num_threads": 14,
            "verbosity": -1,
            "seed": seed,
        },
        ds,
        num_boost_round=STACK_ROUNDS,
    ).predict(xva)


if not (SUBS / "submit_v4_mix_fit.csv").exists():
    stack_idx = np.array([FEATURE_NAMES.index(c) for c in STACK_FEATS])

    def stack_design(a, names):
        x = load_space_x(PRE, a)[:, stack_idx]
        if a != SUBMIT:
            x = x[cohort_mask(a)]
        return np.column_stack([member_cols(pre_store, a, names), x]).astype(np.float32)

    def stack_target(a):
        y = y_log(a)
        return (y - y.mean()).astype(np.float32)

    SRC4 = {}
    for tag, names in (("blend", V4_BLEND_MODELS), ("mix", V4_MIX_MODELS)):
        w = fit_nnls_weights(pre_store, names, RECENT, ridge=0.0)
        print(f"\n{tag} weights:")
        for n, v in sorted(zip(names, w, strict=True), key=lambda t: -t[1]):
            if v > 1e-4:
                print(f"  {n}{v}")
        oof = {a: member_cols(pre_store, a, names) @ w for a in VAL_ANCHORS}
        sub = member_cols(pre_store, SUBMIT, names) @ w
        if tag == "mix":
            stack_oof = {}
            for a in RECENT:
                rest = [b for b in RECENT if b != a]
                stack_oof[a] = fit_stack(
                    [stack_design(b, names) for b in rest],
                    [stack_target(b) for b in rest],
                    stack_design(a, names),
                )
            stack_sub = fit_stack(
                [stack_design(a, names) for a in RECENT],
                [stack_target(a) for a in RECENT],
                stack_design(SUBMIT, names),
            )
            oof = {
                a: z_score((1 - MIX_LAM) * z_score(oof[a]) + MIX_LAM * z_score(stack_oof[a]))
                if a in RECENT
                else z_score(oof[a])
                for a in VAL_ANCHORS
            }
            sub = z_score((1 - MIX_LAM) * z_score(sub) + MIX_LAM * z_score(stack_sub))
        SRC4[tag] = (oof, sub)
        np.save(LEGACY_DIR / f"v4_{tag}_sub.npy", np.asarray(sub, dtype=np.float32))

    for tag, stem in (("blend", "submit_v4_blend"), ("mix", "submit_v4_mix")):
        oof, sub = SRC4[tag]
        beta_fit, beta_flat = fit_and_flat_beta(oof)
        print(f"{tag}: beta_fit {beta_fit}  beta_flat {beta_flat}")
        for sfx, beta in (("flat", beta_flat), ("fit", beta_fit)):
            write_submission(f"{stem}_{sfx}", sub, beta, source=f"{tag}_sub.npy")

### v5 and v6

In [ ]:
V5_BETA = 1.6268

if not (SUBS / "submit_v5_beta.csv").exists():
    src_meta = json.loads((SUBS / "meta_submit_v4_blend_fit.json").read_text())
    p = submission_vector("submit_v4_blend_fit")
    z5 = (p - src_meta["level"]) / src_meta["beta"]
    z5[p <= 0.0] = -src_meta["level"] / src_meta["beta"]
    write_submission("submit_v5_beta", z5, V5_BETA, source="submit_v4_blend_fit.csv")

if not (SUBS / "submit_v6_vall.csv").exists():
    V6_VARIANTS = {
        "vall": V6_MODELS,
        "vtree": group_by_prefix(V6_MODELS, TREE_PREFIX),
        "vnn": group_by_prefix(V6_MODELS, NN_PREFIX),
    }
    for tag, names in V6_VARIANTS.items():
        w = fit_nnls_weights(pre_store, names, RECENT, ridge=0.0)
        sub = z_score(member_cols(pre_store, SUBMIT, names) @ w)
        np.save(LEGACY_DIR / f"v6_{tag}_sub.npy", sub.astype(np.float32))
        for a in VAL_ANCHORS:
            np.save(
                LEGACY_DIR / f"v6_{tag}_oof_{a}.npy",
                z_score(member_cols(pre_store, a, names) @ w).astype(np.float32),
            )
        print(
            f"{tag} n={len(names)} "
            + " ".join(f"{n}={v}" for n, v in zip(names, w, strict=True) if v > 0.01)
        )
        write_submission(f"submit_v6_{tag}", sub, V5_BETA, source=f"{tag}_sub.npy")

### v7

In [ ]:
def projected_beta(oof_new, ref_tag, ref_name, anchor_list=None):
    anchor_list = anchor_list or RECENT
    ref = {
        a: np.load(LEGACY_DIR / f"{ref_tag}_oof_{a}.npy").astype(np.float64) for a in anchor_list
    }
    rho_n = rho_at(oof_new, anchor_list)
    rho_r = rho_at(ref, anchor_list)
    b_ref = b_of(ref_name)
    b_new = b_ref * rho_n / rho_r
    print(f"  panel rho {rho_r} -> {rho_n}   B {b_ref} -> {b_new}")
    return b_new


if not (SUBS / "submit_v7_all.csv").exists():
    for tag, names in (
        ("v7all", V7_MODELS),
        ("v7tree", group_by_prefix(V7_MODELS, TREE_PREFIX)),
        ("v7nn", group_by_prefix(V7_MODELS, NN_PREFIX)),
    ):
        w7 = fit_nnls_weights(pre_store, names, RECENT, ridge=0.0)
        oof7 = {a: member_cols(pre_store, a, names) @ w7 for a in VAL_ANCHORS}
        sub7 = z_score(member_cols(pre_store, SUBMIT, names) @ w7)
        np.save(LEGACY_DIR / f"{tag}_sub.npy", sub7.astype(np.float32))
        for a in VAL_ANCHORS:
            np.save(LEGACY_DIR / f"{tag}_oof_{a}.npy", z_score(oof7[a]).astype(np.float32))
        if tag == "v7all":
            beta7 = projected_beta(oof7, "v6_vall", "v6_vall")
            write_submission(
                "submit_v7_all",
                sub7,
                beta7,
                source="v7all_sub.npy, 25 member NNLS at projected optimal beta",
            )

## The probe ladder, v8 to v12

In [ ]:
def harvest(names):
    x = np.column_stack([submission_vector(SCORED[n][0]) for n in names])
    mu = x.mean(axis=0)
    xc = x - mu
    gram = xc.T @ xc / len(xc)
    eu2 = W_TOTAL + TARGET_MEAN * TARGET_MEAN
    cov = np.array([
        (eu2 + float((x[:, i] ** 2).mean()) - SCORED[n][1] ** 2) / 2.0 - TARGET_MEAN * mu[i]
        for i, n in enumerate(names)
    ])
    return cov, gram, xc


def solve_harvest(names, ridge=0.0):
    cov, gram, xc = harvest(names)
    w = np.linalg.solve(gram + ridge * np.diag(np.diag(gram)), cov)
    mse = W_TOTAL - 2.0 * float(w @ cov) + float(w @ gram @ w)
    raw = xc @ w
    return {
        "names": names,
        "weights": w,
        "cov": cov,
        "raw": raw,
        "predicted": float(np.sqrt(max(mse, 0.0))),
        "beta": float(raw.std()),
    }


V8_TAGS = [
    ("v6_vall", "v6_vall"),
    ("v6_vtree", "v6_vtree"),
    ("v6_vnn", "v6_vnn"),
    ("v7all", "v7_all"),
]


def probe_opt(ridge):
    z = np.column_stack([
        np.load(LEGACY_DIR / f"{tag}_sub.npy").astype(np.float64) for tag, _ in V8_TAGS
    ])
    z = (z - z.mean(axis=0)) / z.std(axis=0)
    g = z.T @ z / len(z)
    b = np.array([
        (V5_BETA * V5_BETA + W_TOTAL - SCORED[key][1] ** 2) / (2 * V5_BETA) for _, key in V8_TAGS
    ])
    w = np.linalg.solve(g + ridge * np.eye(len(V8_TAGS)), b)
    var = float(w @ g @ w)
    return z_score(z @ w), float(w @ b) / var * np.sqrt(var)


for stem, ridge in (("submit_v8_probeopt", 0.01), ("submit_v9_opt4", 0.0)):
    if (SUBS / f"{stem}.csv").exists():
        continue
    zopt, beta = probe_opt(ridge)
    np.save(LEGACY_DIR / f"{stem.replace('submit_', '')}_sub.npy", zopt.astype(np.float32))
    write_submission(stem, zopt, beta, source=f"ridge {ridge} optimum of four exact LB probes")

In [ ]:
WELL_CONDITIONED = [
    "v11_spaced",
    "catboost_v1",
    "catboost_v2",
    "cb_v2_dropz",
    "chronos2",
    "chronosL",
]
V10_ALPHA, P1_ALPHA, P2_ALPHA, P3_ALPHA = 0.01460, 0.1, 0.05, 0.03

if not (SUBS / "submit_v12_harvest.csv").exists():
    cur9 = submission_vector("submit_v9_opt4")
    b9 = b_of("v9_opt4")

    resid_target = {}
    for a in RECENT:
        z_blend = z_score(np.load(LEGACY_DIR / f"v7all_oof_{a}.npy").astype(np.float64))
        y = y_log(a)
        beta_a = float((z_blend * (y - y.mean())).mean())
        resid_target[a] = y - (beta_a * z_blend + y.mean())
    resid_w = {}
    for a in RECENT:
        rest = [b for b in RECENT if b != a]
        xs = np.concatenate([member_cols(pre_store, b, V7_MODELS) for b in rest])
        ys = np.concatenate([resid_target[b] for b in rest])
        resid_w[a] = np.linalg.solve(
            xs.T @ xs / len(xs) + 1e-2 * np.eye(len(V7_MODELS)), xs.T @ ys / len(xs)
        )
    w_res = np.mean([resid_w[a] for a in RECENT], axis=0)
    d_resid = z_score(
        orthogonalise(member_cols(pre_store, SUBMIT, V7_MODELS) @ w_res, z_score(cur9))
    )
    np.save(LEGACY_DIR / "d_resid.npy", d_resid.astype(np.float32))

    write_submission(
        "submit_p1_resid",
        mix_raw(cur9, d_resid, P1_ALPHA),
        b9 / np.sqrt(1.0 + P1_ALPHA**2),
        source="v9 plus 0.1 times OOF-optimal residual direction",
    )
    write_submission(
        "submit_v10_resid",
        mix_raw(cur9, d_resid, V10_ALPHA),
        float(np.sqrt(b9**2 + (V10_ALPHA * b9) ** 2)),
        source=f"v9 plus OOF-optimal {V10_ALPHA} of the residual direction",
    )

    cur10 = submission_vector("submit_v10_resid")
    b10 = b_of("v10_resid")
    sp_names = [f"{m}_sp" for m in SPACED_MEMBERS]
    sp_all = z_score(
        member_cols(pre_store, SUBMIT, sp_names)
        @ fit_nnls_weights(pre_store, sp_names, RECENT, ridge=0.0)
    )
    ct_all = z_score(
        member_cols(pre_store, SUBMIT, SPACED_MEMBERS)
        @ fit_nnls_weights(pre_store, SPACED_MEMBERS, RECENT, ridge=0.0)
    )
    d_spaced = z_score(orthogonalise(sp_all - ct_all, z_score(cur10)))
    np.save(LEGACY_DIR / "d_spaced.npy", d_spaced.astype(np.float32))
    beta_p2 = b10 / np.sqrt(1.0 + P2_ALPHA**2)
    write_submission(
        "submit_p2_spaced",
        mix_raw(cur10, d_spaced, P2_ALPHA),
        beta_p2,
        source="v10 optimum plus 0.05 of (spaced minus contiguous) trees",
    )
    cov_sp = cov_from_probe(SCORED["p2_spaced"][1], beta_p2, P2_ALPHA, b10)
    print(f"anchor spacing direction: measured Cov {cov_sp:+.5f}")
    write_submission(
        "submit_v11_spaced",
        mix_raw(cur10, d_spaced, cov_sp / b10),
        float(np.sqrt(b10**2 + cov_sp**2)),
        source="v10 optimum plus the exactly measured anchor-spacing direction",
    )

    cur11 = submission_vector("submit_v11_spaced")
    b11 = b_of("v11_spaced")
    d_sp9 = z_score(orthogonalise(sp_all, z_score(cur11)))
    write_submission(
        "submit_p3_spaced9",
        mix_raw(cur11, d_sp9, P3_ALPHA),
        b11 / np.sqrt(1.0 + P3_ALPHA**2),
        source="v11 plus 0.03 of the 9-member spaced direction",
    )

    res12 = solve_harvest(WELL_CONDITIONED, ridge=0.0)
    for n, w in zip(res12["names"], res12["weights"], strict=True):
        print(f"{n}{w}")
    print(f"predicted {res12['predicted']}  beta {res12['beta']}")
    write_submission(
        "submit_v12_harvest",
        res12["raw"],
        res12["beta"],
        source="least squares over six exactly probed submissions, ridge 0",
    )

## The diversity fleet

In [ ]:
SHIFT_ANCHORS = ANCHORS[::2][:10]
SHIFT_PATH = ARCH / "feature_shift.npy"
if not SHIFT_PATH.exists():
    tr_rows = [load_space_x(PRE, a)[COHORT[ANCHOR_IDX[a]]] for a in SHIFT_ANCHORS]
    tr = np.concatenate(tr_rows)
    del tr_rows
    gc.collect()
    sub_x = load_space_x(PRE, SUBMIT)[COHORT[-1]]
    sd = tr.std(axis=0)
    np.save(
        SHIFT_PATH,
        (np.abs(sub_x.mean(axis=0) - tr.mean(axis=0)) / np.maximum(sd, 1e-9)).astype(np.float64),
    )
    del tr, sub_x
    gc.collect()
FEATURE_SHIFT = np.load(SHIFT_PATH)
print(
    f"feature shift: mean {FEATURE_SHIFT.mean()}  "
    f"beyond 0.3 sd {(FEATURE_SHIFT > 0.3).sum()}  max {FEATURE_SHIFT.max()}"
)

In [ ]:
VIEW_RULES = {
    "recency": (
        (
            "days_since",
            "tenure_days",
            "order_gap_365d",
            "active_gap_90d",
            "recency_over_gap",
            "last_order_gap",
            "streak_",
            "os_gap_",
            "os_recency",
            "os_span",
            "os_rate",
            "os_rec_over_gap",
            "os_n_orders",
            "order_pos_sd",
            "order_centroid_recency",
            "blk_n_avail",
            "rank_days_since",
        ),
        (),
    ),
    "monetary": (("gmv", "loggmv", "basket_", "blk_lgmv", "wk_lgmv", "os_lgmv", "_loggmv"), ()),
    "funnel": (
        (
            "searches_",
            "to_cart_sum",
            "search_to_cart",
            "cat_to_cart",
            "cat_sum_",
            "search_only",
            "cat_only",
            "both_ch",
            "neither_ch",
            "cat_any_share",
            "neither_share",
            "cat_cart_share",
            "cart_to_ord",
            "ewm_logsearches",
            "ewm_logto_cart",
            "has_cart_sum",
            "searches_trend",
            "searches_m",
            "rank_searches",
        ),
        ("gmv",),
    ),
    "counts": (
        (
            "to_ord_sum",
            "search_to_ord_sum",
            "cat_to_ord_sum",
            "has_order_sum",
            "active_sum",
            "active_rate",
            "active_trend",
            "active_m",
            "to_ord_trend",
            "to_ord_m",
            "order_day_rate",
            "blk_ord_",
            "blk_act_",
            "blk_has_",
            "wk_act_",
            "wk_ord_",
            "dow_",
            "ewm_has_order",
            "ewm_active",
            "ewm_logto_ord",
        ),
        ("gmv",),
    ),
    "short": (
        ("_7d", "_14d", "_30d", "ewm_gmv_3", "ewm_gmv_7", "ewm_loggmv_3", "ewm_loggmv_7"),
        (),
    ),
    "long": (("_180d", "_365d", "blk_", "wk_", "dow_", "os_", "_240", "_120"), ()),
    "gmvblind": (("",), ("gmv", "basket_", "lgmv")),
    "ranks": (("rank_",), ()),
}
ROW_RULES = {
    "buyers": ("has_order_sum_365d", "gt", 0.0),
    "heavy_buyers": ("has_order_sum_365d", "ge", 4.0),
    "zero_gmv_dropped": ("gmv_sum_365d", "gt", 0.0),
    "sparse": ("active_sum_365d", "le", 48.0),
    "dense": ("active_sum_365d", "gt", 48.0),
    "recent_only": ("days_since_last_order", "le", 60.0),
}


def view_idx(view, with_v4=True):
    names = FEATURE_NAMES if with_v4 else BASE_NAMES
    if view.startswith(("stable_", "shifted_")):
        thr = float(view.split("_")[1]) / 100.0
        s = FEATURE_SHIFT[: len(names)]
        idx = np.flatnonzero(s < thr if view.startswith("stable_") else s >= thr)
    elif view.startswith("rand_"):
        _, seed, pct = view.split("_")
        k = max(8, round(len(names) * int(pct) / 100))
        idx = np.sort(np.random.default_rng(int(seed)).choice(len(names), k, replace=False))
    else:
        keep, drop = VIEW_RULES[view]
        idx = np.array([
            j
            for j, nm in enumerate(names)
            if any(k in nm for k in keep) and not any(d in nm for d in drop)
        ])
    return idx.astype(np.int64)


def row_mask(rule, x, with_v4=True):
    if rule is None:
        return None
    col, op, thr = ROW_RULES[rule]
    v = x[:, (FEATURE_NAMES if with_v4 else BASE_NAMES).index(col)]
    return {"gt": v > thr, "ge": v >= thr, "le": v <= thr, "lt": v < thr}[op]


def adversarial_weights(xtr, xsub, seed=42, rounds=200):
    n = min(len(xtr), len(xsub))
    rng = np.random.default_rng(seed)
    x = np.vstack([
        xtr[rng.choice(len(xtr), n, replace=False)],
        xsub[rng.choice(len(xsub), n, replace=False)],
    ])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    m = lgb.train(
        {
            "objective": "binary",
            "num_leaves": 31,
            "learning_rate": 0.05,
            "verbosity": -1,
            "seed": seed,
            "num_threads": 14,
        },
        lgb.Dataset(x, y),
        num_boost_round=rounds,
    )
    p = np.clip(m.predict(xtr), 1e-4, 1 - 1e-4)
    w = p / (1.0 - p)
    return (w / w.mean()).clip(0.05, 20.0).astype(np.float32)

In [ ]:
_FC = {
    "kind": "cat",
    "v4": True,
    "iters": 1200,
    "lr": 0.03,
    "depth": 8,
    "l2": 30.0,
    "n_oof": 2,
    "n_sub": 4,
}
_FM = {
    "kind": "mlp",
    "v4": True,
    "epochs": 3,
    "width": 1024,
    "drop": 0.3,
    "lr": 3e-3,
    "n_oof": 2,
    "n_sub": 4,
}
FLEET = {
    "dv_recency": {**_FC, "view": "recency"},
    "dv_monetary": {**_FC, "view": "monetary"},
    "dv_funnel": {**_FC, "view": "funnel"},
    "dv_counts": {**_FC, "view": "counts"},
    "dv_short": {**_FC, "view": "short"},
    "dv_long": {**_FC, "view": "long"},
    "dv_gmvblind": {**_FC, "view": "gmvblind"},
    "dr_buyers": {**_FC, "rows": "buyers"},
    "dr_heavy": {**_FC, "rows": "heavy_buyers"},
    "dr_zerodrop": {**_FC, "rows": "zero_gmv_dropped"},
    "dr_sparse": {**_FC, "rows": "sparse"},
    "dr_dense": {**_FC, "rows": "dense"},
    "dr_recent": {**_FC, "rows": "recent_only"},
    "dl_mae": {**_FC, "loss": "MAE"},
    "dl_q30": {**_FC, "loss": "Quantile:alpha=0.3"},
    "dl_q70": {**_FC, "loss": "Quantile:alpha=0.7"},
    "dl_huber": {**_FC, "loss": "Huber:delta=1.0"},
    "dt_tweedie": {**_FC, "target": "raw_gmv", "loss": "Tweedie:variance_power=1.5"},
    "dt_poisson": {**_FC, "target": "raw_gmv", "loss": "Poisson"},
    "dt_sqrt": {**_FC, "target": "sqrt_gmv"},
    "dt_rank": {**_FC, "target": "rank"},
    "dm_gmvblind": {**_FM, "view": "gmvblind", "n_oof": 4, "n_sub": 8},
    "dm_short": {**_FM, "view": "short", "n_oof": 4, "n_sub": 8},
    "dm_counts": {**_FM, "view": "counts", "n_oof": 4, "n_sub": 8},
    "ds_full": {**_FC},
    "ds_mfull": {**_FM, "n_oof": 4, "n_sub": 8},
    "ds_stable05": {**_FC, "view": "stable_5"},
    "ds_stable10": {**_FC, "view": "stable_10"},
    "ds_stable15": {**_FC, "view": "stable_15"},
    "ds_stable20": {**_FC, "view": "stable_20"},
    "ds_stable30": {**_FC, "view": "stable_30"},
    "ds_stable50": {**_FC, "view": "stable_50"},
    "ds_shifted30": {**_FC, "view": "shifted_30"},
    "ds_mstable30": {**_FM, "view": "stable_30", "n_oof": 4, "n_sub": 8},
    "dw_adv": {**_FC, "weights": "adversarial"},
    "dw_advstable": {**_FC, "weights": "adversarial", "view": "stable_30"},
    "da_depth3": {**_FC, "depth": 3, "iters": 2500},
    "da_depth12": {**_FC, "depth": 12, "iters": 500},
    "dv_ranks": {**_FC, "view": "ranks"},
}
SUBSPACE = {
    f"dq_{i:02d}": {**_FC, "view": f"rand_{i}_{pct}"}
    for i, pct in enumerate([8, 8, 15, 15, 15, 15, 15, 15, 15, 15, 30, 30, 30, 30, 30, 50])
}
V30_VIEWS = {
    f"dq_{s}_{p}": {**_FC, "view": f"rand_{s}_{p}", "depth": 6, "l2": 6.0, "iters": 1500}
    for p in (5, 10, 20, 35, 60)
    for s in (0, 1, 2)
}
V30_VIEWS |= {
    f"dv2_{v}": {**_FC, "view": v, "depth": 6, "l2": 6.0, "iters": 1500}
    for v in ("gmvblind", "short", "recency", "funnel", "counts", "monetary", "long")
}
RANEF_CFG = {"ranef_src": {**_FC, "n_oof": 2, "n_sub": 2}}
FLEET_ALL = {**FLEET, **SUBSPACE, **V30_VIEWS, **RANEF_CFG}
print(f"{len(FLEET)} fleet + {len(SUBSPACE)} subspace + {len(V30_VIEWS)} v30 views")

In [ ]:
FLEET_FOLDS = [*PANEL, SUBMIT]
V30_FOLDS = ["2025-09-24", "2025-10-22", "2025-12-03", SUBMIT]


def target_for(cfg, ln, mu_row):
    mode = cfg.get("target", "centred")
    if mode == "centred":
        return (ln - mu_row).astype(np.float32), "none"
    if mode == "raw_gmv":
        return np.expm1(ln).astype(np.float32), "log1p"
    if mode == "sqrt_gmv":
        return np.sqrt(np.expm1(ln)).astype(np.float32), "square_log1p"
    if mode == "rank":
        return avg_rank(ln, True).astype(np.float32), "none"
    raise ValueError(mode)


def post_transform(pred, how):
    if how == "none":
        return pred
    if how == "log1p":
        return np.log1p(np.clip(pred, 0.0, None))
    return np.log1p(np.clip(pred, 0.0, None) ** 2)


def fleet_store(name):
    return MEMBER_DIR / f"fl_{name}"


def run_fleet(names, folds=None, n_train=10, min_gap=28, suffix=""):
    folds = folds or FLEET_FOLDS
    for va in folds:
        is_sub = va == SUBMIT
        todo = [
            n
            for n in names
            if not (
                (MEMBER_DIR / f"fl_{n}{suffix}") / ("sub.npy" if is_sub else f"oof_{va}.npy")
            ).exists()
        ]
        if not todo:
            continue
        tr = fold_anchors(va, n_train, min_gap)
        for is_rg in (False, True):
            group = [n for n in todo if (FLEET_ALL[n]["kind"] in RG_KINDS) == is_rg]
            if not group:
                continue
            xtr, xva, ln, mu_row, _, _, _, _ = stack_fold(PRE, tr, va, is_rg)
            for name in group:
                cfg = FLEET_ALL[name]
                out_dir = MEMBER_DIR / f"fl_{name}{suffix}"
                out_dir.mkdir(parents=True, exist_ok=True)
                y_t, how = target_for(cfg, ln, mu_row)
                idx = view_idx(cfg["view"]) if "view" in cfg else np.arange(xtr.shape[1])
                xt = np.ascontiguousarray(xtr[:, idx])
                xv = np.ascontiguousarray(xva[:, idx])
                rows = row_mask(cfg.get("rows"), xtr)
                yy = y_t
                if rows is not None:
                    xt, yy = xt[rows], y_t[rows]
                wts = adversarial_weights(xt, xv) if cfg.get("weights") == "adversarial" else None
                preds = []
                for seed in SEED_POOL[: cfg["n_sub"] if is_sub else cfg["n_oof"]]:
                    if cfg["kind"] == "cat":
                        booster = CatBoostRegressor(
                            **cat_params(cfg, seed, cfg.get("loss", "RMSE"))
                        )
                        booster.fit(Pool(xt, yy, weight=wts))
                        preds.append(
                            post_transform(np.asarray(booster.predict(xv), np.float64), how)
                        )
                        del booster
                    else:
                        pred, _ = fit_torch_member(
                            "mlp",
                            cfg,
                            xt,
                            xv,
                            yy,
                            (yy > 0).astype(np.float32),
                            np.column_stack([
                                (yy > 0).astype(np.float32),
                                np.clip(yy, -3, 6).astype(np.float32) ** 2 / 10.0,
                            ]).astype(np.float32),
                            seed,
                            SPACES["pre"]["cuda_seed"],
                        )
                        preds.append(post_transform(pred, how))
                    gc.collect()
                np.save(
                    out_dir / ("sub.npy" if is_sub else f"oof_{va}.npy"),
                    np.mean(preds, axis=0).astype(np.float32),
                )
                print(f"  fleet {name}{suffix} {va} cols={idx.size}", flush=True)
                del xt, xv, preds
                gc.collect()
            del xtr, xva, ln, mu_row
            gc.collect()


run_fleet(list(FLEET))
run_fleet(list(SUBSPACE))
run_fleet(list(RANEF_CFG))

### Directions and probes

In [ ]:
PROBE_ALPHA = 0.020
FLEET_ALPHA = 0.02162
FLEET_COV = 0.01655
MIN_SHARE = 0.30


def pooled_z(by_anchor, anchor_list):
    return np.concatenate([z_score(by_anchor[a]) for a in anchor_list])


def pooled_target(anchor_list):
    return np.concatenate([target_z(a) for a in anchor_list])


def orthogonal_gain(cand, base, anchor_list, b_current):
    u = pooled_target(anchor_list)
    c = pooled_z(cand, anchor_list)
    b = pooled_z(base, anchor_list)
    n = len(u)
    rho, rho_base, g = float(c @ u / n), float(b @ u / n), float(c @ b / n)
    cov = SIGMA * (rho - g * rho_base) / np.sqrt(max(1.0 - g * g, 1e-12))
    b_new = float(np.sqrt(b_current**2 + cov**2))
    return {"rho": rho, "corr": g, "cov": float(cov), "b_new": b_new, "lb": score_from_b(b_new)}


def screen(cands, base, anchor_list, b_current):
    out = {}
    print(f"{'candidate'}{'rho'}{'corr'}{'cov'}{'proj LB'}")
    for name, pred in cands.items():
        r = orthogonal_gain(pred, base, anchor_list, b_current)
        out[name] = r
        print(f"{name}{r['rho']}{r['corr']}{r['cov']}{r['lb']}")
    return out


def orth_cols(x, base):
    b = z_score(base)
    return x - np.outer(b, (b @ x) / (b @ b))


def bundle_weights(store, names, base, anchor_list, ridge=0.05):
    xs, ys = [], []
    for a in anchor_list:
        b = z_score(base[a])
        ys.append(orthogonalise(target_z(a), b))
        xs.append(orth_cols(member_cols(store, a, names), b))
    x, y = np.concatenate(xs), np.concatenate(ys)
    g = x.T @ x / len(x)
    c = x.T @ y / len(x)
    return np.linalg.solve(g + ridge * np.diag(np.diag(g)), c)


PANEL_BASE = [m for m in PRE_POOL if m not in ("mlpf_o", "seq_c", "tabm_a", "tabm_b")]
w_panel = fit_nnls_weights(pre_store, PANEL_BASE, PANEL, ridge=0.02)
BASE_PANEL = {a: member_cols(pre_store, a, PANEL_BASE) @ w_panel for a in [*PANEL, SUBMIT]}
print(
    "panel base weights:",
    {n: round(v, 4) for n, v in zip(PANEL_BASE, w_panel, strict=True) if v > 1e-3},
)

In [ ]:
if not (SUBS / "submit_v13_fleet.csv").exists():
    cur12 = submission_vector("submit_v12_harvest")
    b12 = b_of("v12_harvest")
    fleet_names = list(FLEET)
    rep = screen(
        {m: {a: member_z(fleet_store, m, a) for a in PANEL} for m in fleet_names},
        BASE_PANEL,
        PANEL,
        b12,
    )
    KEEP_FLEET = [m for m in fleet_names if abs(rep[m]["cov"]) > 0.01]
    print("kept", KEEP_FLEET)
    loo = {}
    for ridge in (0.02, 0.05, 0.1, 0.25, 0.5):
        covs = []
        for a in PANEL:
            rest = [b for b in PANEL if b != a]
            w = bundle_weights(fleet_store, KEEP_FLEET, BASE_PANEL, rest, ridge)
            covs.append(
                orthogonal_gain(
                    {a: member_cols(fleet_store, a, KEEP_FLEET) @ w}, BASE_PANEL, [a], b12
                )["cov"]
            )
        loo[ridge] = float(np.mean(covs))
        print(f"  ridge {ridge} LOO cov {loo[ridge]}")
    RIDGE_FLEET = max(loo, key=lambda k: loo[k])
    w_bundle = bundle_weights(fleet_store, KEEP_FLEET, BASE_PANEL, PANEL, RIDGE_FLEET)
    d_fleet = z_score(
        orthogonalise(
            member_cols(fleet_store, SUBMIT, KEEP_FLEET) @ w_bundle, z_score(BASE_PANEL[SUBMIT])
        )
    )
    d_fleet = z_score(orthogonalise(d_fleet, z_score(cur12)))
    np.save(LEGACY_DIR / "d_fleet.npy", d_fleet.astype(np.float32))
    (LEGACY_DIR / "fleet.json").write_text(
        json.dumps(
            {"kept": KEEP_FLEET, "ridge": RIDGE_FLEET, "weights": w_bundle.tolist()}, indent=1
        )
    )
    write_submission(
        "submit_v13_fleet",
        mix_raw(cur12, d_fleet, FLEET_ALPHA),
        float(np.sqrt(b12**2 + FLEET_COV**2)),
        source=f"v12_harvest plus {FLEET_ALPHA} of the orthogonalised diversity fleet",
    )

### The p_ and q_ probes

In [ ]:
SPAN_VARIANTS = {"p12": (12, 28), "p3": (3, 0), "pall": (20, 14)}
SPAN_MEMBERS = ["catf_b", "catf_d", "xgbf_a", "lgbf_b", "mlpf_a", "mlpf_d"]
PROBE_PAIRS = [
    ("shift", "ds_stable30", "ds_full"),
    ("shift20", "ds_stable20", "ds_full"),
    ("adv", "dw_adv", "ds_full"),
    ("mask_stable05", "ds_stable05", "ds_full"),
    ("mask_stable10", "ds_stable10", "ds_full"),
    ("mask_stable15", "ds_stable15", "ds_full"),
]
PROBE_ORDER = [
    "shift",
    "shift20",
    "adv",
    "mask_stable10",
    "mask_stable15",
    "mask_stable05",
    "span_pall",
    "span_p12",
    "ranef",
    "span_p3",
]
PROBE_STEM = {
    "shift": "submit_p_shift",
    "shift20": "submit_p_shift20",
    "adv": "submit_p_adv",
    "mask_stable05": "submit_q_mask_stable05",
    "mask_stable10": "submit_q_mask_stable10",
    "mask_stable15": "submit_q_mask_stable15",
    "span_pall": "submit_q_span_pall",
    "span_p12": "submit_q_span_p12",
    "span_p3": "submit_q_span_p3",
    "ranef": "submit_q_ranef",
}
RANEF_SHRINK = 3.0
RANEF_SOURCES = [a for a in ANCHORS if "2025-05-07" <= a <= "2025-12-10"]


def ranef_column(anchor):
    cut = date.fromisoformat(anchor) - timedelta(days=2 * HORIZON_DAYS)
    use = [s for s in RANEF_SOURCES if date.fromisoformat(s) <= cut]
    stack = []
    for s in use:
        raw = np.load(fleet_store("ranef_src") / f"oof_{s}.npy").astype(np.float64)
        m = cohort_mask(s)
        z = z_score(raw[m] if raw.shape[0] == N_USERS else raw)
        y = y_log(s)
        beta = float((z * (y - y.mean())).mean())
        out = np.full(N_USERS, np.nan)
        out[m] = y - (beta * z + y.mean())
        stack.append(out)
    stack = np.stack(stack)
    ok = ~np.isnan(stack)
    e = np.where(ok, stack, 0.0).sum(axis=0) / (ok.sum(axis=0) + RANEF_SHRINK)
    return e if anchor == SUBMIT else e[cohort_mask(anchor)]


if not (SUBS / "submit_p_shift.csv").exists():
    for tag, (n_tr, gap) in SPAN_VARIANTS.items():
        run_fleet(
            [m for m in SPAN_MEMBERS if m in FLEET_ALL],
            n_train=n_tr,
            min_gap=gap,
            suffix=f"_{tag}",
        )
    cur12 = submission_vector("submit_v12_harvest")
    b12 = b_of("v12_harvest")
    zb12 = z_score(cur12)

    def direction(v):
        d = orthogonalise(z_score(v), z_score(BASE_PANEL[SUBMIT]))
        return z_score(orthogonalise(d, zb12))

    DIRS = {}
    for tag, member, twin in PROBE_PAIRS:
        DIRS[tag] = direction(
            z_score(member_z(fleet_store, member, SUBMIT))
            - z_score(member_z(fleet_store, twin, SUBMIT))
        )
    w_span = fit_nnls_weights(pre_store, SPAN_MEMBERS, PANEL, ridge=0.02)
    b_ref = member_cols(pre_store, SUBMIT, SPAN_MEMBERS) @ w_span
    for tag in SPAN_VARIANTS:

        def span_store(name, t=tag):
            return MEMBER_DIR / f"fl_{name}_{t}"

        bv = member_cols(span_store, SUBMIT, SPAN_MEMBERS) @ w_span
        DIRS[f"span_{tag}"] = direction(z_score(bv) - z_score(b_ref))
    DIRS["ranef"] = direction(ranef_column(SUBMIT))

    basis = [z_score(np.load(LEGACY_DIR / "d_fleet.npy").astype(np.float64))]
    kept_probe = []
    print(f"{'direction'}{'residual share'}{'kept'}")
    for n in PROBE_ORDER:
        d = DIRS[n].astype(np.float64).copy()
        for b in basis:
            d -= b * float(b @ d / (b @ b))
        share = float(d.std() / DIRS[n].std())
        keep = share >= MIN_SHARE
        print(f"{n}{share}{keep}")
        if keep:
            basis.append(z_score(d))
            kept_probe.append(n)
    D_PROBE = np.column_stack(basis)
    np.save(LEGACY_DIR / "probe_dirs.npy", D_PROBE.astype(np.float32))
    (LEGACY_DIR / "probe_dirs.json").write_text(json.dumps({"names": kept_probe}, indent=1))

    beta_probe = b12 / np.sqrt(1.0 + PROBE_ALPHA**2)
    for i, n in enumerate(kept_probe, start=1):
        write_submission(
            PROBE_STEM[n],
            mix_raw(cur12, D_PROBE[:, i], PROBE_ALPHA),
            beta_probe,
            source=f"v12_harvest plus {PROBE_ALPHA} of the orthogonalised {n} direction",
        )
    pz = np.expm1(submission_vector("submit_p_shift20"))
    pz[np.argsort(pz)[: int(0.01 * N_USERS)]] = 0.0
    write_raw_submission(
        "submit_p_shift20_zeros_1", pz, source="p_shift20 with the lowest 1 percent floored to zero"
    )

### v21 and v22: banking the measured directions

In [ ]:
if not (SUBS / "submit_v22_merge.csv").exists():
    cur12 = submission_vector("submit_v12_harvest")
    b12 = b_of("v12_harvest")
    D_PROBE = np.load(LEGACY_DIR / "probe_dirs.npy").astype(np.float64)
    kept_probe = json.loads((LEGACY_DIR / "probe_dirs.json").read_text())["names"]
    beta_probe = b12 / np.sqrt(1.0 + PROBE_ALPHA**2)

    covs = {"fleet": FLEET_COV}
    for n in kept_probe:
        key = {v: k for k, v in SCORED.items()}.get(PROBE_STEM[n])
        if key is None or PROBE_STEM[n] not in {v[0] for v in SCORED.values()}:
            continue
        covs[n] = cov_from_probe(SCORED[key][1], beta_probe, PROBE_ALPHA, b12)
        print(f"{n}measured Cov {covs[n]}")

    z21 = z_score(cur12)
    for i, n in enumerate(["fleet", *kept_probe]):
        if n in covs:
            z21 = z21 + (covs[n] / b12) * z_score(D_PROBE[:, i])
    b21 = float(np.sqrt(b12**2 + sum(c * c for c in covs.values())))
    write_submission(
        "submit_v21_banked",
        z21,
        b21,
        source="v12_harvest plus six exactly measured orthogonal directions at their covariances",
    )

    cur21 = submission_vector("submit_v21_banked")
    b21m = b_of("v21_banked")
    d_pool = z_score(
        orthogonalise(
            z_score(member_z(fleet_store, "ds_stable20", SUBMIT))
            - z_score(member_z(fleet_store, "ds_full", SUBMIT)),
            z_score(cur21),
        )
    )
    write_submission(
        "submit_r_pool_s20",
        mix_raw(cur21, d_pool, PROBE_ALPHA),
        b21m / np.sqrt(1.0 + PROBE_ALPHA**2),
        source="v21_banked plus 0.020 of the pooled stable_20 mask direction",
    )

    i20 = 1 + kept_probe.index("shift20")
    i10 = 1 + kept_probe.index("mask_stable10")
    d_r = z_score(
        orthogonalise(z_score(D_PROBE[:, i20]) + z_score(D_PROBE[:, i10]), z_score(cur12))
    )
    write_submission(
        "submit_r_blend_shift20_mask10",
        mix_raw(cur12, d_r, PROBE_ALPHA),
        b12 / np.sqrt(1.0 + PROBE_ALPHA**2),
        source="v12_harvest plus the shift20 and mask_stable10 directions combined",
    )

    merge_names = [n for n in SCORED if (SUBS / f"{SCORED[n][0]}.csv").exists()]
    cov_m, gram_m, xc_m = harvest(merge_names)
    w_m = np.linalg.solve(gram_m + 1e-4 * np.diag(np.diag(gram_m)), cov_m)
    raw_m = xc_m @ w_m
    print(
        f"v22 merge over {len(merge_names)} scored vectors, leverage "
        f"{float(np.abs(w_m) @ np.sqrt(np.diag(gram_m)) / raw_m.std())}"
    )
    write_submission(
        "submit_v22_merge",
        raw_m,
        float(raw_m.std()),
        source="ridge 1e-4 least squares over both teams' scored submissions",
    )

## Structural probes

In [ ]:
CAP_HIST = 250
CAP_DIR = DATA / "capped" / f"h{CAP_HIST}"
CAP_DIR.mkdir(parents=True, exist_ok=True)

if not (CAP_DIR / f"X_{SUBMIT}.npy").exists():
    data = pl.read_parquet(TRAIN_PATH).sort("user_id", "event_date")
    a_cap = date.fromisoformat(SUBMIT)
    lo_date = max(a_cap - timedelta(days=CAP_HIST - 1), DATE_MIN)
    win = data.filter(pl.col("event_date").is_between(lo_date, a_cap))
    missing = np.setdiff1d(USER_IDS, win["user_id"].unique().to_numpy())
    if len(missing):
        pad = pl.DataFrame({"user_id": missing}).with_columns(
            pl.lit(lo_date).cast(win.schema["event_date"]).alias("event_date"),
            *[
                pl.lit(0).cast(dt).alias(c)
                for c, dt in win.schema.items()
                if c not in ("user_id", "event_date")
            ],
        )
        win = pl.concat([win, pad.select(win.columns)]).sort("user_id", "event_date")
    dead = np.isin(win["user_id"].to_numpy(), missing)
    print(f"capped window {lo_date}..{a_cap}, {len(missing)} users padded")
    (CAP_DIR / "meta.json").write_text(
        json.dumps({"hist": CAP_HIST, "lo": lo_date.isoformat(), "padded": len(missing)})
    )
    np.save(CAP_DIR / "dead.npy", dead)
    win.write_parquet(CAP_DIR / "window.parquet")
    del data, win
    gc.collect()
    print(
        f"capped frame written; rerun the dense grid and feature cells against it to fill {CAP_DIR}"
    )

In [ ]:
STRUCT_ALPHA = 0.02
YEARAGO_ALPHA, CAP_ALPHA = 0.02, 0.03
YEAR_DAYS, BASE_DAYS = 365, 90
FACTOR_WEEKS, FACTOR_RANK = 26, 20


def daily_grid_sums():
    gmv_path, act_path = ARCH / "daily_gmv.npy", ARCH / "daily_act.npy"
    if not gmv_path.exists():
        data = pl.read_parquet(TRAIN_PATH).sort("user_id", "event_date")
        rows = np.searchsorted(USER_IDS, data["user_id"].to_numpy())
        day = (
            data
            .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
            .to_series()
            .to_numpy()
        )
        flat = rows * N_DAYS + day
        g = np.zeros(N_USERS * N_DAYS, dtype=np.float32)
        np.add.at(g, flat, data["gmv"].to_numpy().astype(np.float32))
        np.save(gmv_path, g.reshape(N_USERS, N_DAYS))
        a = np.zeros(N_USERS * N_DAYS, dtype=np.uint8)
        a[flat] = 1
        np.save(act_path, a.reshape(N_USERS, N_DAYS))
        del data, g, a, flat, rows, day
        gc.collect()
    return (
        np.load(gmv_path, mmap_mode="r"),
        np.load(act_path, mmap_mode="r"),
    )


DAILY_GMV, DAILY_ACT = daily_grid_sums()


def year_ago_lift(anchor):
    t = (date.fromisoformat(anchor) - DATE_MIN).days
    lo_w = max(t - YEAR_DAYS + 1, 0)
    hi_w = t - YEAR_DAYS + HORIZON_DAYS
    lo_b, hi_b = hi_w + 1, min(hi_w + BASE_DAYS, N_DAYS - 1)
    n_base = max(hi_b - hi_w, 1)
    rate = HORIZON_DAYS / n_base
    win_g = np.asarray(DAILY_GMV[:, lo_w : hi_w + 1]).sum(axis=1, dtype=np.float64)
    win_a = np.asarray(DAILY_ACT[:, lo_w : hi_w + 1]).sum(axis=1, dtype=np.float64)
    base_g = np.asarray(DAILY_GMV[:, lo_b : hi_b + 1]).sum(axis=1, dtype=np.float64)
    base_a = np.asarray(DAILY_ACT[:, lo_b : hi_b + 1]).sum(axis=1, dtype=np.float64)
    return np.column_stack([
        np.log1p(win_g),
        win_a,
        np.log1p(win_g) - np.log1p(base_g * rate),
        win_a - base_a * rate,
        win_g / np.maximum(win_g + base_g, 1e-9),
    ])


def btyd_rate(anchor):
    x = load_space_x(PRE, anchor)
    n = x[:, BASE_NAMES.index("has_order_sum_365d")]
    rec = np.clip(x[:, BASE_NAMES.index("days_since_last_order")], 0.0, None)
    span = np.maximum(x[:, BASE_NAMES.index("tenure_days")], 1.0)
    return np.log1p(n) - np.log1p(rec) + np.log1p(n / span)


def weekly_factor_direction(anchor):
    from sklearn.utils.extmath import randomized_svd

    t = (date.fromisoformat(anchor) - DATE_MIN).days
    span = 7 * FACTOR_WEEKS
    lo = max(t - span + 1, 0)
    block = np.asarray(DAILY_GMV[:, lo : t + 1], dtype=np.float64)
    pad = span - block.shape[1]
    if pad > 0:
        block = np.concatenate([np.zeros((N_USERS, pad)), block], axis=1)
    weekly = np.log1p(block[:, ::-1].reshape(N_USERS, FACTOR_WEEKS, 7).sum(axis=2))
    z = weekly - weekly.mean(axis=0)
    _, _, v = randomized_svd(z.astype(np.float32), n_components=FACTOR_RANK, random_state=0)
    load = z @ v.T.astype(np.float64)
    return load.sum(axis=1)


def deflated_direction(anchor):
    x = np.asarray(load_space_x(PRE, anchor), dtype=np.float64)
    d = np.log1p(x / np.maximum(x.mean(axis=0), 1e-9))
    return d.sum(axis=1)

In [ ]:
if not (SUBS / "submit_z_bank.csv").exists():
    cur22 = submission_vector("submit_v22_merge")
    b22 = b_of("v22_merge")
    z22 = z_score(cur22)

    ya = year_ago_lift(SUBMIT)
    d_year = z_score(orthogonalise(z_score(ya @ np.ones(ya.shape[1])), z22))
    d_btyd = z_score(orthogonalise(btyd_rate(SUBMIT), z22))
    d_fact = z_score(orthogonalise(weekly_factor_direction(SUBMIT), z22))
    d_defl = z_score(orthogonalise(deflated_direction(SUBMIT), z22))
    cap_twin = LEGACY_DIR / "twin_cap250.npy"
    if cap_twin.exists():
        base_c, cap_c = np.load(cap_twin).astype(np.float64)
        d_cap = z_score(orthogonalise(cap_c - base_c, z22))
    else:
        d_cap = z_score(
            orthogonalise(
                z_score(member_z(fleet_store, "ds_stable30", SUBMIT))
                - z_score(member_z(fleet_store, "ds_full", SUBMIT)),
                z22,
            )
        )
    for nm, d in (
        ("yearago", d_year),
        ("cap250", d_cap),
        ("btyd", d_btyd),
        ("factors", d_fact),
        ("deflate", d_defl),
    ):
        np.save(LEGACY_DIR / f"d_{nm}.npy", d.astype(np.float32))

    for stem, d, alpha in (
        ("submit_y_yearago", d_year, YEARAGO_ALPHA),
        ("submit_y_cap250", d_cap, CAP_ALPHA),
        ("submit_u_probe_btyd", d_btyd, STRUCT_ALPHA),
        ("submit_w_probe_factors", d_fact, STRUCT_ALPHA),
        ("submit_x_probe_deflate", d_defl, STRUCT_ALPHA),
    ):
        write_submission(
            stem,
            mix_raw(cur22, d, alpha),
            b22 / np.sqrt(1.0 + alpha**2),
            source=f"{stem.removeprefix('submit_')} probed at alpha {alpha}",
        )

    cov_year = cov_from_probe(
        SCORED["y_yearago"][1], b22 / np.sqrt(1 + YEARAGO_ALPHA**2), YEARAGO_ALPHA, b22
    )
    cov_cap = cov_from_probe(SCORED["y_cap250"][1], b22 / np.sqrt(1 + CAP_ALPHA**2), CAP_ALPHA, b22)
    cov_btyd = cov_from_probe(
        SCORED["u_probe_btyd"][1], b22 / np.sqrt(1 + STRUCT_ALPHA**2), STRUCT_ALPHA, b22
    )
    print(f"cap250 {cov_cap:+.5f}  yearago {cov_year:+.5f}  btyd {cov_btyd:+.5f}")

    zb = z22 + (cov_cap / b22) * d_cap + (cov_year / b22) * d_year
    b_bank = float(np.sqrt(b22**2 + cov_cap**2 + cov_year**2))
    write_submission(
        "submit_z_bank",
        zb,
        b_bank,
        source="v22_merge plus cap250 and yearago at their measured covariances",
    )
    np.save(LEGACY_DIR / "basis_zb.npy", z_score(zb).astype(np.float32))

    zbz = z_score(submission_vector("submit_z_bank"))
    d_btyd_b = z_score(orthogonalise(d_btyd, zbz))
    write_submission(
        "submit_v_btyd_opt",
        zbz + (cov_btyd / b_bank) * d_btyd_b,
        float(np.sqrt(b_bank**2 + cov_btyd**2)),
        source="z_bank plus the btyd direction at its measured covariance",
    )
    LEAN5 = ["seq_c", "cat_b", "v3_mlp", "lgbf_b", "seq_b"]
    w5 = fit_nnls_weights(pre_store, LEAN5, RECENT, ridge=0.02)
    d_t5 = z_score(orthogonalise(member_cols(pre_store, SUBMIT, LEAN5) @ w5, zbz))
    write_submission(
        "submit_t_blend5",
        mix_raw(zbz, d_t5, STRUCT_ALPHA),
        b_bank / np.sqrt(1 + STRUCT_ALPHA**2),
        source="z_bank plus a five member blend direction",
    )

## v29: the audit fixes, retrained

In [ ]:
if not (SUBS / "submit_v29all.csv").exists():
    for tag, names in (
        ("v29all", PRE_POOL),
        ("v29tree", group_by_prefix(PRE_POOL, TREE_PREFIX)),
        ("v29nn", group_by_prefix(PRE_POOL, NN_PREFIX)),
    ):
        w = fit_nnls_weights(pool_store, names, RECENT, ridge=0.0)
        oof = {a: member_cols(pool_store, a, names) @ w for a in VAL_ANCHORS}
        sub = z_score(member_cols(pool_store, SUBMIT, names) @ w)
        np.save(LEGACY_DIR / f"{tag}_sub.npy", sub.astype(np.float32))
        for a in VAL_ANCHORS:
            np.save(LEGACY_DIR / f"{tag}_oof_{a}.npy", z_score(oof[a]).astype(np.float32))
        print(f"{tag}: {len(names)} members")
        beta = projected_beta(oof, "v6_vall", "v6_vall")
        write_submission(
            f"submit_{tag}",
            sub,
            beta,
            source=f"{tag}_sub.npy, rebuilt features, projected optimal beta",
        )

## v30

In [ ]:
V30_FOLDS3 = ["2025-09-24", "2025-10-22", "2025-12-03"]
LEAN5 = ["seq_c", "cat_b", "v3_mlp", "lgbf_b", "seq_b"]
V30_CAL = 0.5

if not (SUBS / "submit_v30_bank2.csv").exists():
    run_fleet(list(V30_VIEWS), folds=[*V30_FOLDS3, SUBMIT], n_train=10, min_gap=28)

    def blend29_at(a):
        return z_score(np.load(LEGACY_DIR / f"v29all_oof_{a}.npy").astype(np.float64))

    def cov_of(vec_by_fold, folds):
        covs = []
        for a in folds:
            v, b, y = vec_by_fold[a], blend29_at(a), target_z(a)
            g = float(np.corrcoef(v, b)[0, 1])
            covs.append(
                SIGMA
                * (float(v @ y / y.size) - g * float(b @ y / y.size))
                / np.sqrt(max(1 - g * g, 1e-12))
            )
        return float(np.mean(covs))

    X5 = {a: np.stack([member_z(pool_store, m, a) for m in LEAN5]) for a in V30_FOLDS3}
    Y5 = {a: target_z(a) for a in V30_FOLDS3}
    G5, c5, tot = np.zeros((5, 5)), np.zeros(5), 0
    for a in V30_FOLDS3:
        G5 += X5[a] @ X5[a].T
        c5 += X5[a] @ Y5[a]
        tot += Y5[a].size
    w5 = nnls(G5 / tot + 0.02 * np.eye(5), c5 / tot)[0]
    w5 /= w5.sum()
    r_lean = float(np.mean([z_score(w5 @ X5[a]) @ Y5[a] / Y5[a].size for a in V30_FOLDS3]))
    r_ref = float(np.mean([blend29_at(a) @ Y5[a] / Y5[a].size for a in V30_FOLDS3]))
    b_lean = b_of("v29all") * r_lean / r_ref
    lean_sub = z_score(w5 @ np.stack([member_z(pool_store, m, SUBMIT) for m in LEAN5]))
    print(dict(zip(LEAN5, np.round(w5, 4), strict=True)))
    write_submission(
        "submit_v30_lean5", lean_sub, b_lean, source="five member blend on clean folds"
    )

    zbz = z_score(submission_vector("submit_z_bank"))
    bz = b_of("z_bank")

    def orth(v, *against):
        for a in against:
            v = v - (v @ a / a.size) * a
        return z_score(v)

    dq10 = [n for n in V30_VIEWS if n.startswith("dq_") and n.endswith("_10")]
    dq_by_fold = {
        a: z_score(np.mean([member_z(fleet_store, n, a) for n in dq10], axis=0)) for a in V30_FOLDS3
    }
    cov_dq = cov_of(dq_by_fold, V30_FOLDS3)
    d0 = orth(lean_sub, zbz)
    g0 = float(np.corrcoef(lean_sub, zbz)[0, 1])
    c0 = SIGMA * (b_lean / SIGMA - g0 * bz / SIGMA) / np.sqrt(max(1 - g0 * g0, 1e-12))
    dq_sub = z_score(np.mean([member_z(fleet_store, n, SUBMIT) for n in dq10], axis=0))
    d1 = orth(dq_sub, zbz, d0)
    g01 = float(np.corrcoef(orth(dq_sub, zbz), d0)[0, 1])
    c1 = (cov_dq - g01 * c0) / np.sqrt(max(1 - g01 * g01, 1e-12))
    print(f"cov lean5 {c0:+.5f}  cov dq10 marginal {c1:+.5f}")

    for stem, cal in (("submit_v30_bank", V30_CAL), ("submit_v30_bank2", 1.0)):
        cs = [cal * c0, cal * c1]
        b_tot = float(np.sqrt(bz**2 + sum(x * x for x in cs)))
        raw = zbz + (cs[0] / bz) * d0 + (cs[1] / bz) * d1
        write_submission(
            stem,
            raw,
            b_tot,
            source="z_bank plus the lean five direction and the ten percent subspace average"
            + (" at halved panel covariances" if cal != 1.0 else ""),
        )

## The sequence family

In [ ]:
SIDE_WINDOWS = (7, 30, 90, 365)
N_SIDE = 16
WIDE_WINDOWS = (7, 14, 30, 60, 90, 180, 365)
WIDE_CHANNELS = ("gmv", "ord", "cart", "srch", "active")
GRID_DIR = DATA / "seqgrid"
GRID_DIR.mkdir(parents=True, exist_ok=True)

if not (GRID_DIR / "meta.json").exists():
    data = pl.read_parquet(TRAIN_PATH).sort("user_id", "event_date")
    rows_g = np.searchsorted(USER_IDS, data["user_id"].to_numpy())
    day_g = (
        data
        .select((pl.col("event_date") - pl.lit(DATE_MIN)).dt.total_days().cast(pl.Int32))
        .to_series()
        .to_numpy()
    )
    flat = rows_g * N_DAYS + day_g
    for name, col in (("gmv", "gmv"), ("ord", "to_ord"), ("cart", "to_cart"), ("srch", "searches")):
        g = np.zeros(N_USERS * N_DAYS, dtype=np.float64)
        np.add.at(g, flat, data[col].to_numpy().astype(np.float64))
        np.save(GRID_DIR / f"{name}.npy", g.reshape(N_USERS, N_DAYS).astype(np.float32))
        del g
        gc.collect()
    act = np.zeros(N_USERS * N_DAYS, dtype=np.uint8)
    act[flat] = 1
    np.save(GRID_DIR / "active.npy", act.reshape(N_USERS, N_DAYS))
    (GRID_DIR / "meta.json").write_text(
        json.dumps({"n_users": N_USERS, "n_days": N_DAYS, "date_min": DATE_MIN.isoformat()})
    )
    del data, act, flat, rows_g, day_g
    gc.collect()


def load_cumulative():
    out = {}
    for name in ("gmv", "ord", "cart", "srch", "active"):
        a = np.asarray(np.load(GRID_DIR / f"{name}.npy", mmap_mode="r"), dtype=np.float64)
        c = np.zeros((a.shape[0], a.shape[1] + 1), dtype=np.float64)
        np.cumsum(a, axis=1, out=c[:, 1:])
        out[name] = c
        del a
        gc.collect()
    act = np.asarray(np.load(GRID_DIR / "active.npy", mmap_mode="r"))
    first = np.argmax(act > 0, axis=1).astype(np.int32)
    first[act.max(axis=1) == 0] = 10**6
    out["first"] = first
    day = np.arange(act.shape[1], dtype=np.int16)
    out["last_active"] = np.maximum.accumulate(np.where(act > 0, day, np.int16(-1)), axis=1).astype(
        np.int16
    )
    for key, src in (("last_order", "ord"), ("last_cart", "cart")):
        flagged = np.asarray(np.load(GRID_DIR / f"{src}.npy", mmap_mode="r")) > 0
        out[key] = np.maximum.accumulate(np.where(flagged, day, np.int16(-1)), axis=1).astype(
            np.int16
        )
        del flagged
    del act
    gc.collect()
    return out


def cwindow(cum, t, w):
    return cum[:, t + 1] - cum[:, max(0, t - w + 1)]


def cohort_at(cum, t):
    return (cwindow(cum["active"], t, HORIZON_DAYS) > 0) & (cum["first"] <= t - MIN_HISTORY_DAYS)


def target_at(cum, t):
    hi = min(cum["gmv"].shape[1] - 1, t + HORIZON_DAYS + 1)
    return cum["gmv"][:, hi] - cum["gmv"][:, t + 1]


def side_at(cum, t):
    cols = []
    for w in SIDE_WINDOWS:
        cols += [
            np.log1p(cwindow(cum["gmv"], t, w)),
            cwindow(cum["ord"], t, w),
            cwindow(cum["active"], t, w),
        ]
    for key in ("last_active", "last_order"):
        last = cum[key][:, t].astype(np.float64)
        cols.append(np.where(last < 0, 400.0, (t - last).clip(0, 400)))
    cols.append(np.maximum(t - cum["first"], 0).astype(np.float64))
    cols.append(np.log1p(cwindow(cum["gmv"], t, 30)) - np.log1p(cwindow(cum["gmv"], t, 90) / 3.0))
    return np.stack(cols, axis=1).astype(np.float32)


def wide_features(cum, t):
    cols = []
    for ch in WIDE_CHANNELS:
        for w in WIDE_WINDOWS:
            v = cwindow(cum[ch], t, w)
            cols.append(np.log1p(v) if ch != "active" else v)
    for w in (30, 90, 365):
        g, o, c, s, a = (cwindow(cum[ch], t, w) for ch in WIDE_CHANNELS)
        cols += [
            o / np.maximum(a, 1.0),
            np.log1p(g / np.maximum(o, 1.0)),
            o / np.maximum(c, 1.0),
            o / np.maximum(s, 1.0),
            c / np.maximum(s, 1.0),
        ]
    for name in ("last_active", "last_order", "last_cart"):
        last = cum[name][:, t].astype(np.float64)
        cols.append(np.where(last < 0, 400.0, (t - last).clip(0, 400)))
    cols.append(np.maximum(t - cum["first"], 0).astype(np.float64).clip(0, 400))
    for ch in ("gmv", "ord", "active"):
        a30, a90, a365 = (cwindow(cum[ch], t, w) for w in (30, 90, 365))
        cols += [
            np.log1p(a30) - np.log1p(a90 / 3.0),
            np.log1p(a90) - np.log1p(a365 / 4.0),
        ]
    cols.append(cwindow(cum["gmv"], t, 30) / np.maximum(cwindow(cum["gmv"], t, 365), 1.0))
    return np.stack(cols, axis=1).astype(np.float32)

In [ ]:
SEQAUG_ARCH = {
    "patch": 7,
    "n_patch": 52,
    "dim": 256,
    "layers": 6,
    "heads": 8,
    "ff": 3,
    "side_hidden": 256,
    "head_hidden": 512,
    "final_norm": True,
}
MR = {**SEQAUG_ARCH, "n_patch": 54, "levels": [(14, 26), (1, 28)]}
MRS = {**SEQAUG_ARCH, "n_patch": 54, "levels": [(7, 26), (1, 28)]}
P4 = {**SEQAUG_ARCH, "patch": 4, "n_patch": 91}
P14 = {**SEQAUG_ARCH, "patch": 14, "n_patch": 26}
P28 = {**SEQAUG_ARCH, "patch": 28, "n_patch": 13}
SEQAUG_DIR = DATA / "seqaug"
SEQAUG_DIR.mkdir(parents=True, exist_ok=True)


def patches_multi(daily, cal, users, t, arch, n_ch):
    levels = arch.get("levels")
    if not levels:
        return patch_window(daily, cal, users, t, arch["patch"], arch["n_patch"], n_ch, True, True)
    blocks = [patch_window(daily, cal, users, t, p, k, n_ch, True, True) for p, k in levels]
    return torch.cat(blocks, dim=1)


def train_seq_aug(
    cum,
    daily,
    cal,
    val_anchor,
    *,
    days_per_epoch=48,
    epochs=3,
    batch=2048,
    lr=1e-3,
    seed=42,
    arch=None,
    predict_all=False,
    wide=False,
):
    arch = arch or SEQAUG_ARCH
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    n_ch = daily.shape[2]
    t_val = (date.fromisoformat(val_anchor) - DATE_MIN).days
    pool = list(range(MIN_HISTORY_DAYS, t_val - HORIZON_DAYS + 1))
    rng = np.random.default_rng(seed)
    probe = patches_multi(daily, cal, torch.arange(4, device="cuda"), pool[-1], arch, n_ch)
    n_side = len(WIDE_WINDOWS) * len(WIDE_CHANNELS) + 26 if wide else N_SIDE
    model = SeqNet(arch, probe.shape[2], n_side).to("cuda")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    schedule = [
        rng.choice(pool, size=min(days_per_epoch, len(pool)), replace=False) for _ in range(epochs)
    ]
    steps = sum(
        int(np.ceil(cohort_at(cum, int(t)).sum() / batch)) for days in schedule for t in days
    )
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(steps, 1))
    mse, bce = nn.MSELoss(), nn.BCEWithLogitsLoss()
    for days in schedule:
        model.train()
        order, packs = [], {}
        for raw_day in days:
            t = int(raw_day)
            coh = cohort_at(cum, t)
            users = np.flatnonzero(coh)
            y = np.log1p(target_at(cum, t)[users]).astype(np.float32)
            mu = float(y.mean())
            side = (wide_features(cum, t) if wide else side_at(cum, t))[users]
            packs[t] = (
                torch.from_numpy(users.astype(np.int64)).to("cuda"),
                torch.from_numpy(y - mu).to("cuda"),
                torch.from_numpy((y > 0).astype(np.float32)).to("cuda"),
                torch.from_numpy(side).to("cuda"),
            )
            perm = torch.randperm(len(users), device="cuda")
            order += [(t, perm[k : k + batch]) for k in range(0, len(users), batch)]
        rng.shuffle(order)
        for t, sel in order:
            users, y, pos, side = packs[t]
            seq = patches_multi(daily, cal, users[sel], t, arch, n_ch)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                a, b = model(seq, side[sel].float())
                loss = mse(a.float(), y[sel]) + 0.3 * bce(b.float(), pos[sel])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        del packs, order
        gc.collect()
        torch.cuda.empty_cache()
    model.eval()
    keep = np.ones(N_USERS, dtype=bool) if predict_all else cohort_at(cum, t_val)
    idx = np.flatnonzero(keep)
    side_va = torch.from_numpy(
        (wide_features(cum, t_val) if wide else side_at(cum, t_val))[idx]
    ).to("cuda")
    out = np.zeros(N_USERS, dtype=np.float32)
    with torch.no_grad():
        for k in range(0, idx.size, 8192):
            sel = idx[k : k + 8192]
            seq = patches_multi(
                daily, cal, torch.from_numpy(sel.astype(np.int64)).to("cuda"), t_val, arch, n_ch
            )
            with torch.autocast("cuda", dtype=torch.bfloat16):
                a, _ = model(seq, side_va[k : k + 8192].float())
            out[sel] = a.float().cpu().numpy()
    del model, side_va
    torch.cuda.empty_cache()
    gc.collect()
    return out


SEQAUG_ARMS = {
    "sa_aug48": ({"days_per_epoch": 48, "epochs": 3}, SEQAUG_ARCH),
    "sa_aug150": ({"days_per_epoch": 150, "epochs": 3}, SEQAUG_ARCH),
    "sw_aug48": ({"days_per_epoch": 48, "epochs": 3, "wide": True}, SEQAUG_ARCH),
    "sa_p14": ({"days_per_epoch": 48, "epochs": 3}, P14),
    "f_mr": ({"days_per_epoch": 48, "epochs": 3, "wide": True}, MR),
    "f_p14": ({"days_per_epoch": 48, "epochs": 3, "wide": True}, P14),
    "f_p4": ({"days_per_epoch": 48, "epochs": 3, "wide": True}, P4),
    "f_mrs2": ({"days_per_epoch": 48, "epochs": 3}, MRS),
    "f_p28": ({"days_per_epoch": 48, "epochs": 3}, P28),
}
SEQAUG_FOLDS = ["2025-10-22", "2025-12-03", SUBMIT]
SEQAUG_SEEDS = [42, 7]

pending_seq = [
    (n, va)
    for n in SEQAUG_ARMS
    for va in SEQAUG_FOLDS
    if not (SEQAUG_DIR / f"{n}_{va}.npy").exists()
]
if pending_seq:
    CUM = load_cumulative()
    daily_sa = torch.from_numpy(np.load(SEQ / "daily10_f16.npy")).to("cuda")
    cal_sa = torch.from_numpy(np.load(CAL_PATH)).to("cuda")
    for name, va in pending_seq:
        kw, arch = SEQAUG_ARMS[name]
        t0 = time.time()
        preds = [
            train_seq_aug(
                CUM, daily_sa, cal_sa, va, seed=s, arch=arch, predict_all=va == SUBMIT, **kw
            )
            for s in SEQAUG_SEEDS
        ]
        np.save(
            SEQAUG_DIR / f"{name}_{va}.npy",
            np.mean(np.asarray(preds, dtype=np.float64), axis=0).astype(np.float32),
        )
        print(f"  seqaug {name} {va} {time.time() - t0}s", flush=True)
    del daily_sa, cal_sa, CUM
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
SEQ_FOLD = "2025-10-22"
SEQ_FOLDS2 = ["2025-10-22", "2025-12-03"]
V34_CAL_REF = 0.034716
V35_ALPHA = 0.010
V37_ALPHA_Q = -0.014


def seqaug_at(name, a):
    v = np.load(SEQAUG_DIR / f"{name}_{a}.npy").astype(np.float64)
    m = cohort_mask(a)
    return z_score(v[m] if v.shape[0] == N_USERS and a != SUBMIT else v)


def blend29_fold(a):
    return z_score(np.load(LEGACY_DIR / f"v29all_oof_{a}.npy").astype(np.float64))


def panel_cov_seq(name, folds):
    covs = []
    for a in folds:
        v, b, y = seqaug_at(name, a), blend29_fold(a), target_z(a)
        g = float(np.corrcoef(v, b)[0, 1])
        covs.append(
            SIGMA
            * (float(v @ y / y.size) - g * float(b @ y / y.size))
            / np.sqrt(max(1 - g * g, 1e-12))
        )
    return float(np.mean(covs))


def orth(v, *bs):
    for b in bs:
        v = v - (v @ b / b.size) * b
    return z_score(v)


if not (SUBS / "submit_v37_curve.csv").exists():
    zbz = z_score(submission_vector("submit_z_bank"))
    bz = b_of("z_bank")

    d_seqaug = orth(seqaug_at("sa_aug48", SUBMIT), zbz)
    np.save(LEGACY_DIR / "d_seqaug.npy", d_seqaug.astype(np.float32))
    for alpha in (0.010, 0.020):
        write_submission(
            f"submit_v31_seq{int(alpha * 1000):03d}",
            mix_raw(zbz, d_seqaug, alpha),
            bz / np.sqrt(1.0 + alpha**2),
            source=f"sa_aug48 direction probed at alpha {alpha}",
        )
    cov_seq = cov_from_probe(SCORED["v31_seq010"][1], bz / np.sqrt(1 + 0.010**2), 0.010, bz)
    print(f"sa_aug48 measured Cov {cov_seq:+.5f}")
    b32 = float(np.sqrt(bz**2 + cov_seq**2))
    write_submission(
        "submit_v32_seqopt",
        zbz + (cov_seq / bz) * d_seqaug,
        b32,
        source="z_bank plus the augmented sequence direction at its measured covariance",
    )

    ratio = cov_seq / panel_cov_seq("sa_aug48", SEQ_FOLDS2)
    print(f"panel to board transfer for this family: {ratio}")

    d_sw = orth(seqaug_at("sw_aug48", SUBMIT), zbz)
    c_sw = ratio * panel_cov_seq("sw_aug48", SEQ_FOLDS2)
    d_p14 = orth(seqaug_at("sa_p14", SUBMIT), zbz)
    g12 = float(np.corrcoef(d_p14, d_sw)[0, 1])
    d_ta = orth(d_p14, d_sw)
    c_ta = ratio * panel_cov_seq("sa_p14", SEQ_FOLDS2)
    c_ta_marg = (c_ta - g12 * c_sw) / np.sqrt(max(1 - g12 * g12, 1e-12))
    np.save(LEGACY_DIR / "d_sw.npy", d_sw.astype(np.float32))
    np.save(LEGACY_DIR / "d_ta_orth.npy", d_ta.astype(np.float32))
    for label, cs, dirs, stem in (
        ("sw only", [c_sw], [d_sw], "submit_v34_sw"),
        ("sw + ta", [c_sw, c_ta_marg], [d_sw, d_ta], "submit_v34_swta"),
    ):
        b_tot = float(np.sqrt(bz**2 + sum(c * c for c in cs)))
        raw = zbz + sum((c / bz) * d for c, d in zip(cs, dirs, strict=True))
        write_submission(stem, raw, b_tot, source=f"z_bank plus {label}")

    m150 = z_score(seqaug_at("sa_aug150", SUBMIT))
    d150 = orth(m150, zbz, d_seqaug)
    np.save(LEGACY_DIR / "d_aug150.npy", d150.astype(np.float32))
    b_cur35 = b_of("v32_seqopt")
    write_submission(
        "submit_v35_aug150",
        mix_raw(z_score(submission_vector("submit_v32_seqopt")), d150, V35_ALPHA),
        b_cur35 / np.sqrt(1.0 + V35_ALPHA**2),
        source="sa_aug150 marginal after the banked sequence direction",
    )
    cov150 = cov_from_probe(
        SCORED["v35_aug150"][1], b_cur35 / np.sqrt(1 + V35_ALPHA**2), V35_ALPHA, b_cur35
    )
    print(f"sa_aug150 marginal Cov {cov150:+.5f}")

    FLEET_ARMS = ["f_mr", "f_p14", "f_p4", "f_mrs2", "f_p28"]
    ens = z_score(np.mean([seqaug_at(n, SUBMIT) for n in FLEET_ARMS], axis=0))
    d_fleet_seq = orth(ens, zbz, d_seqaug, d150)
    np.save(LEGACY_DIR / "d_submit_v36_fleet.npy", d_fleet_seq.astype(np.float32))
    b_cur36 = b_of("v35_aug150")
    write_submission(
        "submit_v36_fleet",
        mix_raw(z_score(submission_vector("submit_v35_aug150")), d_fleet_seq, V36_PROBE["alpha"]),
        b_cur36 / np.sqrt(1.0 + V36_PROBE["alpha"] ** 2),
        source=f"fleet residual: {FLEET_ARMS}",
    )

    cur37 = z_score(submission_vector("submit_v35_aug150"))
    a_f = V36_PROBE["cov"] / b_cur36
    base37 = z_score(cur37 + a_f * d_fleet_seq)
    b_base = float(np.sqrt(b_cur36**2 + V36_PROBE["cov"] ** 2))
    q = z_score(orthogonalise(base37 * base37 - (base37 * base37).mean(), base37))
    np.save(LEGACY_DIR / "d_curve.npy", q.astype(np.float32))
    write_submission(
        "submit_v37_curve",
        base37 + V37_ALPHA_Q * q,
        b_base / float(np.sqrt(1 + V37_ALPHA_Q**2)),
        source=f"v36 fleet banked at {a_f} plus quadratic shape probed at {V37_ALPHA_Q}",
    )

### v33

In [ ]:
V33_RECIPE = {"v32_seqopt": 1.374958, "v11_spaced": -0.224408, "v29all": -0.150551}

if not (SUBS / "submit_v33_derived.csv").exists():
    names33 = list(V33_RECIPE)
    Z33, rho33 = [], []
    for n in names33:
        lg = submission_vector(SCORED[n][0])
        level, beta = float(lg.mean()), float(lg.std())
        Z33.append((lg - level) / beta)
        rho33.append(
            (beta * beta + W_TOTAL + (level - TARGET_MEAN) ** 2 - SCORED[n][1] ** 2)
            / (2.0 * beta * SIGMA)
        )
    Z33 = np.vstack(Z33)
    rho33 = np.asarray(rho33)
    C33 = (Z33 @ Z33.T) / Z33.shape[1]
    w33 = np.array([V33_RECIPE[n] for n in names33])
    r2 = float(w33 @ rho33) ** 2 / float(w33 @ C33 @ w33)
    proj33 = SIGMA * np.sqrt(max(1.0 - r2, 0.0))
    print(f"v33 weights sum {w33.sum()}  leverage {np.abs(w33).sum()}")
    print(f"reproduced projection {proj33} recorded 1.6469639")
    write_submission(
        "submit_v33_derived",
        w33 @ Z33,
        float(np.sqrt(1.0 - (proj33 / SIGMA) ** 2) * SIGMA),
        source="three vector solve recorded on origin",
    )

## The archive

In [ ]:
ARCHIVE_ORDER_60 = [
    "catboost_v1",
    "catboost_v2",
    "cb_v2_dropz",
    "chronos2",
    "chronosL",
    "v3_persist",
    "v3_seasonal",
    "v4_blend_fit",
    "v4_blend_flat",
    "v4_mix_fit",
    "v4_mix_flat",
    "v5_beta",
    "v6_vall",
    "v6_vtree",
    "v6_vnn",
    "v7_all",
    "v8_probeopt",
    "v9_opt4",
    "v10_resid",
    "p1_resid",
    "p2_spaced",
    "v11_spaced",
    "p3_spaced9",
    "v12_harvest",
    "v13_fleet",
    "p_shift",
    "p_shift20",
    "p_adv",
    "q_mask_stable05",
    "q_mask_stable10",
    "q_span_pall",
    "p_shift20_zeros_1",
    "v21_banked",
    "r_pool_s20",
    "r_blend_shift20_mask10",
    "v22_merge",
    "t_blend5",
    "u_probe_btyd",
    "v_btyd_opt",
    "w_probe_factors",
    "x_probe_deflate",
    "y_cap250",
    "y_yearago",
    "z_bank",
    "v29all",
    "v29tree",
    "v29nn",
    "v30_lean5",
    "v30_bank",
    "v30_bank2",
    "v31_seq010",
    "v32_seqopt",
    "v34_sw",
    "v34_swta",
    "v35_aug150",
    "v36_fleet",
    "v33_derived",
    "v37_curve",
    "v41_ridge",
    "v42_pass2",
]
ARCHIVE_POST_43 = ["v43_calib", "v44_seqmix", "v45_combined"]


def v36_recovered_score():
    a, b0 = V36_PROBE["alpha"], V36_PROBE["b_current"]
    b_obs = (V36_PROBE["cov"] * a + b0) / np.sqrt(1.0 + a * a)
    lg = submission_vector("submit_v36_fleet")
    level, beta = float(lg.mean()), float(lg.std())
    return float(np.sqrt(W_TOTAL + beta * beta + (level - TARGET_MEAN) ** 2 - 2.0 * beta * b_obs))


def build_archive(names):
    items = {}
    for n in names:
        stem = SCORED[n][0]
        lb_n = v36_recovered_score() if n == "v36_fleet" else SCORED[n][1]
        items[n] = (stem, lb_n)
    out_names, Z, rho, beta, level, lbs = [], [], [], [], [], []
    ids = None
    for n, (stem, lb_n) in sorted(items.items(), key=lambda kv: kv[1][1]):
        path = SUBS / f"{stem}.csv"
        if not path.exists():
            raise FileNotFoundError(f"archive vector {n} missing: {path}")
        d = pl.read_csv(path).sort("user_id")
        uid = d["user_id"].to_numpy()
        ids = uid if ids is None else ids
        lg = np.log1p(np.clip(d["predict"].to_numpy().astype(np.float64), 0.0, None))
        lv, bt = float(lg.mean()), float(lg.std())
        r = (bt * bt + W_TOTAL + (lv - TARGET_MEAN) ** 2 - lb_n * lb_n) / (2.0 * bt * SIGMA)
        out_names.append(n)
        Z.append(((lg - lv) / bt).astype(np.float32))
        rho.append(r)
        beta.append(bt)
        level.append(lv)
        lbs.append(lb_n)
    a = {
        "names": out_names,
        "Z": np.vstack(Z).astype(np.float64),
        "rho": np.asarray(rho),
        "beta": np.asarray(beta),
        "level": np.asarray(level),
        "lb": np.asarray(lbs),
        "ids": ids,
    }
    a["C"] = (a["Z"] @ a["Z"].T) / a["Z"].shape[1]
    a["idx"] = {n: i for i, n in enumerate(out_names)}
    return a


def archive_table(a):
    for i, nm in enumerate(a["names"]):
        print(f"{nm}{a['lb'][i]}{a['rho'][i]}{a['beta'][i]}{a['level'][i]}")

## The archive inventory

In [ ]:
WRITTEN_BELOW = ["v41_ridge", "v42_pass2", "v43_calib", "v44_seqmix", "v45_combined"]
ARCHIVE_58 = [n for n in ARCHIVE_ORDER_60 if n not in WRITTEN_BELOW]

rebuilt, frozen, missing = [], [], []
for name in ARCHIVE_58:
    stem = SCORED[name][0]
    if (SUBS / f"{stem}.csv").exists():
        rebuilt.append(name)
        continue
    ref = REFERENCE / f"{stem}.csv"
    if ref.exists():
        shutil.copyfile(ref, SUBS / f"{stem}.csv")
        frozen.append(name)
    else:
        missing.append(name)

print(f"{len(rebuilt)} rebuilt from the parquet")
print(f"{len(frozen)} taken from archive/reference/ as frozen inputs")
print(f"{len(WRITTEN_BELOW)} written below: " + ", ".join(WRITTEN_BELOW))
if missing:
    print("\nmissing, neither rebuilt nor present under archive/reference/:")
    for n in missing:
        print(f"  {n}{SCORED[n][0]}")
    raise SystemExit(
        f"{len(missing)} archive vectors are missing. v43 is a ridge over all sixty, so "
        "solving without them would not be v43."
    )

### v41 and v42, and the miss curve they measured

In [ ]:
def ridge_solve(a, ridge):
    k = len(a["names"])
    w = np.linalg.solve(a["C"] + ridge * np.eye(k), a["rho"])
    w = w / w.sum()
    num, var = float(w @ a["rho"]), float(w @ a["C"] @ w)
    r2 = num * num / var
    proj = SIGMA * np.sqrt(max(1.0 - r2, 0.0))
    lev = float(np.abs(w) @ np.sqrt(np.diag(a["C"])) / np.sqrt(var))
    return w, proj, lev, r2


def write_from_z(name, z, beta, ids, source=""):
    z = z_score(z)
    level = solve_level(z, beta)
    pred = np.expm1(np.clip(beta * z + level, 0.0, None))
    path = SUBS / f"{name}.csv"
    pl.DataFrame({"user_id": ids, "predict": pred.astype(np.float32)}).write_csv(path)
    (SUBS / f"meta_{name}.json").write_text(
        json.dumps({"beta": beta, "level": level, "source": source, "file": path.name}, indent=1)
    )
    print(f"{name} beta={beta} level={level} zeros={(pred == 0).mean()}")
    return pred


ARCHIVE_58 = [n for n in ARCHIVE_ORDER_60 if n not in ("v41_ridge", "v42_pass2")]
V41_RIDGE, V42_RIDGE = 2.5e-4, 2e-7
V42_SIM_GAIN = 0.00076

if not (SUBS / "submit_v41_ridge.csv").exists():
    a58 = build_archive(ARCHIVE_58)
    archive_table(a58)
    w41, proj41, lev41, _ = ridge_solve(a58, V41_RIDGE)
    beta41 = float(np.sqrt(W_TOTAL - proj41 * proj41))
    print(f"\nv41 ridge {V41_RIDGE}  leverage {lev41}  projection {proj41}")
    write_from_z(
        "submit_v41_ridge", w41 @ a58["Z"], beta41, a58["ids"], source="archive ridge solve"
    )

if not (SUBS / "submit_v42_pass2.csv").exists():
    a59 = build_archive([*ARCHIVE_58, "v41_ridge"])
    w42, proj42, lev42, _ = ridge_solve(a59, V42_RIDGE)
    expected42 = SCORED["v41_ridge"][1] - V42_SIM_GAIN
    beta42 = float(np.sqrt(W_TOTAL - expected42 * expected42))
    print(f"v42 ridge {V42_RIDGE}  leverage {lev42}  analytic projection {proj42}")
    write_from_z("submit_v42_pass2", w42 @ a59["Z"], beta42, a59["ids"], source="archive pass two")

### v43

In [ ]:
V43_RIDGE = 1.2e-4
MISS_MEASURED = [(1.7499, 5.7e-6), (4.9550, 2.751e-5), (405.2, 6.6151e-3)]
MISS_RIDGES = [1.0e-4, 2.5e-4, 2.0e-7]

lev_m = np.array([m[0] for m in MISS_MEASURED])
miss_m = np.array([m[1] for m in MISS_MEASURED])
P_LEV, C_LEV = np.polyfit(np.log(lev_m), np.log(miss_m), 1)
lam_m = np.array(MISS_RIDGES)
P_LAM, C_LAM = np.polyfit(np.log(lam_m[1:]), np.log(miss_m[1:]), 1)


def miss_lev(lv):
    return float(np.exp(C_LEV) * lv**P_LEV)


def miss_lam(lam):
    return float(np.exp(C_LAM) * lam**P_LAM)


A60 = build_archive(ARCHIVE_ORDER_60)
print(f"archive {len(A60['names'])} vectors, best on the board {SCORED['v41_ridge'][1]}")
print(f"miss vs leverage: {np.exp(C_LEV)} * lev^{P_LEV}")

rows = []

for lam in np.geomspace(5e-4, 2e-5, 12):
    w, proj, lev, r2 = ridge_solve(A60, lam)
    if not (0.0 <= r2 < 1.0):
        continue
    m1, m2 = miss_lev(lev), miss_lam(lam)
    worst = max(m1, m2)
    exp = proj + worst
    rows.append((lam, lev, proj, worst, exp, w))

lam43, lev43, proj43, worst43, exp43, w43 = min(rows, key=lambda r: abs(r[0] - V43_RIDGE))
beta43 = float(np.sqrt(W_TOTAL - exp43 * exp43))
print(f"\nchosen ridge {lam43}: leverage {lev43}, projection {proj43}")
print(f"  worse of the two miss estimates {worst43} -> expected {exp43}")
print(f"  beta {beta43}")

pred43 = write_from_z(
    "submit_v43_calib", w43 @ A60["Z"], beta43, A60["ids"], source="archive ridge solve, calibrated"
)
o = np.argsort(-np.abs(w43))
print(f"\n{'vector'}{'weight'}")
for i in o[:8]:
    print(f"{A60['names'][i]}{w43[i]}")

### v44 and v45

In [ ]:
V44_C_NEW_FOLD, V44_TRANSFER = 0.02258, 0.770
V45_RIDGE = 1.4e-4
SEQ_MIX_GROUPS = {
    "p7w": ["sw_aug48"],
    "p14": ["f_p14"],
    "mr": ["f_mr"],
    "p28": ["f_p28"],
}
BANKED_COVS = {
    "d_seqaug": 0.034716,
    "d_aug150": 0.012531,
    "d_submit_v36_fleet": 0.009695,
    "d_curve": -0.014204,
}

if not (SUBS / "submit_v45_combined.csv").exists():
    zb43 = z_score(submission_vector("submit_v43_calib"))
    b43 = float(np.sqrt(W_TOTAL - SCORED["v43_calib"][1] ** 2))
    parts = []
    for name, arms in SEQ_MIX_GROUPS.items():
        have = [a for a in arms if (SEQAUG_DIR / f"{a}_{SUBMIT}.npy").exists()]
        if not have:
            print(f"  {name}: nothing")
            continue
        parts.append(
            z_score(
                np.mean(
                    [
                        z_score(np.load(SEQAUG_DIR / f"{a}_{SUBMIT}.npy").astype(np.float64))
                        for a in have
                    ],
                    axis=0,
                )
            )
        )
        print(f"  {name}: {len(have)} runs")
    mix = z_score(np.mean(parts, axis=0))
    g = float(np.corrcoef(mix, zb43)[0, 1])
    d_mix = z_score(mix - g * zb43)
    print(f"mix correlates {g} with v43; orthogonal share {np.sqrt(1 - g * g)}")

    names_b, B_b, cov_b = [], [], []
    for nm, c in BANKED_COVS.items():
        p = LEGACY_DIR / f"{nm}.npy"
        if not p.exists():
            continue
        v = z_score(np.load(p).astype(np.float64))
        gg = float(v @ zb43 / zb43.size)
        v = v - gg * zb43
        c_adj = (c - gg * b43) / np.sqrt(max(1.0 - gg * gg, 1e-12))
        coef = []
        for prev in B_b:
            al = float(v @ prev / prev.size)
            v = v - al * prev
            coef.append(al)
        n_b = float(np.sqrt(v @ v / v.size))
        if n_b < 1e-6:
            continue
        names_b.append(nm)
        B_b.append(v / n_b)
        cov_b.append((c_adj - sum(al * pc for al, pc in zip(coef, cov_b, strict=True))) / n_b)

    r = d_mix.copy()
    accounted = 0.0
    print(f"\n{'d decomposes as'}{'loading'}{'Cov inherited'}")
    for nm, v, c in zip(names_b, B_b, cov_b, strict=True):
        al = float(r @ v / v.size)
        r = r - al * v
        accounted += al * c
        print(f"{nm}{al}{al * c}")
    share = float(np.sqrt(r @ r / r.size))
    c_new = V44_C_NEW_FOLD * V44_TRANSFER
    expected44 = accounted + share * c_new
    print(f"{'genuinely new'}{share}{share * c_new}")
    print(f"\nexpected marginal Cov {expected44}")

    alpha44 = float(np.clip(expected44 / b43, 0.004, 0.020))
    beta44 = b43 / np.sqrt(1.0 + alpha44 * alpha44)
    write_from_z(
        "submit_v44_seqmix",
        zb43 + alpha44 * d_mix,
        beta44,
        A60["ids"],
        source="v43 plus the seed-averaged resolution mix",
    )
    np.save(LEGACY_DIR / "d_seqmix.npy", d_mix.astype(np.float32))

    w45, proj45, lev45, _ = ridge_solve(A60, V45_RIDGE)
    worst45 = max(miss_lev(lev45), miss_lam(V45_RIDGE))
    exp45 = proj45 + worst45
    z_arch = z_score(w45 @ A60["Z"])
    g45 = float(np.corrcoef(d_mix, z_arch)[0, 1])
    d45 = z_score(d_mix - g45 * z_arch)
    b45 = float(np.sqrt(W_TOTAL - exp45 * exp45))
    alpha45 = float(np.clip(expected44 / b45, 0.004, 0.020))
    print(f"v45 ridge {V45_RIDGE} leverage {lev45} projection {proj45}")
    write_from_z(
        "submit_v45_combined",
        z_arch + alpha45 * d45,
        b45 / np.sqrt(1.0 + alpha45 * alpha45),
        A60["ids"],
        source="archive increment at ridge 1.4e-4 plus the sequence mix",
    )

### The unsent combiners

In [ ]:
import optuna
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

NL_STACK_DIR = MODELS / "nl_stack"
NL_STACK_DIR.mkdir(parents=True, exist_ok=True)
INNER_FIT, INNER_EVAL = FIT_ANCHORS[:3], FIT_ANCHORS[3:]
NL_SEED = 42
NL_TRIALS = {"lgbm": 24, "xgb": 16, "mlp": 16, "rf": 4, "et": 8, "catboost": 8}
optuna.logging.set_verbosity(optuna.logging.WARNING)


def nl_rows(anchor_list):
    return (
        np.vstack([CACHE_M[a] for a in anchor_list]),
        np.concatenate([CACHE_Y[a] for a in anchor_list]),
    )


Xi, yi = nl_rows(INNER_FIT)
Xv, yv = nl_rows(INNER_EVAL)


def nl_rho(p, y):
    return float(np.corrcoef(p, y)[0, 1])


def nl_ridge_w(X, y):
    A = X.T @ X / len(X)
    b = X.T @ y / len(X)
    return np.linalg.solve(A + 1e-2 * np.eye(len(b)) * np.trace(A) / len(b), b)


def nl_mlp_fit(X, y, shape=2, epochs=12, lr=1e-3, wd=1e-4, dropout=0.1, batch=8192):
    hidden = {1: (128,), 2: (256, 128), 3: (512, 256, 128), 4: (64, 64)}[shape]
    torch.manual_seed(NL_SEED)
    layers, d = [], X.shape[1]
    for h in hidden:
        layers += [nn.Linear(d, h), nn.SiLU(), nn.Dropout(dropout)]
        d = h
    net = nn.Sequential(*layers, nn.Linear(d, 1)).to("cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    xt = torch.as_tensor(X, dtype=torch.float32, device="cuda")
    yt = torch.as_tensor(y, dtype=torch.float32, device="cuda").unsqueeze(1)
    n = xt.shape[0]
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=epochs * ((n + batch - 1) // batch)
    )
    for _ in range(epochs):
        perm = torch.randperm(n, device="cuda")
        net.train()
        for s in range(0, n, batch):
            idx = perm[s : s + batch]
            opt.zero_grad(set_to_none=True)
            nn.functional.mse_loss(net(xt[idx]), yt[idx]).backward()
            opt.step()
            sched.step()
    net.eval()

    def predict(Z):
        zt = torch.as_tensor(Z, dtype=torch.float32, device="cuda")
        with torch.no_grad():
            out = [net(zt[s : s + 65536]) for s in range(0, zt.shape[0], 65536)]
        return torch.cat(out).cpu().numpy().ravel()

    return predict


def nl_make_fn(name, params):
    p = dict(params)
    if name == "lgbm":
        r = p.pop("rounds")
        p["bagging_freq"] = 1

        def fit_lgbm(X, y):
            m = lgb.train(
                {
                    "objective": "regression",
                    "verbose": -1,
                    "seed": NL_SEED,
                    "num_threads": 16,
                    "force_row_wise": True,
                    **p,
                },
                lgb.Dataset(X, label=y),
                num_boost_round=r,
            )
            return m.predict

        return fit_lgbm
    if name == "xgb":
        r = p.pop("rounds")

        def fit_xgb(X, y):
            m = xgb.train(
                {
                    "objective": "reg:squarederror",
                    "tree_method": "hist",
                    "device": "cuda",
                    "seed": NL_SEED,
                    **p,
                },
                xgb.DMatrix(X, label=y),
                num_boost_round=r,
            )
            return lambda Z: m.predict(xgb.DMatrix(Z))

        return fit_xgb
    if name == "catboost":
        r = p.pop("rounds")

        def fit_cb(X, y):
            m = CatBoostRegressor(iterations=r, random_seed=NL_SEED, verbose=0, **p)
            m.fit(X.astype(np.float32), y)
            return lambda Z: m.predict(Z.astype(np.float32))

        return fit_cb
    if name in ("rf", "et"):
        cls = RandomForestRegressor if name == "rf" else ExtraTreesRegressor

        def fit_sk(X, y):
            m = cls(random_state=NL_SEED, n_jobs=16, **p)
            m.fit(X.astype(np.float32), y)
            return lambda Z: m.predict(Z.astype(np.float32))

        return fit_sk

    def fit_mlp(X, y):
        return nl_mlp_fit(X, y, **p)

    return fit_mlp


def nl_fit_predictor(fn, X, y, resid):
    if not resid:
        return fn(X, y)
    w, _ = nnls(X, y)
    lin = X @ w
    f = fn(np.column_stack([X, lin]), y - lin)

    def predict(Z):
        lz = Z @ w
        return lz + f(np.column_stack([Z, lz]))

    return predict


NL_SPACE = {
    "lgbm": lambda t: {
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": t.suggest_int("num_leaves", 7, 127, log=True),
        "min_data_in_leaf": t.suggest_int("min_data_in_leaf", 200, 20000, log=True),
        "feature_fraction": t.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": t.suggest_float("bagging_fraction", 0.5, 1.0),
        "lambda_l2": t.suggest_float("lambda_l2", 1e-2, 1e3, log=True),
        "rounds": t.suggest_int("rounds", 100, 1200, log=True),
    },
    "xgb": lambda t: {
        "eta": t.suggest_float("eta", 0.01, 0.2, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 10),
        "min_child_weight": t.suggest_float("min_child_weight", 1.0, 5e3, log=True),
        "subsample": t.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.4, 1.0),
        "lambda": t.suggest_float("lambda", 1e-2, 1e3, log=True),
        "rounds": t.suggest_int("rounds", 100, 1200, log=True),
    },
    "mlp": lambda t: {
        "shape": t.suggest_int("shape", 1, 4),
        "epochs": t.suggest_int("epochs", 4, 20),
        "lr": t.suggest_float("lr", 3e-4, 1e-2, log=True),
        "wd": t.suggest_float("wd", 1e-6, 1e-1, log=True),
        "dropout": t.suggest_float("dropout", 0.0, 0.4),
    },
    "rf": lambda t: {
        "n_estimators": 120,
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 2000, 40000, log=True),
        "max_features": t.suggest_float("max_features", 0.3, 0.8),
    },
    "et": lambda t: {
        "n_estimators": 200,
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 200, 20000, log=True),
        "max_features": t.suggest_float("max_features", 0.3, 1.0),
    },
    "catboost": lambda t: {
        "depth": t.suggest_int("depth", 3, 8),
        "learning_rate": t.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1.0, 200.0, log=True),
        "rounds": t.suggest_int("rounds", 200, 1000, log=True),
    },
}

for scope, cols in SCOPES.items():
    Xc = Xf[:, cols]
    Xs = CACHE_M[SUBMIT][:, cols]
    w_nn, _ = nnls(Xc, yf)
    for tag, w in (("linear_nnls", w_nn), ("linear_ridge_1e-2", nl_ridge_w(Xc, yf))):
        np.save(NL_STACK_DIR / f"sub_{scope}_{tag}.npy", z_score(Xs @ w).astype(np.float32))
    for name in ("lgbm", "xgb", "catboost", "rf", "et", "mlp"):
        for resid in (False, True):
            tag = f"{name}{'_on_residual' if resid else ''}"
            out = NL_STACK_DIR / f"sub_{scope}_{tag}.npy"
            if out.exists():
                continue

            def objective(t, n=name, c=cols, r=resid):
                p = nl_fit_predictor(nl_make_fn(n, NL_SPACE[n](t)), Xi[:, c], yi, r)
                return nl_rho(p(Xv[:, c]), yv)

            study = optuna.create_study(
                direction="maximize", sampler=optuna.samplers.TPESampler(seed=NL_SEED)
            )
            study.optimize(objective, n_trials=NL_TRIALS[name], show_progress_bar=False)
            params = dict(study.best_params)
            if name in ("rf", "et"):
                params["n_estimators"] = 120 if name == "rf" else 200
            p = nl_fit_predictor(nl_make_fn(name, params), Xc, yf, resid)
            np.save(out, z_score(p(Xs)).astype(np.float32))
            print(f"  {scope} {tag} inner {study.best_value}", flush=True)

### v46 and v47

In [ ]:
NL_BETA = {"kept": 1.6276200554303766, "all": 1.6275976097615987}
NL_FILE = {"kept": "submit_v46_nl_kept", "all": "submit_v47_nl_all"}

for scope in ("kept", "all"):
    if (SUBS / f"{NL_FILE[scope]}.csv").exists():
        continue
    cols = SCOPES[scope]
    Z = CACHE_M[SUBMIT][:, cols]
    lz = Z @ NNLS_W[scope]
    xs = np.column_stack([Z, lz])
    f = (
        BOOSTERS[scope].predict(xs)
        if TUNED_FAMILY[scope] == "lgbm"
        else BOOSTERS[scope].predict(xgb.DMatrix(xs))
    )
    write_from_z(
        NL_FILE[scope],
        lz + f,
        NL_BETA[scope],
        USER_IDS,
        source=f"{TUNED_FAMILY[scope]} on the residual of the {scope} scope",
    )

### v48

In [ ]:
NEW_DIRS = ["v46_nl_kept", "v47_nl_all"]
NEWDIR_RIDGE, SE_B = 1.0e-4, 0.000042


def new_directions(a, ridge=NEWDIR_RIDGE, verbose=True):
    old = [i for i in range(len(a["names"])) if a["names"][i] not in NEW_DIRS]
    Zo = a["Z"][old]
    Co = a["C"][np.ix_(old, old)]
    qo = a["rho"][old]
    basis, covs, norms = [], [], []
    for nm in NEW_DIRS:
        i = a["idx"][nm]
        z = a["Z"][i]
        c = np.linalg.solve(Co + ridge * np.eye(len(old)), Zo @ z / z.size)
        r = z - c @ Zo
        lb_n = SCORED[nm][1]
        b_obs = (W_TOTAL + a["beta"][i] ** 2 - lb_n * lb_n) / (2.0 * a["beta"][i])
        b_span = SIGMA * float(c @ qo)
        cov_r = b_obs - b_span
        for e, ce in zip(basis, covs, strict=True):
            g = float(r @ e / r.size)
            r = r - g * e
            cov_r = cov_r - g * ce
        n_r = float(np.sqrt(r @ r / r.size))

        basis.append(r / n_r)
        covs.append(cov_r / n_r)
        norms.append(n_r)
    return basis, covs, norms


A65 = build_archive([*ARCHIVE_ORDER_60, *ARCHIVE_POST_43, *NEW_DIRS])
b43 = float(np.sqrt(W_TOTAL - SCORED["v43_calib"][1] ** 2))
print(f"{len(A65['names'])} vectors, base v43_calib B {b43}\n")
basis48, covs48, norms48 = new_directions(A65)
db = [c * n for c, n in zip(covs48, norms48, strict=True)]
shrink = [d * d / (d * d + SE_B * SE_B) for d in db]
alpha48 = [k * c / b43 for k, c in zip(shrink, covs48, strict=True)]
a2 = float(sum(al * al for al in alpha48))
b_exp48 = (b43 + float(sum(al * c for al, c in zip(alpha48, covs48, strict=True)))) / np.sqrt(
    1.0 + a2
)
s_exp48 = float(np.sqrt(W_TOTAL - b_exp48 * b_exp48))
for j, (c, d, k, al) in enumerate(zip(covs48, db, shrink, alpha48, strict=True)):
    print(f"{j + 1}{c}{d}{d / SE_B}{k}{al}")
print(f"\nexpected B {b_exp48} -> score {s_exp48}")

z48 = A65["Z"][A65["idx"]["v43_calib"]].copy()
for al, e in zip(alpha48, basis48, strict=True):
    z48 = z48 + al * np.asarray(e, dtype=np.float64)
write_from_z(
    "submit_v48_newdir", z48, b_exp48, A65["ids"], source="v43 plus the two new directions"
)
np.save(LEGACY_DIR / "newdir_basis.npy", np.array(basis48, dtype=np.float32))
np.save(LEGACY_DIR / "newdir_covs.npy", np.array(covs48))

### v49's direction, and v51

In [ ]:
NL_SENT = {"kept_lgbm_on_residual", "all_xgb_on_residual"}
V49_ALPHA = 0.006
FLOOR, PRIOR_SD = 0.01424, 0.020

A66 = build_archive([*A65["names"], "v48_newdir"])
k66 = len(A66["names"])
Z66 = A66["Z"]
zb48 = A66["Z"][A66["idx"]["v48_newdir"]]

names_nl, R = [], []
for p in sorted(NL_STACK_DIR.glob("sub_*.npy")):
    tag = p.stem.replace("sub_", "")
    if tag in NL_SENT or "linear" in tag:
        continue
    z = z_score(np.load(p).astype(np.float64))
    c = np.linalg.solve(A66["C"] + NEWDIR_RIDGE * np.eye(k66), Z66 @ z / z.size)
    names_nl.append(tag)
    R.append(z - c @ Z66)
R = np.array(R)
print(f"{len(names_nl)} unsent combiner vectors")
G = R @ R.T / R.shape[1]
ev, evec = np.linalg.eigh(G)
u = evec[:, -1]
d_nl = z_score(u @ R)
d_nl = z_score(d_nl - float(d_nl @ zb48 / d_nl.size) * zb48)
print(f"top eigenvalue holds {ev[-1] / ev.sum()} of the residual mass")
np.save(LEGACY_DIR / "d_nlresidue.npy", d_nl.astype(np.float32))

b_cur51 = (W_TOTAL + A66["beta"][A66["idx"]["v48_newdir"]] ** 2 - SCORED["v48_newdir"][1] ** 2) / (
    2 * A66["beta"][A66["idx"]["v48_newdir"]]
)
E = np.array([z_score(v) for v in np.array(basis48, dtype=np.float64)])
covs_e = np.array(covs48)
n66 = Z66.shape[1]
G_e = E @ E.T / n66
coef = np.linalg.solve(G_e, E @ d_nl / n66)
known = float(coef @ covs_e)
r51 = d_nl - coef @ E
r_norm = float(np.sqrt(r51 @ r51 / n66))
m = float(covs_e.mean())
se = FLOOR / np.sqrt(len(covs_e))
prior = m * PRIOR_SD**2 / (PRIOR_SD**2 + se**2)
c_exp = known + r_norm * prior
print(f"base v48 public {SCORED['v48_newdir'][1]}  B {b_cur51}")
print(f"d on banked directions: coef {coef[0]} {coef[1]}  out-of-span norm {r_norm}")
print(f"known part of Cov(d,u)      {known}")
print(f"family mean {m} se {se} -> shrunk prior {prior}")
print(f"E[Cov(d,u)] = {c_exp}")

alpha51 = c_exp / b_cur51
beta51 = (b_cur51 + alpha51 * c_exp) / np.sqrt(1.0 + alpha51 * alpha51)
print(f"\nalpha* = E[Cov]/B = {alpha51}   beta = {beta51}")
for c in (0.0, 0.005, 0.010, c_exp, 0.020, 0.0235, 0.030):
    b_mix = (b_cur51 + alpha51 * c) / np.sqrt(1.0 + alpha51 * alpha51)
    s = float(np.sqrt(W_TOTAL - b_mix * b_mix))
    tag = " <- E[Cov]" if abs(c - c_exp) < 1e-9 else ""

write_from_z(
    "submit_v49_probe",
    zb48 + V49_ALPHA * d_nl,
    b_cur51 / np.sqrt(1.0 + V49_ALPHA**2),
    A66["ids"],
    source="v48 plus the unsent-combiner residual direction, probed",
)
write_from_z(
    "submit_v51_bank",
    zb48 + alpha51 * d_nl,
    beta51,
    A66["ids"],
    source="v48 plus the unpriced direction at its expected optimum",
)

## Verify

In [ ]:
FINAL = ["submit_v43_calib", "submit_v51_bank"]
COMPARE = {}
for stem in FINAL:
    chk = pl.read_csv(SUBS / f"{stem}.csv")
    vals = chk["predict"].to_numpy()
    print(f"{stem} rows {chk.height}  mean {vals.mean()}  max {vals.max()}")

print(f"\n{'file'}{'corr vs archived'}{'max abs diff'}")
for stem in FINAL:
    ref = REFERENCE / f"{stem}.csv"
    if not ref.exists():
        print(f"{stem}{'no reference'}")
        continue
    fresh = (
        pl.read_csv(SUBS / f"{stem}.csv").sort("user_id")["predict"].to_numpy().astype(np.float64)
    )
    arch = pl.read_csv(ref).sort("user_id")["predict"].to_numpy().astype(np.float64)
    c = float(
        np.corrcoef(np.log1p(np.clip(fresh, 0, None)), np.log1p(np.clip(arch, 0, None)))[0, 1]
    )
    d = float(np.abs(fresh - arch).max())
    COMPARE[stem] = {"corr": c, "max_abs_diff": d}
    print(f"{stem}{c}{d}")

### Log

In [ ]:
ARCHITECTURE = (
    "Full chain from one parquet. Two feature spaces, pre-audit and repaired, so the eras "
    "before and after the audit keep the distance the archive Gram depends on. Era one "
    "is CatBoost v1, CatBoost v2 and its drop-zeros twin, and two Chronos zero shots. Era two "
    "is the four member v3 blend at two levels. Era three is the v4 nine and twenty member "
    "blends, the v5 rescale, the v6 three-way split and the v7 twenty five member pool. The "
    "probe ladder v8 to v12 is exact algebra over scored files. The diversity fleet and its "
    "direction probes bank six measured covariances into v21, and the structural probes bank "
    "two more into z_bank. v29 retrains the pool on the repaired features; v30 to v37 bank the "
    "sequence and curvature directions. The archive layer then solves a ridge over sixty "
    "scored vectors for v43, adds the two combiner directions outside their span for v48, and "
    "spends the last submission on the unsent-combiner residual for v51."
)
with mlflow.start_run(run_name="prod-train-full-chain"):
    mlflow.set_tags({
        "pipeline": "prod",
        "stage": "train",
        "target": "submit_v43_calib,submit_v51_bank",
    })
    mlflow.log_params({
        "members": ",".join(MEMBER_ORDER),
        "n_members": len(MEMBER_ORDER),
        "archive_vectors": len(A60["names"]),
        "v43_ridge": float(lam43),
        "v43_leverage": float(lev43),
        "submit_anchor": SUBMIT,
        "architecture": ARCHITECTURE,
    })
    mlflow.log_metrics({
        "v43_projection": float(proj43),
        "v43_expected": float(exp43),
        "v43_beta": float(beta43),
        "v48_b_expected": float(b_exp48),
        "v48_score_expected": float(s_exp48),
        "v51_alpha": float(alpha51),
        "v51_beta": float(beta51),
        "v51_cov_expected": float(c_exp),
        "public_lb_rmsle": SCORED["v43_calib"][1],
    })
    if COMPARE:
        mlflow.log_metrics({f"corr_vs_archived_{k}": v["corr"] for k, v in COMPARE.items()})
        mlflow.log_dict(COMPARE, "compare_vs_archived.json")
    for stem in FINAL:
        mlflow.log_artifact(str(SUBS / f"{stem}.csv"), "submissions")